In [3]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date
from collections import defaultdict

# =============================================================
#  Smart APS V17  —  Half/Full Run Machine + Sheet Fix
# =============================================================
# CHANGES FROM V16:
#   1. HALF/FULL RUN MACHINE LOGIC:
#      - Full Run machines load TWO tools of the same part simultaneously.
#        If a part has >= 2 tools available AND is planned on a Full Run
#        machine, its effective production rate is DOUBLED.
#      - Half Run machines load ONE tool at a time — normal rate.
#      - If a part has only 1 tool available, it always runs at 1x rate
#        regardless of machine type (only one tool to load).
#      - Only ONE PART runs on a machine at a time (1 or 2 tools).
#      - Machine type (FULL/HALF) is read from a "Run_Type" column in
#        the VT_Changeover sheet. Default = HALF if column absent.
#      - The effective rate is used in all quantity calculations.
#        Plan rows show both Base_Rate_Per_Hour and Rate_Per_Hour
#        (effective).  A new column Machine_Run_Type is added.
#   2. SHEET BREAKING FIX:
#      - Fixed the output_path f-string bug (date was not being evaluated).
#      - Added _safe_df() sanitizer: mixed-type columns (e.g. numeric rows
#        plus "—" summary rows) are converted to consistent types before
#        writing, eliminating Excel's "repair needed" dialog.
# ALL V16 LOGIC IS PRESERVED UNCHANGED.
# =============================================================

# =============================================================
# SECTION 1 — DAILY SETTINGS
# =============================================================

PLANNING_DATE = date(2026, 4, 9)
INDENT_MONTH  = date(2026, 4, 1)

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS          = 19.8
AVAILABLE_HOURS_EXTENDED = 23
MIN_RUN_HOURS            = 4
MACHINE_STATE_FILE       = "machine_state.json"

MIN_DAILY_INDENT         = 150
MIN_INDENT_HOURS         = 4.0

SAFETY_DAYS              = 3
TARGET_DAYS              = 5

OPD_SCENARIO_0 = 3.0
OPD_SCENARIO_1 = 3.0
OPD_SCENARIO_2 = 4.0
OPD_SCENARIO_3 = 5.0

W_URGENCY  = 0.55
W_CATEGORY = 0.25
W_INDENT   = 0.20

UTIL_TARGET_PCT  = 90.0
COLOR_PURGE_HRS  = 10 / 60.0

RUNNER_PRIORITY_DAYS = 2.0

# Max distinct parts per machine
MAX_PARTS_PER_MACHINE    = 3

# Runtime dynamic cap (recomputed each scheduling run)
_dynamic_max_parts       = MAX_PARTS_PER_MACHINE

MAX_DAILY_CO             = 25
FORWARD_LOOK_DAYS        = 7

# NEW terminal relaxation: 10-15% shortfall allowed if MIN_RUN_HOURS is met
# Hard floor = required_qty * (1 - TERMINAL_RELAXATION_PCT)
# If terminal_inv < hard floor -> HARD BLOCK (no production)
# If hard floor <= terminal_inv < required_qty -> RELAXED (allowed, flagged)
# If terminal_inv >= required_qty -> OK
TERMINAL_RELAXATION_PCT  = 0.15   # 15% relaxation

# =============================================================
# TERMINAL CONSTRAINT BYPASS FLAG
# Set to True  -> terminals are assumed ALWAYS AVAILABLE.
#                 terminal_blocked() always returns (False,'',False)
#                 terminal_coverage_ratio() always returns 1.0
#                 All parts that were previously hard-blocked
#                 by terminal inventory will now be scheduled.
# Set to False -> normal terminal constraint logic applies.
# =============================================================
IGNORE_TERMINAL_CONSTRAINT = True   # <-- SET TO True FOR 'TERMINALS ALWAYS AVAILABLE' SCENARIO

STRATEGIC_BUFFER_DAYS        = 7
STRATEGIC_PRIORITY_DISCOUNT  = 0.5
EFFICIENCY_MODE_UTIL_FLOOR   = 95.0
ABSOLUTE_MAX_DAYS            = 15

# V17: Full Run multiplier — when a Full Run machine loads 2 tools simultaneously
FULL_RUN_MULTIPLIER = 2.0

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
terminal_path   = "C:/Users/Ex0164/Important codes/terminals and raw marterial - vt.xlsx"
demand_path     = "C:/Users/Ex0164/Important codes/Child_for_10TH may_actual.xlsx"

# V17 FIX: corrected f-string so the date is actually evaluated
output_path = (
    f"Smart_APS_V17_Plan_{PLANNING_DATE.strftime('%Y%m%d')}"
    + ("_NO_TERMINAL_CONSTRAINT" if IGNORE_TERMINAL_CONSTRAINT else "")
    + "_19.8 HRS aps v50 8TH MAY FULL AND HALF RUN LOGIC.xlsx"
)

DEMAND_SHEET_NAME  = "VT_Demand"
DEMAND_PART_COL    = "Part"
DEMAND_DAILY_COL   = "Daily_Demand"

# =============================================================
# SECTION 4 — WORKING DAYS
# =============================================================

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print("\n" + "─"*65)
print(f"  Smart APS V17  —  Half/Full Run + Sheet Fix")
print(f"  Planning date    : {PLANNING_DATE}")
print(f"  Indent month     : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days     : {WORKING_DAYS}  ({TOTAL_DAYS} days - {SUNDAY_COUNT} Sundays)")
print(f"  Safety floor     : {SAFETY_DAYS}d  |  Target ceiling : {TARGET_DAYS}d")
print(f"  Max parts/machine: {MAX_PARTS_PER_MACHINE}")
print(f"  Runner priority  : inv < {RUNNER_PRIORITY_DAYS}x daily")
print(f"  Max daily CO     : {MAX_DAILY_CO}")
print(f"  Forward look     : {FORWARD_LOOK_DAYS} days")
print(f"  Terminal relax   : {int(TERMINAL_RELAXATION_PCT*100)}%  (min-run NON-NEGOTIABLE)")
print(f"  Terminal logic   : required = max(demand_daily, indent_daily)")
print(f"                     hard_block if inv < required*(1-{int(TERMINAL_RELAXATION_PCT*100)}%)")
print(f"  TERMINAL BYPASS  : {'ACTIVE — terminals assumed always available (IGNORE_TERMINAL_CONSTRAINT=True)' if IGNORE_TERMINAL_CONSTRAINT else 'INACTIVE — normal terminal constraint in effect'}")
print(f"  Full Run mult    : {FULL_RUN_MULTIPLIER}x  (applies when tools>=2 on FULL machine)")
print(f"{'='*65}\n")

# =============================================================
# SECTION 5 — LOAD DATA
# =============================================================

print("Loading data...")
vt_parts_raw         = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix            = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw            = pd.read_excel(changeover_path, sheet_name="VT_Changeover")
vt_machine_count_raw = pd.read_excel(matrix_path,     sheet_name="VT_Machine_Part_Count")

try:
    vt_fixed_raw = pd.read_excel(matrix_path, sheet_name="VT_Fixed")
    print(f"  VT_Fixed sheet loaded  ({len(vt_fixed_raw)} rows)")
except Exception as _fe:
    vt_fixed_raw = None
    print(f"  WARNING: VT_Fixed sheet not found ({_fe})")

try:
    vt_terminals_raw      = pd.read_excel(terminal_path, sheet_name="VT_Terminals")
    vt_terminal_avail_raw = pd.read_excel(terminal_path, sheet_name="VT_Terminal_Inventory")
    print(f"  Terminal data loaded from  : {terminal_path}")
except FileNotFoundError:
    vt_terminals_raw      = None
    vt_terminal_avail_raw = None
    print(f"  WARNING: terminal_path not found — terminal constraint DISABLED.")
except Exception as _te:
    vt_terminals_raw      = None
    vt_terminal_avail_raw = None
    print(f"  WARNING: Could not load terminal data ({_te}) — constraint DISABLED.")

# ── V17: Load machine run type (FULL / HALF) from VT_Changeover ──────────────
# The VT_Changeover sheet may have a column named "Run_Type" or "Machine_Type".
# Values: "FULL" = full run (2 tools simultaneously), "HALF" = half run (1 tool).
# If the column is absent, all machines default to HALF RUN.
machine_run_type = {}   # {machine_name: "FULL" or "HALF"}

_run_type_col = next(
    (c for c in vt_co_raw.columns
     if str(c).strip().lower() in ("run_type", "machine_type", "type", "run type", "machine type")),
    None
)
if _run_type_col:
    for _, _row_rt in vt_co_raw.iterrows():
        _m_rt = str(_row_rt.get("Unique Machines", "")).strip()
        _rt   = str(_row_rt[_run_type_col]).strip().upper() if pd.notna(_row_rt[_run_type_col]) else "HALF"
        if _m_rt:
            machine_run_type[_m_rt] = "FULL" if _rt == "FULL" else "HALF"
    _full_count = sum(1 for v in machine_run_type.values() if v == "FULL")
    _half_count = sum(1 for v in machine_run_type.values() if v == "HALF")
    print(f"  Machine run types loaded   : {_full_count} FULL, {_half_count} HALF")
    if _full_count:
        print(f"  FULL run machines          : {[m for m,t in machine_run_type.items() if t=='FULL']}")
else:
    print(f"  NOTE: No 'Run_Type' column found in VT_Changeover — all machines default to HALF RUN")
    print(f"        Add a 'Run_Type' column (FULL/HALF) to enable full-run rate doubling.")

# ── Load demand file ──────────────────────────────────────────
demand_monthly_raw = {}
demand_daily_raw   = {}

try:
    demand_df = pd.read_excel(demand_path, sheet_name=DEMAND_SHEET_NAME)
    demand_df.columns = [str(c).strip().lstrip('\ufeff') for c in demand_df.columns]
    print(f"  Demand file columns found  : {list(demand_df.columns)}")
    print(f"  Demand file rows           : {len(demand_df)}")

    _part_col_d = next(
        (c for c in demand_df.columns if c.strip().lower() == DEMAND_PART_COL.lower()), None
    )
    _dem_col_d = next(
        (c for c in demand_df.columns if c.strip().lower() == DEMAND_DAILY_COL.lower()), None
    )
    if _part_col_d is None:
        _part_col_d = next((c for c in demand_df.columns if 'part' in c.strip().lower()), None)
        if _part_col_d:
            print(f"  NOTE: Using '{_part_col_d}' as Part column (partial match)")
    if _dem_col_d is None:
        _dem_col_d = next(
            (c for c in demand_df.columns if 'daily' in c.strip().lower() or 'demand' in c.strip().lower()), None
        )
        if _dem_col_d:
            print(f"  NOTE: Using '{_dem_col_d}' as Daily_Demand column (partial match)")

    if _part_col_d and _dem_col_d:
        loaded_count = skipped_count = 0
        for _, row in demand_df.iterrows():
            p = row[_part_col_d]
            v = row[_dem_col_d]
            if pd.isna(p) or str(p).strip() == "":
                skipped_count += 1
                continue
            part_key = str(p).strip()
            try:
                daily_dem = float(v) if pd.notna(v) else 0.0
            except (ValueError, TypeError):
                daily_dem = 0.0
            demand_daily_raw[part_key]   = round(daily_dem, 4)
            demand_monthly_raw[part_key] = round(daily_dem * WORKING_DAYS, 4)
            loaded_count += 1
        print(f"  Demand loaded (DAILY)      : {loaded_count} parts")
        if skipped_count:
            print(f"  Demand rows skipped        : {skipped_count}")
        pos_dem = sum(1 for d in demand_daily_raw.values() if d > 0)
        print(f"  Parts with positive demand : {pos_dem}")
        sample = [p for p, d in demand_daily_raw.items() if d > 0][:5]
        if sample:
            print(f"  Demand spot-check (first 5):")
            for sp in sample:
                print(f"    {sp:<30} daily={demand_daily_raw[sp]:.4f}")
    else:
        print(f"  WARNING: Demand columns not found — demand constraint DISABLED.")
except FileNotFoundError:
    print(f"  WARNING: demand_path not found — demand constraint DISABLED.")
except Exception as _de:
    print(f"  WARNING: Could not load demand data ({_de}) — demand constraint DISABLED.")
    import traceback; traceback.print_exc()

# =============================================================
# SECTION 5A — EFFECTIVE DAILY / MONTHLY HELPERS
# =============================================================

def effective_daily(part: str) -> float:
    """Return max(indent_daily, demand_daily) for a part."""
    ind = indent_daily.get(part, 0.0)
    dem = demand_daily_raw.get(part, 0.0)
    return max(ind, dem)

def effective_monthly(part: str) -> float:
    ind = indent_monthly.get(part, 0.0)
    dem = demand_monthly_raw.get(part, 0.0)
    return max(ind, dem)

def demand_driver(part: str) -> str:
    ind = indent_daily.get(part, 0.0)
    dem = demand_daily_raw.get(part, 0.0)
    if dem > ind + 0.001:
        return f"DEMAND({dem:.2f}>indent {ind:.2f})"
    elif ind > dem + 0.001:
        return f"INDENT({ind:.2f})"
    elif ind > 0:
        return f"EQUAL({ind:.2f})"
    else:
        return "NO_DRIVER"

def demand_today_qty(part: str, current_inv: float) -> float:
    """How much we must produce TODAY to cover today's demand gap."""
    dem = demand_daily_raw.get(part, 0.0)
    return max(0.0, dem - current_inv) if dem > 0 else 0.0

def is_demand_met(part: str, inv_before: float, produced: float) -> bool:
    """Demand is met if inventory_before + produced >= demand_daily."""
    dem = demand_daily_raw.get(part, 0.0)
    if dem <= 0:
        return True
    return (inv_before + produced) >= (dem - 0.5)

# =============================================================
# SECTION 6 — PARSE VT SHEET
# =============================================================

def find_col(df, name, sheet):
    match = next((c for c in df.columns if str(c).strip().lower() == name.lower()), None)
    if match is None:
        raise ValueError(f"Column '{name}' not found in sheet '{sheet}'.\nAvailable: {list(df.columns)}")
    return match

vt_col_part      = find_col(vt_parts_raw, "Part",       "VT")
vt_col_cycletime = find_col(vt_parts_raw, "Cycle time", "VT")
vt_col_cavity    = find_col(vt_parts_raw, "Cavity",     "VT")
vt_col_inventory = find_col(vt_parts_raw, "Inventory",  "VT")
vt_col_indent    = find_col(vt_parts_raw, "Indent",     "VT")
vt_col_tools     = find_col(vt_parts_raw, "Tools",      "VT")
vt_col_color     = find_col(vt_parts_raw, "Color",      "VT")

data = vt_parts_raw[
    vt_parts_raw[vt_col_part].notna() &
    (vt_parts_raw[vt_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=vt_col_part).copy()
data["Material"] = data[vt_col_part].astype(str).str.strip()
data["_ct"] = pd.to_numeric(data[vt_col_cycletime], errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]
data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()
print(f"  VT parts in sheet       : {len(data)}  |  With valid rate: {len(data_valid)}")

# =============================================================
# SECTION 7 — LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", vt_col_inventory)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", vt_col_indent)
indent_daily   = {p: round(qty / WORKING_DAYS, 4) for p, qty in indent_monthly.items()}

tools_available = {}
part_color      = {}

for _, row in data.iterrows():
    p = str(row["Material"]).strip()
    v = row[vt_col_tools]
    tools_available[p] = max(1, int(float(v))) if pd.notna(v) and str(v).strip() != "" else 1
    c = row[vt_col_color]
    part_color[p] = str(c).strip().upper() if pd.notna(c) and str(c).strip() not in ("", "nan") else "UNKNOWN"

ALL_KNOWN_COLORS = dict(part_color)

color_groups = {}
for p, c in part_color.items():
    color_groups.setdefault(c, []).append(p)
print(f"  Distinct colours        : {len(color_groups)}")
for col, pts in sorted(color_groups.items()):
    print(f"    {col:<20} -> {len(pts)} part(s)")

today_target_qty = {
    p: max(0.0, effective_daily(p) - inventory.get(p, 0.0))
    for p in set(list(indent_monthly.keys()) + list(demand_daily_raw.keys()))
}

# =============================================================
# SECTION 7A0 — V17: EFFECTIVE RATE (HALF / FULL RUN)
# =============================================================
# Rule:
#   Full Run machine AND part.tools >= 2  ->  rate × FULL_RUN_MULTIPLIER (2×)
#   All other cases                       ->  rate × 1  (normal)
#
# Rationale:
#   A Full Run machine loads BOTH tools of a part simultaneously, so the
#   part is produced on both cavities at once — effectively doubling output
#   per hour.  A Half Run machine (or a part with only 1 tool) loads a
#   single tool, so rate is unchanged.
#
#   Only ONE PART runs on any machine at a given time whether it has
#   1 or 2 tools.  This function controls only the throughput rate,
#   not the part-per-machine cap.
# =============================================================

def effective_rate(part: str, machine: str) -> float:
    """
    Return the effective production rate (pcs/hr) for a part on a machine.

    Full Run machine + part has >= 2 tools  ->  base_rate * 2
    Otherwise                               ->  base_rate * 1

    Returns 0.0 if the base rate is unknown.
    """
    base = rate.get(part, 0.0)
    if base <= 0:
        return 0.0
    tools = tools_available.get(part, 1)
    mtype = machine_run_type.get(machine, "HALF")
    multiplier = FULL_RUN_MULTIPLIER if (mtype == "FULL" and tools >= 2) else 1.0
    return base * multiplier


def machine_run_label(part: str, machine: str) -> str:
    """Human-readable label for the run mode used for this part/machine pair."""
    tools = tools_available.get(part, 1)
    mtype = machine_run_type.get(machine, "HALF")
    if mtype == "FULL" and tools >= 2:
        return f"FULL-RUN (2 tools x {FULL_RUN_MULTIPLIER}x)"
    elif mtype == "FULL" and tools < 2:
        return "FULL-machine / 1-tool (normal rate)"
    else:
        return "HALF-RUN (1 tool)"


# =============================================================
# SECTION 7A — FIXED MACHINE CONSTRAINT
# =============================================================

def build_fixed_machine_dicts(df):
    pfm, mfp = {}, {}
    if df is None or df.empty:
        return pfm, mfp
    machine_col = next((c for c in df.columns if str(c).strip().lower() == "machine"), None)
    if machine_col is None:
        print("  WARNING: VT_Fixed has no 'Machine' column — fixed constraint disabled")
        return pfm, mfp
    part_cols = [c for c in df.columns if str(c).strip().lower() != "machine"]
    if not part_cols:
        print("  WARNING: VT_Fixed has no part columns — fixed constraint disabled")
        return pfm, mfp
    for _, row in df.iterrows():
        machine = row[machine_col]
        if pd.isna(machine) or str(machine).strip() == "":
            continue
        m = str(machine).strip()
        for col in part_cols:
            val = row[col]
            if pd.isna(val) or str(val).strip() in ("", "nan"):
                continue
            p = str(val).strip()
            if p in pfm:
                print(f"  WARNING: Part '{p}' in VT_Fixed more than once — keeping {pfm[p]}")
                continue
            pfm[p] = m
            mfp.setdefault(m, []).append(p)
    return pfm, mfp

part_fixed_machine, machine_fixed_parts = build_fixed_machine_dicts(vt_fixed_raw)
print(f"  Fixed machine mappings  : {len(part_fixed_machine)} parts")
if part_fixed_machine:
    for m, parts in sorted(machine_fixed_parts.items()):
        print(f"    {m:<25} <- {', '.join(parts)}")

# =============================================================
# SECTION 7A2 — FIXED MACHINE PHASE HELPERS
# =============================================================

def fixed_machine_phase(machine, current_inventory):
    for p in machine_fixed_parts.get(machine, []):
        daily = effective_daily(p)
        if daily > 0 and current_inventory.get(p, 0) < SAFETY_DAYS * daily:
            return "A"
    return "B"

def pick_fixed_part_for_today(machine, current_inventory, machine_last_part):
    candidates = []
    for p in machine_fixed_parts.get(machine, []):
        daily = effective_daily(p)
        r_val = effective_rate(p, machine)
        inv   = current_inventory.get(p, 0)
        if daily <= 0:
            continue
        candidates.append((p, inv / daily, daily / r_val if r_val > 0 else 0))
    if not candidates:
        return None
    candidates.sort(key=lambda x: (round(x[1], 4), -x[2]))
    if len(candidates) >= 2:
        top, second = candidates[0], candidates[1]
        if abs(top[1] - second[1]) < 0.01 and machine_last_part.get(machine) == top[0]:
            return second[0]
    return candidates[0][0]

def fixed_part_run_hours(part, machine, phase):
    daily = effective_daily(part)
    r_val = effective_rate(part, machine)
    if daily <= 0 or r_val <= 0:
        return AVAILABLE_HOURS
    indent_hrs = daily / r_val
    if phase == "A":
        return AVAILABLE_HOURS_EXTENDED if indent_hrs > 20.0 else AVAILABLE_HOURS
    else:
        return max(MIN_RUN_HOURS, indent_hrs)

# =============================================================
# SECTION 7B — NEW TERMINAL CONSTRAINT
# =============================================================
# Logic:
#   required_qty = max(demand_daily, indent_daily)  [per-terminal, for this part]
#   hard_floor   = required_qty * (1 - TERMINAL_RELAXATION_PCT)
#
#   terminal_inv >= required_qty       -> OK (fully covered)
#   hard_floor <= terminal_inv < req   -> RELAXED (allow, flag in output)
#   terminal_inv < hard_floor          -> HARD BLOCK (no production)
#
#   MIN_RUN_HOURS is non-negotiable — even with relaxation, if a part
#   cannot run for MIN_RUN_HOURS given terminal stock, it is blocked.
#
#   Relaxation applies to ALL parts (fixed and non-fixed).
# =============================================================

def _build_part_terminals(df):
    result = {}
    if df is None or df.empty:
        return result
    part_col = next((c for c in df.columns if str(c).strip().lower() in ("part", "material")), None)
    if part_col is None:
        print("  WARNING: VT_Terminals has no 'Part'/'Material' column")
        return result
    terminal_cols = [c for c in df.columns if str(c).strip().lower() not in ("part", "material")]
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        p = str(part).strip()
        terminals = [
            str(row[col]).strip().upper()
            for col in terminal_cols
            if pd.notna(row[col]) and str(row[col]).strip() not in ("", "nan")
        ]
        if terminals:
            result[p] = terminals
    return result

def _build_terminal_status(df):
    result = {}
    if df is None or df.empty:
        return result
    term_col = next((c for c in df.columns if str(c).strip().lower() == "terminal"), None)
    inv_col  = next((c for c in df.columns if str(c).strip().lower() == "inventory"), None)
    if term_col is None or inv_col is None:
        print("  WARNING: VT_Terminal_Inventory missing 'Terminal' or 'Inventory' column")
        return result
    for _, row in df.iterrows():
        t = row[term_col]
        v = row[inv_col]
        if pd.isna(t) or str(t).strip() == "":
            continue
        key = str(t).strip().upper()
        try:
            qty = float(v) if pd.notna(v) else 0.0
        except (ValueError, TypeError):
            qty = 0.0
        result[key] = qty
    return result

part_terminals  = _build_part_terminals(vt_terminals_raw)
terminal_status = _build_terminal_status(vt_terminal_avail_raw)

if terminal_status:
    zero_count  = sum(1 for v in terminal_status.values() if v <= 0)
    avail_count = len(terminal_status) - zero_count
    print(f"  Terminals loaded : {len(terminal_status)} total  |  "
          f"{avail_count} with stock  |  {zero_count} at ZERO inventory")
else:
    print(f"  Terminals        : no data loaded — constraint inactive")


def terminal_blocked(part):
    """
    NEW terminal constraint logic (V14):

    required_qty = max(demand_daily, indent_daily)
    hard_floor   = required_qty * (1 - TERMINAL_RELAXATION_PCT)

    For EACH terminal required by the part:
      - If terminal_inv < hard_floor  -> HARD BLOCK
      - If hard_floor <= terminal_inv < required_qty -> RELAXED (allowed with flag)
      - If terminal_inv >= required_qty -> OK

    Returns (blocked: bool, reason: str, relaxed: bool)
    blocked=True means hard block — do not schedule.
    relaxed=True means within relaxation band — schedule but flag it.
    """
    # ── TERMINAL BYPASS: terminals assumed always available ──
    if IGNORE_TERMINAL_CONSTRAINT:
        return False, "", False
    required_terminals = part_terminals.get(part, [])
    if not required_terminals:
        return False, "", False

    dem_d   = demand_daily_raw.get(part, 0.0)
    ind_d   = indent_daily.get(part, 0.0)
    req_qty = max(dem_d, ind_d)

    r_val = rate.get(part, 0.0)
    min_run_qty = MIN_RUN_HOURS * r_val if r_val > 0 else 0.0

    effective_required = max(req_qty, min_run_qty)

    hard_floor   = effective_required * (1.0 - TERMINAL_RELAXATION_PCT)

    hard_blocking = []
    soft_blocking = []

    for t in required_terminals:
        t_inv = terminal_status.get(t, 0.0)

        if t_inv < hard_floor:
            hard_blocking.append(
                f"{t}(inv={t_inv:.0f} < hard_floor={hard_floor:.0f} "
                f"[required={effective_required:.0f}, -{int(TERMINAL_RELAXATION_PCT*100)}% relax])"
            )
        elif t_inv < effective_required:
            soft_blocking.append(
                f"{t}(inv={t_inv:.0f} within {int(TERMINAL_RELAXATION_PCT*100)}% of "
                f"required={effective_required:.0f} [dem={dem_d:.2f}, indent={ind_d:.2f}])"
            )

    if hard_blocking:
        return (True,
                f"Terminal HARD BLOCK: {', '.join(hard_blocking)}",
                False)
    if soft_blocking:
        return (False,
                f"Terminal RELAXED ({int(TERMINAL_RELAXATION_PCT*100)}%): {', '.join(soft_blocking)}",
                True)
    return False, "", False


def terminal_blocked_legacy(part):
    """Backward-compat wrapper returning (blocked, reason)."""
    blocked, reason, _ = terminal_blocked(part)
    return blocked, reason


def terminal_coverage_ratio(part):
    """
    0.0-1.0: how well-stocked this part's terminals are vs required quantity.
    required = max(demand_daily, indent_daily, min_run_qty).
    Used as tie-breaker when two parts have equal demand.
    """
    # ── TERMINAL BYPASS: all terminals treated as fully stocked ──
    if IGNORE_TERMINAL_CONSTRAINT:
        return 1.0
    required_terminals = part_terminals.get(part, [])
    if not required_terminals:
        return 1.0
    dem_d = demand_daily_raw.get(part, 0.0)
    ind_d = indent_daily.get(part, 0.0)
    req_qty = max(dem_d, ind_d)
    r_val = rate.get(part, 0.0)
    min_run_qty = MIN_RUN_HOURS * r_val if r_val > 0 else 0.0
    effective_required = max(req_qty, min_run_qty)
    if effective_required <= 0:
        return 1.0
    ratios = []
    for t in required_terminals:
        t_inv = terminal_status.get(t, 0.0)
        ratios.append(min(1.0, t_inv / effective_required))
    return round(min(ratios), 4)


def get_terminal_detail_for_part(part):
    """
    Returns a dict with full terminal check detail for reporting.
    """
    required_terminals = part_terminals.get(part, [])
    if not required_terminals:
        return {
            "Terminals_Required": "None",
            "Terminal_Required_Qty": 0,
            "Terminal_Hard_Floor": 0,
            "Terminal_Status": "No terminals required",
            "Terminal_Status_Detail": "No terminals required",
            "Terminal_Blocked": "No",
            "Terminal_Relaxed": "No",
            "Terminal_Coverage_Ratio": 1.0,
            "Terminal_Block_Reason": "—",
        }
    dem_d   = demand_daily_raw.get(part, 0.0)
    ind_d   = indent_daily.get(part, 0.0)
    req_qty = max(dem_d, ind_d)
    r_val   = rate.get(part, 0.0)
    min_run_qty = MIN_RUN_HOURS * r_val if r_val > 0 else 0.0
    effective_required = max(req_qty, min_run_qty)
    hard_floor = effective_required * (1.0 - TERMINAL_RELAXATION_PCT)

    statuses = []
    any_hard  = False
    any_soft  = False
    for t in required_terminals:
        t_inv = terminal_status.get(t, None)
        if t_inv is None:
            statuses.append(f"{t}:MISSING")
            any_hard = True
        elif t_inv < hard_floor:
            statuses.append(f"{t}:inv={t_inv:.0f}:HARD_BLOCK(need>={hard_floor:.0f})")
            any_hard = True
        elif t_inv < effective_required:
            statuses.append(f"{t}:inv={t_inv:.0f}:RELAXED(need>={effective_required:.0f})")
            any_soft = True
        else:
            statuses.append(f"{t}:inv={t_inv:.0f}:OK")

    blocked, reason, relaxed = terminal_blocked(part)
    return {
        "Terminals_Required": ", ".join(required_terminals),
        "Terminal_Required_Qty": round(effective_required, 2),
        "Terminal_Hard_Floor": round(hard_floor, 2),
        "Terminal_Status_Detail": " | ".join(statuses),
        "Terminal_Blocked": "YES" if blocked else "No",
        "Terminal_Relaxed": "YES" if relaxed else "No",
        "Terminal_Coverage_Ratio": terminal_coverage_ratio(part),
        "Terminal_Block_Reason": reason if (blocked or relaxed) else "—",
    }

# =============================================================
# SECTION 7B2 — DEMAND-SORTED PART ORDERING
# =============================================================

def sort_parts_by_demand(parts, exclude_fixed=True):
    """
    Sort parts from largest to smallest demand.
    Fixed parts are excluded (handled in fixed-machine pass).

    Tie-breaking:
      1. Terminal coverage ratio (higher = better)
      2. Current inventory (lower = more urgent)
    """
    fixed_set  = set(part_fixed_machine.keys()) if exclude_fixed else set()
    non_fixed  = [p for p in parts if p not in fixed_set]
    fixed_out  = [p for p in parts if p in fixed_set]

    def _sort_key(p):
        dem   = demand_daily_raw.get(p, 0.0)
        tcov  = terminal_coverage_ratio(p)
        inv_p = inventory.get(p, 0.0)
        return (-dem, -tcov, inv_p)

    non_fixed.sort(key=_sort_key)
    return non_fixed, fixed_out


def get_terminal_eligible_parts(parts, exclude_fixed=True):
    """
    From the full parts list, return only those that pass the terminal constraint.
    Parts are sorted largest demand first (fixed parts excluded).
    Returns (eligible_non_fixed, eligible_fixed, blocked_parts_info)
    """
    fixed_set = set(part_fixed_machine.keys()) if exclude_fixed else set()

    eligible_non_fixed = []
    eligible_fixed     = []
    blocked_info       = []

    non_fixed_parts = [p for p in parts if p not in fixed_set]
    def _sort_key(p):
        dem   = demand_daily_raw.get(p, 0.0)
        tcov  = terminal_coverage_ratio(p)
        inv_p = inventory.get(p, 0.0)
        return (-dem, -tcov, inv_p)
    non_fixed_parts.sort(key=_sort_key)

    for p in non_fixed_parts:
        blocked, reason, relaxed = terminal_blocked(p)
        if blocked:
            blocked_info.append({"Part": p, "Reason": reason, "Fixed": "No"})
        else:
            eligible_non_fixed.append(p)

    for p in [p for p in parts if p in fixed_set]:
        blocked, reason, relaxed = terminal_blocked(p)
        if blocked:
            blocked_info.append({"Part": p, "Reason": reason, "Fixed": "YES (fixed machine)"})
        else:
            eligible_fixed.append(p)

    return eligible_non_fixed, eligible_fixed, blocked_info

# =============================================================
# SECTION 7C — SKIP / ELIGIBILITY RULES
# =============================================================

def should_skip(part):
    """
    Demand-protected: parts with demand > 0 and uncovered stock are NEVER skipped
    by indent/inventory rules — only by HARD terminal blocks.
    """
    ind_daily  = indent_daily.get(part, 0.0)
    dem_daily  = demand_daily_raw.get(part, 0.0)
    eff_daily  = effective_daily(part)
    monthly    = indent_monthly.get(part, 0.0)
    r          = rate.get(part, 1.0)
    inv        = inventory.get(part, 0.0)

    has_demand      = dem_daily > 0.0
    has_demand_gap  = has_demand and inv < dem_daily

    if has_demand_gap:
        t_blocked, t_reason, _ = terminal_blocked(part)
        if t_blocked:
            return True, t_reason
        return False, ""

    if not has_demand:
        if ind_daily <= MIN_DAILY_INDENT:
            return True, f"Daily indent {ind_daily:.2f} <= {MIN_DAILY_INDENT} threshold (no demand)"
        indent_hrs = monthly / r if r > 0 else 0.0
        if indent_hrs <= MIN_INDENT_HOURS:
            return True, f"Monthly indent = {indent_hrs:.2f}h <= {MIN_INDENT_HOURS}h threshold (no demand)"

    if eff_daily > 0 and inv >= TARGET_DAYS * eff_daily:
        if has_demand and not is_demand_met(part, inv, 0):
            return False, ""
        return True, (
            f"Inventory ({inv:.0f}) >= {TARGET_DAYS}-day target "
            f"({TARGET_DAYS * eff_daily:.0f} pcs using eff_daily={eff_daily:.2f}) — at ceiling"
        )

    t_blocked, t_reason, _ = terminal_blocked(part)
    if t_blocked:
        return True, t_reason

    return False, ""


def is_hard_skip(part):
    t_blocked, _, _ = terminal_blocked(part)
    return t_blocked


def is_scheduling_skip(part):
    return should_skip(part)[0]

# =============================================================
# SECTION 7D — CHANGEOVER TIMES
# =============================================================

def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover          = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

# =============================================================
# SECTION 7E — MACHINE PART COUNT
# =============================================================

def build_machine_part_count(df):
    mpc = {}
    machine_col = next((c for c in df.columns if str(c).strip().lower() == "machine"), None)
    count_col   = next((c for c in df.columns if str(c).strip().lower() == "part_count"), None)
    if machine_col is None or count_col is None:
        print("  WARNING: VT_Machine_Part_Count missing columns")
        return {}
    for _, row in df.iterrows():
        m = str(row[machine_col]).strip()
        v = row[count_col]
        if m and pd.notna(v):
            try:
                mpc[m] = int(float(v))
            except (ValueError, TypeError):
                pass
    return mpc

machine_part_count = build_machine_part_count(vt_machine_count_raw)
max_part_count     = max(machine_part_count.values(), default=1) or 1
print(f"  Machine part counts loaded: {len(machine_part_count)} machines")

# =============================================================
# SECTION 7F — PART CATEGORY
# =============================================================

def build_category(df):
    cat = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"), None)
    cat_col  = next((c for c in df.columns if str(c).strip().lower() == "category"), None)
    if part_col is None:
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        val = "Stranger"
        if cat_col and pd.notna(row[cat_col]):
            val = str(row[cat_col]).strip().capitalize()
            if val not in ("Runner", "Repeater", "Stranger"):
                val = "Stranger"
        cat[str(part).strip()] = val
    return cat

part_category  = build_category(vt_parts_raw)
CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}
for p in demand_daily_raw:
    if p not in part_category:
        part_category[p] = "Stranger"

# =============================================================
# SECTION 7G — MAX PARTS PER MACHINE GUARD
# =============================================================

def parts_on_machine(machine, plan):
    """Count distinct parts currently assigned to this machine."""
    return len({r["Part"] for r in plan if r["Machine"] == machine})

def machine_has_capacity_for_new_part(machine, part, plan):
    """
    Enforce _dynamic_max_parts cap (set each run by pre_distribute).
    If part is already on this machine it is an extension — always OK.
    If it is a new part, check the cap has not been reached.
    """
    parts_already = {r["Part"] for r in plan if r["Machine"] == machine}
    if part in parts_already:
        return True
    return len(parts_already) < _dynamic_max_parts

# =============================================================
# SECTION 8 — MACHINE STATE
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        try:
            with open(MACHINE_STATE_FILE) as f:
                content = f.read().strip()
            if not content:
                print(f"  Machine state : file empty — first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            state = json.loads(content)
            if not isinstance(state, dict):
                print(f"  Machine state : corrupt — first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            unknown = [p for p in state.values() if p not in part_color]
            if unknown:
                print(f"  Machine state : {len(unknown)} part(s) not in today's sheet:")
                for p in unknown:
                    ALL_KNOWN_COLORS[p] = "NEEDS_PURGE"
            print(f"  Machine state loaded  ({len(state)} machines)")
            return state
        except Exception as e:
            print(f"  Machine state : error ({e}) — first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
    print(f"  Machine state : FIRST RUN — no changeover today")
    return {}

def save_machine_state(state):
    combined = {m: p for m, p in state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved -> '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)

machine_compatible_parts = {m: [] for m in vt_machines}
for p, machines in vt_compat.items():
    for m in machines:
        if m in machine_compatible_parts:
            machine_compatible_parts[m].append(p)

# =============================================================
# SECTION 10 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = effective_daily(p)
        if daily == 0:
            continue
        coverage.append(inv / daily)
    if not coverage:
        return 3, "SCENARIO 3 — No active parts today"
    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < SAFETY_DAYS)
    if critical == n:
        return 0, f"SCENARIO 0 — ALL {n} active parts critical (inv < 1 day)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical  |  {n-critical} have buffer"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} parts below {SAFETY_DAYS}-day safety floor"
    else:
        return 3, f"SCENARIO 3 — All {n} parts healthy (>={SAFETY_DAYS} days)"

# =============================================================
# SECTION 11 — OPD CAP
# =============================================================

def opd_cap(scenario_id):
    return min(
        {0: OPD_SCENARIO_0, 1: OPD_SCENARIO_1, 2: OPD_SCENARIO_2, 3: OPD_SCENARIO_3}.get(scenario_id, OPD_SCENARIO_2),
        TARGET_DAYS
    )

def effective_opd_cap_qty(part, scenario_id, current_inv):
    eff_d = effective_daily(part)
    dem_d = demand_daily_raw.get(part, 0.0)
    cap_q = opd_cap(scenario_id) * eff_d
    demand_floor_qty = max(0.0, dem_d - current_inv) if dem_d > 0 else 0.0
    return max(cap_q, demand_floor_qty)

# =============================================================
# SECTION 12 — PRIORITY SCORING
# =============================================================

def compute_priority_scores(active_parts):
    rows = []
    for p in active_parts:
        inv      = inventory.get(p, 0)
        daily    = effective_daily(p)
        cat      = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0
        gap_score    = min(1.0, max(0.0, (TARGET_DAYS - days_cov) / TARGET_DAYS))
        velocity_raw = daily / max(float(inv), 1.0) if daily > 0 else 0.0
        has_demand_gap = demand_daily_raw.get(p, 0) > 0 and inv < demand_daily_raw.get(p, 0)
        rows.append({
            "part": p, "inv": inv, "daily": daily, "days_cov": days_cov, "cat": cat,
            "gap_score": gap_score, "velocity_raw": velocity_raw,
            "has_demand_gap": has_demand_gap,
        })
    if not rows:
        return {}, []
    max_daily    = max(r["daily"]        for r in rows) or 1.0
    max_velocity = max(r["velocity_raw"] for r in rows) or 1.0
    scores, score_rows = {}, []
    for r in rows:
        p = r["part"]
        gap_pct      = r["gap_score"] * 100.0
        velocity_pct = (r["velocity_raw"] / max_velocity) * 100.0
        urgency_score  = 0.60 * gap_pct + 0.40 * velocity_pct
        category_score = CATEGORY_SCORE.get(r["cat"], 20)
        indent_score   = (r["daily"] / max_daily) * 100.0
        final_score = (
            W_URGENCY  * urgency_score +
            W_CATEGORY * category_score +
            W_INDENT   * indent_score
        )
        if r["has_demand_gap"]:
            final_score += 200.0
        scores[p] = round(final_score, 2)
        _, t_reason, t_relaxed = terminal_blocked(p)
        inv_today = inventory.get(p, 0)
        dem_d     = demand_daily_raw.get(p, 0.0)
        score_rows.append({
            "Part": p, "Category": r["cat"],
            "Color": part_color.get(p, "UNKNOWN"),
            "Fixed_Machine": part_fixed_machine.get(p, "—"),
            "Tools": tools_available.get(p, 1),
            "Inventory_Now": round(r["inv"], 0),
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
            "Effective_Daily": round(r["daily"], 2),
            "Demand_Driver": demand_driver(p),
            "Demand_Gap_Today": round(max(0.0, dem_d - inv_today), 2),
            "Demand_Covered_By_Stock": "YES" if inv_today >= dem_d and dem_d > 0 else ("N/A" if dem_d == 0 else "NO"),
            "Terminal_Coverage_Ratio": terminal_coverage_ratio(p),
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Terminal_Note": t_reason if t_relaxed else "—",
            "Days_Coverage": round(r["days_cov"], 2),
            "Buffer_Status": (
                "CRITICAL"     if r["days_cov"] < 1 else
                "BELOW_SAFETY" if r["days_cov"] < SAFETY_DAYS else
                "BUILDING"     if r["days_cov"] < TARGET_DAYS else
                "AT_TARGET"
            ),
            "Final_Score": round(final_score, 2),
            "Demand_Boost_Applied": "YES" if r["has_demand_gap"] else "No",
        })
    return scores, score_rows

# =============================================================
# SECTION 13 — COLOUR-AWARE CHANGEOVER HELPER
# =============================================================

def _co_hrs_for(part, machine, machine_last_part):
    last = machine_last_part.get(machine)
    if last is None or last == part:
        return 0.0
    base_co    = vt_changeover.get(machine, DEFAULT_CHANGEOVER_HRS)
    last_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN")
    new_color  = part_color.get(part, "UNKNOWN")
    if last_color == "NEEDS_PURGE":
        return base_co + COLOR_PURGE_HRS
    purge = (
        COLOR_PURGE_HRS
        if last_color != new_color and last_color not in ("UNKNOWN",) and new_color not in ("UNKNOWN",)
        else 0.0
    )
    return base_co + purge

# =============================================================
# SECTION 14 — MACHINE RANKER (3-part cap integrated)
# =============================================================

def rank_machines(part, machines_to_try, machine_hours,
                  machine_last_part, inv_days, plan,
                  exclude_fixed_machines=True,
                  allow_fixed_overflow=False):
    category  = part_category.get(part, "Stranger")
    new_color = part_color.get(part, "UNKNOWN")
    fixed_m   = part_fixed_machine.get(part)
    is_fixed  = fixed_m is not None
    runner_lock = (category == "Runner" and inv_days < RUNNER_PRIORITY_DAYS and not is_fixed)

    effective_candidates = []
    for m in machines_to_try:
        if exclude_fixed_machines and not allow_fixed_overflow and m in machine_fixed_parts:
            if not is_fixed:
                continue
            elif m != fixed_m:
                continue
        effective_candidates.append(m)

    if is_fixed and fixed_m in effective_candidates:
        used_f = machine_hours.get(fixed_m, 0)
        free_f = round(AVAILABLE_HOURS - used_f, 4)
        co_f   = _co_hrs_for(part, fixed_m, machine_last_part)
        eff_f  = round(free_f - co_f, 4)
        if eff_f >= MIN_RUN_HOURS and machine_has_capacity_for_new_part(fixed_m, part, plan):
            fallback = _rank_normal(
                part, [m for m in effective_candidates if m != fixed_m],
                machine_hours, machine_last_part, new_color, runner_lock, plan
            )
            return [(fixed_m, co_f, eff_f, -1.0)] + fallback, runner_lock
        remaining = [m for m in effective_candidates if m != fixed_m]
        return _rank_normal(part, remaining, machine_hours, machine_last_part, new_color, runner_lock, plan), runner_lock

    return _rank_normal(part, effective_candidates, machine_hours, machine_last_part, new_color, runner_lock, plan), runner_lock


def _rank_normal(part, machines_to_try, machine_hours,
                 machine_last_part, new_color, runner_lock, plan):
    ranked = []
    for m in machines_to_try:
        if not machine_has_capacity_for_new_part(m, part, plan):
            continue
        used = machine_hours.get(m, 0)
        free = round(AVAILABLE_HOURS - used, 4)
        if free < MIN_RUN_HOURS:
            continue
        last = machine_last_part.get(m)
        if runner_lock and last != part:
            continue
        if last is None or last == part:
            co_hrs = 0.0; color_bonus = 0.0
        else:
            base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
            last_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN")
            same_color = (last_color == new_color and last_color not in ("UNKNOWN", "NEEDS_PURGE") and new_color not in ("UNKNOWN",))
            purge      = 0.0 if same_color else (COLOR_PURGE_HRS if last_color not in ("UNKNOWN",) and new_color not in ("UNKNOWN",) else 0.0)
            co_hrs     = base_co + purge
            color_bonus = -0.08 if same_color else 0.0
        effective_free = round(free - co_hrs, 4)
        if effective_free < MIN_RUN_HOURS:
            continue
        part_count      = machine_part_count.get(m, max_part_count)
        count_score     = part_count / max_part_count
        co_penalty      = (co_hrs / AVAILABLE_HOURS) * 0.3
        util_penalty    = (used  / AVAILABLE_HOURS) * 0.2
        same_part_bonus = -0.15 if (last == part) else 0.0
        cost            = count_score + co_penalty + util_penalty + same_part_bonus + color_bonus
        ranked.append((m, co_hrs, effective_free, cost))
    ranked.sort(key=lambda x: x[3])
    return ranked

_phase_a_machines: set = set()

# =============================================================
# SECTION 14A — PLAN HELPERS
# =============================================================

def _get_part_total_qty(part, plan):
    return sum(float(r.get("Production_Qty", 0)) for r in plan if r["Part"] == part)

def _get_part_machines(part, plan):
    return {r["Machine"] for r in plan if r["Part"] == part}

def _is_indent_met(part, plan):
    daily = effective_daily(part)
    if daily <= 0:
        return True
    total_qty = _get_part_total_qty(part, plan)
    inv_now   = inventory.get(part, 0)
    return (inv_now + total_qty) >= (daily - 0.5)

def _parts_with_indent_met(plan):
    part_qty = defaultdict(float)
    for row in plan:
        part_qty[row["Part"]] += float(row.get("Production_Qty", 0))
    result = set()
    for p, qty in part_qty.items():
        daily   = effective_daily(p)
        inv_now = inventory.get(p, 0)
        if daily > 0 and (inv_now + qty) >= (daily - 0.5):
            result.add(p)
    return result

def _current_co_count(plan):
    return sum(1 for r in plan if r.get("Changeover") == "Yes")

def _make_plan_row(part, machine, run_hrs, co_hrs, qty, scenario_id,
                   type_label, role_label, runner_lock=False, phase=1):
    """Unified plan-row builder."""
    daily    = effective_daily(part)
    monthly  = effective_monthly(part)
    # V17: use effective rate for this machine
    base_r   = rate.get(part, 1)
    r_val    = effective_rate(part, machine)
    inv_now  = inventory.get(part, 0)
    color    = part_color.get(part, "UNKNOWN")
    fixed_m  = part_fixed_machine.get(part)
    last     = machine_state.get(machine)
    l_color  = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
    has_purge = (last is not None and last != part and color != l_color
                 and color != "UNKNOWN" and l_color not in ("UNKNOWN", "NEEDS_PURGE"))
    fixed_used = "YES" if (fixed_m and machine == fixed_m) else ("FALLBACK" if fixed_m else "N/A")
    dem_d = demand_daily_raw.get(part, 0.0)
    indent_met_flag = (
        "YES" if (inv_now + qty) >= (daily - 0.5)
        else f"NO — need {daily:.2f}/d, have {inv_now+qty:.0f} pcs"
    )
    demand_met_flag = "YES" if is_demand_met(part, inv_now, qty) else f"NO — need {dem_d:.2f}/d"
    _, t_reason, t_relaxed = terminal_blocked(part)
    # V17 fields
    mrun_label = machine_run_label(part, machine)
    tools_cnt  = tools_available.get(part, 1)
    return {
        "Part": part,
        "Color": color,
        "Category": part_category.get(part, "Stranger"),
        "Fixed_Machine": fixed_m or "—",
        "Fixed_Used": fixed_used,
        "Machine": machine,
        "Machine_Run_Type": machine_run_type.get(machine, "HALF"),   # V17
        "Run_Mode": mrun_label,                                        # V17
        "Run_Hours": round(run_hrs, 3),
        "Changeover_Hrs": round(co_hrs, 3),
        "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
        "Base_Rate_Per_Hour": round(base_r, 2),                        # V17
        "Rate_Per_Hour": round(r_val, 2),                              # V17: effective (may be 2×)
        "Production_Qty": qty,
        "Inventory_Before": round(inv_now, 0),
        "Demand_Today_Required": round(max(0.0, dem_d - inv_now), 2),
        "Demand_Daily": round(dem_d, 2),
        "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
        "Effective_Daily": round(daily, 2),
        "Demand_Driver": demand_driver(part),
        "Monthly_Indent": round(monthly, 0),
        "Today_Target": round(today_target_qty.get(part, 0), 0),
        "Changeover": "No" if co_hrs == 0 else "Yes",
        "Color_Purge": "Yes" if has_purge else "No",
        "Terminal_Relaxed": "YES" if t_relaxed else "No",
        "Terminal_Note": t_reason if t_relaxed else "—",
        "Type": type_label,
        "Role": role_label,
        "Tools_Available": tools_cnt,
        "Tools_Used": 2 if (machine_run_type.get(machine, "HALF") == "FULL" and tools_cnt >= 2) else 1,  # V17
        "Runner_Lock": "YES" if runner_lock else "No",
        "Priority_Score": 0,
        "Phase": phase,
        "Indent_Met": indent_met_flag,
        "Demand_Met": demand_met_flag,
        "Stagger_Adjusted": "No",
    }

# =============================================================
# SECTION 14B — INTRA-MACHINE CO RESEQUENCING
# =============================================================

def resequence_machine_rows(plan, machine_last_part_yesterday):
    machine_rows = defaultdict(list)
    other_rows   = []
    for row in plan:
        m = row.get("Machine")
        if m in vt_machines:
            machine_rows[m].append(row)
        else:
            other_rows.append(row)
    resequenced_plan = []
    for m in vt_machines:
        rows = machine_rows.get(m, [])
        if len(rows) <= 1:
            resequenced_plan.extend(rows)
            continue
        yesterday_part = machine_last_part_yesterday.get(m)
        ordered   = []
        remaining = list(rows)
        seed = None
        if yesterday_part:
            for r in remaining:
                if r["Part"] == yesterday_part:
                    seed = r
                    break
        if seed is None:
            seed = max(remaining, key=lambda r: float(r.get("Priority_Score", 0) or 0))
        ordered.append(seed)
        remaining.remove(seed)
        while remaining:
            last_part      = ordered[-1]["Part"]
            last_color_val = part_color.get(last_part, "UNKNOWN")
            def _co_cost(r, _lc=last_color_val):
                p = r["Part"]
                c = part_color.get(p, "UNKNOWN")
                if p == last_part:
                    return -1.0
                base  = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
                purge = (COLOR_PURGE_HRS if _lc not in ("UNKNOWN", "NEEDS_PURGE") and c not in ("UNKNOWN",) and _lc != c else 0.0)
                return base + purge
            remaining.sort(key=_co_cost)
            ordered.append(remaining.pop(0))
        for i, row in enumerate(ordered):
            p         = row["Part"]
            prev_part = yesterday_part if i == 0 else ordered[i - 1]["Part"]
            if prev_part is None or prev_part == p:
                new_co = 0.0
            else:
                base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
                prev_color = ALL_KNOWN_COLORS.get(prev_part, "UNKNOWN")
                new_color  = part_color.get(p, "UNKNOWN")
                purge = (COLOR_PURGE_HRS if prev_color not in ("UNKNOWN", "NEEDS_PURGE") and new_color not in ("UNKNOWN",) and prev_color != new_color else 0.0)
                new_co = base_co + purge
            row["Changeover_Hrs"]  = round(new_co, 3)
            row["Changeover"]      = "No" if new_co == 0 else "Yes"
            row["Total_Hrs_Used"]  = round(new_co + float(row.get("Run_Hours", 0) or 0), 3)
            row["Color_Purge"] = "Yes" if new_co > vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS) + 0.001 else "No"
        resequenced_plan.extend(ordered)
    resequenced_plan.extend(other_rows)
    return resequenced_plan

# =============================================================
# SECTION 14C — MACHINE HOURS RECONCILER
# =============================================================

def reconcile_machine_hours(plan, machine_hours):
    recomputed = {m: 0.0 for m in vt_machines}
    for row in plan:
        m = row.get("Machine")
        if m in recomputed:
            recomputed[m] += float(row.get("Run_Hours", 0) or 0) + float(row.get("Changeover_Hrs", 0) or 0)
    for m in vt_machines:
        machine_hours[m] = round(recomputed[m], 4)

# =============================================================
# SECTION 15 — PASS 1: DEMAND-FIRST ASSIGNMENT
# =============================================================

def assign_demand_for_part(part, scenario_id, machine_hours, machine_last_part,
                            current_inventory, plan, already_planned, priority_scores):
    """
    Exclusively satisfies today's demand gap. Runs BEFORE inventory building.
    Sorted by demand (largest first) before calling this function.
    """
    daily      = effective_daily(part)
    inv_now    = current_inventory.get(part, 0)
    inv_before = inventory.get(part, 0)
    dem_d      = demand_daily_raw.get(part, 0.0)
    dem_gap    = max(0.0, dem_d - inv_now)

    if dem_gap <= 0:
        return []

    compatible = vt_compat.get(part, [])
    if not compatible:
        return []

    fixed_m = part_fixed_machine.get(part)
    _, t_note, t_relaxed = terminal_blocked(part)

    if fixed_m:
        ordered = [fixed_m] if fixed_m in compatible else []
    else:
        ordered = [m for m in compatible if m not in machine_fixed_parts]

    new_rows = []
    produced = 0.0
    used_machines_demand = set()          # FIX-2: prevent same machine twice

    for m in ordered:
        if produced >= dem_gap - 0.5:
            break
        if used_machines_demand:          # V16: only block 2nd machine if demand already met
            inv_after_first = inventory.get(part, 0) + produced
            if is_demand_met(part, inventory.get(part, 0), produced):
                break  # demand satisfied — single machine is enough
            # demand still unmet — allow second machine to help
        if m in used_machines_demand:     # FIX-2: belt-and-suspenders guard
            continue
        if not machine_has_capacity_for_new_part(m, part, plan):
            continue
        if m in _phase_a_machines and m != fixed_m:
            continue
        co_hrs = _co_hrs_for(part, m, machine_last_part)
        if co_hrs > 0 and _current_co_count(plan) >= MAX_DAILY_CO and inv_now > 0:
            continue
        free   = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        eff    = round(free - co_hrs, 4)
        if eff < MIN_RUN_HOURS:
            continue

        # V17: use effective rate for this machine
        r_val = effective_rate(part, m)
        if r_val <= 0:
            continue

        remain_dem = dem_gap - produced
        run_hrs = max(MIN_RUN_HOURS, min(eff, remain_dem / r_val if r_val > 0 else MIN_RUN_HOURS))
        qty     = round(run_hrs * r_val, 0)

        machine_hours[m]        = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
        current_inventory[part] = round(current_inventory.get(part, 0) + qty, 0)
        machine_last_part[m]    = part
        produced               += qty
        used_machines_demand.add(m)       # FIX-2/3: record machine used

        last    = machine_state.get(m)
        l_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
        p_color = part_color.get(part, "UNKNOWN")
        has_purge = (last is not None and last != part and p_color != l_color
                     and p_color != "UNKNOWN" and l_color not in ("UNKNOWN", "NEEDS_PURGE"))

        dem_met_flag = "YES" if (inv_before + produced) >= (dem_d - 0.5) else f"NO — need {max(0,dem_d-(inv_before+produced)):.0f} more"
        ind_met_flag = "YES" if (inv_before + produced) >= (daily - 0.5) else f"NO — need {daily:.2f}/d"

        # V17 fields
        base_r     = rate.get(part, 1)
        mrun_label = machine_run_label(part, m)
        tools_cnt  = tools_available.get(part, 1)

        new_rows.append({
            "Part": part,
            "Color": p_color,
            "Category": part_category.get(part, "Stranger"),
            "Fixed_Machine": fixed_m or "—",
            "Fixed_Used": "YES" if (fixed_m and m == fixed_m) else ("FALLBACK" if fixed_m else "N/A"),
            "Machine": m,
            "Machine_Run_Type": machine_run_type.get(m, "HALF"),    # V17
            "Run_Mode": mrun_label,                                   # V17
            "Run_Hours": round(run_hrs, 3),
            "Changeover_Hrs": round(co_hrs, 3),
            "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
            "Base_Rate_Per_Hour": round(base_r, 2),                   # V17
            "Rate_Per_Hour": round(r_val, 2),
            "Production_Qty": qty,
            "Inventory_Before": round(inv_before, 0),
            "Demand_Today_Required": round(dem_gap, 2),
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(part),
            "Monthly_Indent": round(effective_monthly(part), 0),
            "Today_Target": round(today_target_qty.get(part, 0), 0),
            "Changeover": "No" if co_hrs == 0 else "Yes",
            "Color_Purge": "Yes" if has_purge else "No",
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Terminal_Note": t_note if t_relaxed else "—",
            "Type": "DEMAND-FIRST" + (" [TERMINAL-RELAXED]" if t_relaxed else ""),
            "Role": "Primary",
            "Tools_Available": tools_cnt,
            "Tools_Used": 2 if (machine_run_type.get(m, "HALF") == "FULL" and tools_cnt >= 2) else 1,  # V17
            "Runner_Lock": "No",
            "Priority_Score": priority_scores.get(part, 0),
            "Phase": 0,
            "Indent_Met": ind_met_flag,
            "Demand_Met": dem_met_flag,
            "Stagger_Adjusted": "No",
        })

    if new_rows:
        already_planned.add(part)
    return new_rows

# =============================================================
# SECTION 16 — PASS 2: INVENTORY BUILD + OPD
# =============================================================

def assign_inventory_build(part, scenario_id, machine_hours, machine_last_part,
                            current_inventory, plan, already_planned, priority_scores):
    """
    Runs after demand pass. Builds inventory up to OPD cap.
    """
    daily      = effective_daily(part)
    inv_now    = current_inventory.get(part, 0)
    inv_before = inventory.get(part, 0)
    compatible = vt_compat.get(part, [])
    fixed_m    = part_fixed_machine.get(part)
    _, t_note, t_relaxed = terminal_blocked(part)

    if not compatible:
        return []

    if fixed_m:
        machines_to_rank = [fixed_m] if fixed_m in compatible else []
    else:
        machines_to_rank = [m for m in compatible if m not in machine_fixed_parts]

    if not machines_to_rank:
        return []

    inv_days        = inv_now / daily if daily > 0 else 999
    total_shortfall = max(0.0, daily - inv_now)

    new_rows      = []
    produced      = 0.0
    tools_used    = 0
    used_machines = set()

    for r in plan:
        if r["Part"] == part:
            produced      += float(r.get("Production_Qty", 0))
            used_machines.add(r["Machine"])
            tools_used    += 1

    if produced >= total_shortfall - 0.5:
        _do_inv_build_on_existing(part, plan, scenario_id, machine_hours,
                                   current_inventory, daily, inv_now)
        return []

    ranked, runner_lock = rank_machines(
        part, machines_to_rank, machine_hours, machine_last_part, inv_days, plan
    )

    tools_cat = tools_available.get(part, 1)
    category  = part_category.get(part, "Stranger")
    is_critical = (inv_now == 0)
    # FIX-3: always limit to 1 machine per part regardless of category
    tool_hard_cap = 1

    for m, co, eff, _ in ranked:
        if produced >= total_shortfall - 0.5:
            break
        if tools_used >= tool_hard_cap:
            break
        if m in used_machines:
            continue
        if m in _phase_a_machines and m != fixed_m:
            continue
        if co > 0 and _current_co_count(plan) >= MAX_DAILY_CO:
            continue

        # V17: effective rate for this machine
        r_val = effective_rate(part, m)
        if r_val <= 0:
            continue

        hrs_for_full = max(MIN_RUN_HOURS, total_shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
        remain = total_shortfall - produced
        run_hrs = max(MIN_RUN_HOURS, min(eff, remain / r_val if r_val > 0 else MIN_RUN_HOURS))
        qty     = round(run_hrs * r_val, 0)

        machine_hours[m]        = round(machine_hours.get(m, 0) + co + run_hrs, 4)
        current_inventory[part] = round(current_inventory.get(part, 0) + qty, 0)
        machine_last_part[m]    = part
        produced               += qty
        tools_used             += 1
        used_machines.add(m)

        last    = machine_state.get(m)
        l_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
        p_color = part_color.get(part, "UNKNOWN")
        has_purge = (last is not None and last != part and p_color != l_color
                     and p_color != "UNKNOWN" and l_color not in ("UNKNOWN", "NEEDS_PURGE"))
        fixed_used_str = "YES" if (fixed_m and m == fixed_m) else ("FALLBACK" if fixed_m else "N/A")
        dem_d = demand_daily_raw.get(part, 0.0)
        type_tag = "Indent-Build" + (" [TERMINAL-RELAXED]" if t_relaxed else "")

        # V17 fields
        base_r     = rate.get(part, 1)
        mrun_label = machine_run_label(part, m)
        tools_cnt  = tools_available.get(part, 1)

        new_rows.append({
            "Part": part,
            "Color": p_color,
            "Category": category,
            "Fixed_Machine": fixed_m or "—",
            "Fixed_Used": fixed_used_str,
            "Machine": m,
            "Machine_Run_Type": machine_run_type.get(m, "HALF"),    # V17
            "Run_Mode": mrun_label,                                   # V17
            "Run_Hours": round(run_hrs, 3),
            "Changeover_Hrs": round(co, 3),
            "Total_Hrs_Used": round(co + run_hrs, 3),
            "Base_Rate_Per_Hour": round(base_r, 2),                   # V17
            "Rate_Per_Hour": round(r_val, 2),
            "Production_Qty": qty,
            "Inventory_Before": round(inv_before, 0),
            "Demand_Today_Required": round(max(0.0, dem_d - inv_before), 2),
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(part),
            "Monthly_Indent": round(effective_monthly(part), 0),
            "Today_Target": round(today_target_qty.get(part, 0), 0),
            "Changeover": "No" if co == 0 else "Yes",
            "Color_Purge": "Yes" if has_purge else "No",
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Terminal_Note": t_note if t_relaxed else "—",
            "Type": type_tag,
            "Role": f"Tool-Expansion (tool {tools_used})" if tools_used > 1 else "Primary",
            "Tools_Available": tools_cnt,
            "Tools_Used": 2 if (machine_run_type.get(m, "HALF") == "FULL" and tools_cnt >= 2) else 1,  # V17
            "Runner_Lock": "YES" if runner_lock else "No",
            "Priority_Score": priority_scores.get(part, 0),
            "Phase": 1,
            "Indent_Met": "YES" if (inv_before + produced) >= (daily - 0.5) else f"NO — need {daily:.2f}/d",
            "Demand_Met": "YES" if is_demand_met(part, inv_before, produced) else f"NO — need {dem_d:.2f}/d",
            "Stagger_Adjusted": "No",
        })
        print(f"      INDENT-BUILD {part:26s} tool {tools_used}/{tool_hard_cap} -> {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}"
              f"  [{machine_run_type.get(m,'HALF')}]")

    if new_rows:
        already_planned.add(part)
        plan.extend(new_rows)
        _do_inv_build_on_existing(
            part, plan, scenario_id, machine_hours, current_inventory,
            effective_daily(part), current_inventory.get(part, 0)
        )
    return new_rows


def _do_inv_build_on_existing(part, plan, scenario_id, machine_hours,
                               current_inventory, daily, inv_after):
    """Extend the first plan row for this part to reach OPD cap."""
    cap_qty  = effective_opd_cap_qty(part, scenario_id, inv_after)
    headroom = max(0.0, cap_qty - inv_after)
    if headroom <= 0:
        return
    target_row = next((r for r in plan if r["Part"] == part), None)
    if target_row is None:
        return
    m = target_row["Machine"]
    if m in _phase_a_machines:
        return
    free_m = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
    if free_m < 0.05:
        return
    # V17: effective rate for this machine
    r_val = effective_rate(part, m)
    if r_val <= 0:
        return
    extend_hrs = min(free_m, headroom / r_val if r_val > 0 else 0)
    if extend_hrs < 0.05:
        return
    extra_qty = round(extend_hrs * r_val, 0)
    target_row["Run_Hours"]      = round(float(target_row["Run_Hours"]) + extend_hrs, 3)
    target_row["Total_Hrs_Used"] = round(float(target_row["Changeover_Hrs"]) + float(target_row["Run_Hours"]), 3)
    target_row["Production_Qty"] = round(float(target_row["Production_Qty"]) + extra_qty, 0)
    target_row["Type"] = str(target_row["Type"]) + "+OPD-Build"
    machine_hours[m]        = round(machine_hours.get(m, 0) + extend_hrs, 4)
    current_inventory[part] = round(current_inventory.get(part, 0) + extra_qty, 0)
    print(f"      OPD-BUILD  {part:25s} on {m:15s}  +{extend_hrs:.2f}h  qty+={extra_qty:.0f}")

# =============================================================
# SECTION 17 — FIXED MACHINE SCHEDULING PASS
# =============================================================

def schedule_fixed_machines(machine_hours, machine_last_part,
                              current_inventory, plan, already_planned,
                              priority_scores, scenario_id):
    print("\n" + "─"*65)
    print(f"  FIXED MACHINE SCHEDULING PASS")
    print("─"*65)

    phase_a_machines = set()
    fixed_plan_rows  = []

    for machine, fixed_parts in sorted(machine_fixed_parts.items()):
        phase = fixed_machine_phase(machine, current_inventory)
        inv_summary = []
        for p in fixed_parts:
            daily    = effective_daily(p)
            inv      = current_inventory.get(p, 0)
            days_cov = inv / daily if daily > 0 else 999
            r_val    = effective_rate(p, machine)   # V17
            hrs_need = daily / r_val if r_val > 0 else 0
            _, t_note, t_relaxed = terminal_blocked(p)
            inv_summary.append(
                f"{p}(inv={inv:.0f}={days_cov:.2f}d, need={hrs_need:.1f}h/d, "
                f"driver={demand_driver(p)}" + (" [RELAX]" if t_relaxed else "") + ")"
            )
        print(f"\n  Machine: {machine}  Phase: {phase}  [{machine_run_type.get(machine,'HALF')} RUN]")
        print(f"    Fixed parts: {' | '.join(inv_summary)}")

        if phase == "A":
            phase_a_machines.add(machine)
            chosen = pick_fixed_part_for_today(machine, current_inventory, machine_last_part)
            if chosen is None:
                print(f"    No eligible fixed part — machine skipped")
                continue
            # V16: fixed machines always run — terminal shortfall becomes a reminder
            t_blk_a, t_note, t_relaxed = terminal_blocked(chosen)
            terminal_reminder_a = None
            if t_blk_a:
                dem_d_rem = demand_daily_raw.get(chosen, 0.0)
                ind_d_rem = indent_daily.get(chosen, 0.0)
                r_rem = effective_rate(chosen, machine)   # V17
                req_rem = max(dem_d_rem, ind_d_rem, MIN_RUN_HOURS * r_rem)
                terminal_reminder_a = (
                    f"REMINDER: Terminal stock insufficient for {chosen} — "
                    f"need ~{req_rem:.0f} pcs. {t_note[:80]}"
                )
                print(f"    Phase A TERMINAL REMINDER (fixed machine continues) -> {chosen}: {t_note[:60]}")
                t_blk_a = False  # override for fixed machines — never stop them
            daily   = effective_daily(chosen)
            r_val   = effective_rate(chosen, machine)   # V17
            score   = priority_scores.get(chosen, 0)
            inv_before_chosen = inventory.get(chosen, 0)
            run_hrs_cap = fixed_part_run_hours(chosen, machine, "A")   # V17: pass machine
            qty = round(run_hrs_cap * r_val, 0)
            machine_hours[machine]      = round(run_hrs_cap, 4)
            current_inventory[chosen]   = round(current_inventory.get(chosen, 0) + qty, 0)
            machine_last_part[machine]  = chosen
            already_planned.add(chosen)
            dem_d = demand_daily_raw.get(chosen, 0.0)
            row = _make_plan_row(chosen, machine, run_hrs_cap, 0.0, qty, scenario_id,
                                 f"Fixed-PhaseA [{run_hrs_cap}h cap]", "Primary", phase=1)
            row["Priority_Score"]   = score
            row["Inventory_Before"] = round(inv_before_chosen, 0)
            row.setdefault("Terminal_Reminder", "—")
            if terminal_reminder_a:
                row["Terminal_Reminder"] = terminal_reminder_a
                row["Terminal_Relaxed"]  = "YES (Fixed Override)"
            plan.append(row)
            fixed_plan_rows.append(row)
            inv_after  = current_inventory.get(chosen, 0)
            days_after = inv_after / daily if daily > 0 else 0
            print(f"    Phase A -> {chosen}  {run_hrs_cap:.2f}h  qty={qty:.0f}  "
                  f"inv_after={inv_after:.0f} ({days_after:.2f}d)  driver={demand_driver(chosen)}"
                  + (" [TERMINAL RELAXED]" if t_relaxed else "")
                  + f"  [{machine_run_type.get(machine,'HALF')} RUN]")

        else:
            for chosen in fixed_parts:
                if machine_hours.get(machine, 0) >= AVAILABLE_HOURS - 0.05:
                    already_planned.add(chosen)
                    continue
                # V16: fixed machines always run — terminal shortfall becomes a reminder
                t_blk_b, t_note_b, _ = terminal_blocked(chosen)
                terminal_reminder_b = None
                if t_blk_b:
                    dem_d_rem_b = demand_daily_raw.get(chosen, 0.0)
                    ind_d_rem_b = indent_daily.get(chosen, 0.0)
                    r_rem_b = effective_rate(chosen, machine)   # V17
                    req_rem_b = max(dem_d_rem_b, ind_d_rem_b, MIN_RUN_HOURS * r_rem_b)
                    terminal_reminder_b = (
                        f"REMINDER: Terminal stock insufficient for {chosen} — "
                        f"need ~{req_rem_b:.0f} pcs. {t_note_b[:80]}"
                    )
                    print(f"    Phase B TERMINAL REMINDER (fixed machine continues) -> {chosen}: {t_note_b[:60]}")
                    t_blk_b = False  # override for fixed machines
                daily   = effective_daily(chosen)
                r_val   = effective_rate(chosen, machine)   # V17
                score   = priority_scores.get(chosen, 0)
                inv_before_chosen = inventory.get(chosen, 0)
                min_run_for_indent = fixed_part_run_hours(chosen, machine, "B")   # V17
                available_now = round(AVAILABLE_HOURS - machine_hours.get(machine, 0), 4)
                effective_run = min(min_run_for_indent, available_now)
                if effective_run < MIN_RUN_HOURS:
                    already_planned.add(chosen)
                    continue
                inv_now_chosen = current_inventory.get(chosen, 0)
                cap_qty  = effective_opd_cap_qty(chosen, scenario_id, inv_now_chosen)
                headroom = max(0.0, cap_qty - inv_now_chosen)
                if headroom > 0 and r_val > 0:
                    effective_run = min(available_now, max(effective_run, headroom / r_val))
                    effective_run = max(effective_run, MIN_RUN_HOURS)
                qty = round(effective_run * r_val, 0)
                machine_hours[machine]    = round(machine_hours.get(machine, 0) + effective_run, 4)
                current_inventory[chosen] = round(current_inventory.get(chosen, 0) + qty, 0)
                machine_last_part[machine] = chosen
                already_planned.add(chosen)
                _, t_note, t_relaxed = terminal_blocked(chosen)
                row = _make_plan_row(chosen, machine, effective_run, 0.0, qty, scenario_id,
                                     "Fixed-PhaseB", "Primary", phase=1)
                row["Priority_Score"]   = score
                row["Inventory_Before"] = round(inv_before_chosen, 0)
                row.setdefault("Terminal_Reminder", "—")
                if terminal_reminder_b:
                    row["Terminal_Reminder"] = terminal_reminder_b
                    row["Terminal_Relaxed"]  = "YES (Fixed Override)"
                plan.append(row)
                fixed_plan_rows.append(row)
                remaining_hrs = round(AVAILABLE_HOURS - machine_hours.get(machine, 0), 4)
                print(f"    Phase B -> {chosen}  {effective_run:.2f}h  qty={qty:.0f}  "
                      f"remaining={remaining_hrs:.2f}h  driver={demand_driver(chosen)}"
                      + (" [TERMINAL RELAXED]" if t_relaxed else "")
                      + f"  [{machine_run_type.get(machine,'HALF')} RUN]")

    print(f"\n  Fixed machine pass: {len(phase_a_machines)} Phase A  |  "
          f"{len(machine_fixed_parts) - len(phase_a_machines)} Phase B")
    return phase_a_machines, fixed_plan_rows

# =============================================================
# SECTION 18 — RUNNER PRIORITY ENFORCEMENT
# =============================================================

def enforce_runner_priority(plan, machine_hours, machine_last_part,
                             current_inventory, already_planned,
                             priority_scores, inventory_start_of_day):
    print("\n" + "─"*65)
    print(f"  RUNNER PRIORITY ENFORCEMENT (threshold: < {RUNNER_PRIORITY_DAYS} days)")
    print("─"*65)
    runner_priority_log = []

    critical_runners = []
    for part in vt_compat:
        if part in already_planned:
            continue
        if part_category.get(part, "Stranger") != "Runner":
            continue
        daily = effective_daily(part)
        r_val = rate.get(part, 1)
        if daily <= 0 or r_val <= 0:
            continue
        inv_now = current_inventory.get(part, 0)
        if inv_now / daily >= RUNNER_PRIORITY_DAYS:
            continue
        skip, _ = should_skip(part)
        if skip:
            continue
        if not vt_compat.get(part):
            continue
        critical_runners.append(part)

    if not critical_runners:
        print(f"  No critical unplanned Runners found")
        return runner_priority_log

    critical_runners.sort(key=lambda p: effective_daily(p), reverse=True)
    print(f"  Critical Runners: {len(critical_runners)}")

    for runner in critical_runners:
        r_daily   = effective_daily(runner)
        r_inv     = current_inventory.get(runner, 0)
        r_inv_sod = inventory_start_of_day.get(runner, r_inv)
        r_inv_b   = inventory.get(runner, 0)
        r_color   = part_color.get(runner, "UNKNOWN")
        r_score   = priority_scores.get(runner, 0)
        r_monthly = effective_monthly(runner)
        fixed_m   = part_fixed_machine.get(runner)
        r_days_now = r_inv / r_daily if r_daily > 0 else 0
        r_days_sod = r_inv_sod / r_daily if r_daily > 0 else 0
        _, t_note, t_relaxed = terminal_blocked(runner)
        dem_d = demand_daily_raw.get(runner, 0.0)

        shortfall_qty = max(0.0, r_daily - r_inv)

        machines_runner_on = {row["Machine"] for row in plan if row["Part"] == runner}
        if len(machines_runner_on) >= tools_available.get(runner, 1):
            runner_priority_log.append({
                "Runner_Part": runner, "Demand_Driver": demand_driver(runner),
                "Runner_Days_SOD": round(r_days_sod, 2),
                "Runner_Days_Now": round(r_days_now, 2), "Fixed_Machine": fixed_m or "—",
                "Fixed_Used": "FAILED — tool cap",
                "Runner_Daily": round(r_daily, 2),
                "Runner_Inv_SOD": round(r_inv_sod, 0),
                "Runner_Shortfall": round(shortfall_qty, 0),
                "Machine_Assigned": "—", "Hours_Needed": 0,
                "Hours_Assigned": 0, "Qty_Produced": 0, "Displacement_Used": "N/A",
                "Victims": "—", "Total_Hours_Reclaimed": "—",
                "Terminal_Relaxed": "YES" if t_relaxed else "No",
                "Result": "FAILED — all tools deployed",
            })
            continue

        compatible_machines = list(vt_compat.get(runner, []))
        if fixed_m and fixed_m in compatible_machines:
            ordered_machines = [fixed_m] + [m for m in compatible_machines if m != fixed_m]
        else:
            ordered_machines = [m for m in compatible_machines if m not in machine_fixed_parts]

        best_machine    = None
        best_co_hrs     = 0.0
        best_victims    = []
        best_disruption = float("inf")

        for m in ordered_machines:
            if not machine_has_capacity_for_new_part(m, runner, plan):
                continue
            # V17: effective rate for this machine
            r_rate = effective_rate(runner, m)
            if r_rate <= 0:
                continue
            hours_needed = max(MIN_RUN_HOURS, shortfall_qty / r_rate if r_rate > 0 else MIN_RUN_HOURS)

            co_hrs   = _co_hrs_for(runner, m, machine_last_part)
            used_hrs = machine_hours.get(m, 0)
            free_hrs = round(AVAILABLE_HOURS - used_hrs, 4)
            eff_free = round(free_hrs - co_hrs, 4)

            if eff_free >= hours_needed:
                run_h = max(MIN_RUN_HOURS, min(eff_free, hours_needed))
                qty   = round(run_h * r_rate, 0)
                fixed_used = (fixed_m is not None and m == fixed_m)
                machine_hours[m]        = round(used_hrs + co_hrs + run_h, 4)
                current_inventory[runner] = round(current_inventory.get(runner, 0) + qty, 0)
                machine_last_part[m]    = runner
                already_planned.add(runner)
                purge = co_hrs > vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS) + 0.001
                dem_met = is_demand_met(runner, r_inv_b, qty)
                # V17 fields
                base_r_run = rate.get(runner, 1)
                mrun_label = machine_run_label(runner, m)
                tools_cnt  = tools_available.get(runner, 1)
                plan.append({
                    "Part": runner, "Color": r_color, "Category": "Runner",
                    "Fixed_Machine": fixed_m or "—",
                    "Fixed_Used": "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
                    "Machine": m,
                    "Machine_Run_Type": machine_run_type.get(m, "HALF"),    # V17
                    "Run_Mode": mrun_label,                                   # V17
                    "Run_Hours": round(run_h, 3),
                    "Changeover_Hrs": round(co_hrs, 3),
                    "Total_Hrs_Used": round(co_hrs + run_h, 3),
                    "Base_Rate_Per_Hour": round(base_r_run, 2),               # V17
                    "Rate_Per_Hour": round(r_rate, 2),
                    "Production_Qty": qty,
                    "Inventory_Before": round(r_inv_b, 0),
                    "Demand_Today_Required": round(max(0.0, dem_d - r_inv_b), 2),
                    "Demand_Daily": round(dem_d, 2),
                    "Indent_Daily": round(indent_daily.get(runner, 0.0), 2),
                    "Effective_Daily": round(r_daily, 2),
                    "Demand_Driver": demand_driver(runner),
                    "Monthly_Indent": round(r_monthly, 0),
                    "Today_Target": round(today_target_qty.get(runner, 0), 0),
                    "Changeover": "No" if co_hrs == 0 else "Yes",
                    "Color_Purge": "Yes" if purge else "No",
                    "Terminal_Relaxed": "YES" if t_relaxed else "No",
                    "Terminal_Note": t_note if t_relaxed else "—",
                    "Type": "Runner-Priority (free capacity)" + (" [TERMINAL-RELAXED]" if t_relaxed else ""),
                    "Role": "Primary",
                    "Tools_Available": tools_cnt,
                    "Tools_Used": 2 if (machine_run_type.get(m, "HALF") == "FULL" and tools_cnt >= 2) else 1,  # V17
                    "Runner_Lock": "No",
                    "Priority_Score": r_score, "Phase": 1,
                    "Indent_Met": "YES" if qty >= shortfall_qty else f"NO — need {r_daily:.2f}/d",
                    "Demand_Met": "YES" if dem_met else "NO",
                    "Stagger_Adjusted": "No",
                })
                runner_priority_log.append({
                    "Runner_Part": runner, "Demand_Driver": demand_driver(runner),
                    "Runner_Days_SOD": round(r_days_sod, 2),
                    "Runner_Days_Now": round(r_days_now, 2), "Fixed_Machine": fixed_m or "—",
                    "Fixed_Used": "YES" if fixed_used else "N/A",
                    "Runner_Daily": round(r_daily, 2),
                    "Runner_Inv_SOD": round(r_inv_sod, 0),
                    "Runner_Shortfall": round(shortfall_qty, 0),
                    "Machine_Assigned": m, "Hours_Needed": round(hours_needed, 3),
                    "Hours_Assigned": round(run_h, 3), "Qty_Produced": qty,
                    "Displacement_Used": "No — free capacity", "Victims": "—",
                    "Total_Hours_Reclaimed": "—",
                    "Terminal_Relaxed": "YES" if t_relaxed else "No",
                    "Result": "PLANNED — free capacity",
                })
                print(f"    {runner:30s} -> {m}  run={run_h:.2f}h  qty={qty:.0f}"
                      + (" [TERMINAL RELAXED]" if t_relaxed else "")
                      + f"  [{machine_run_type.get(m,'HALF')} rate={r_rate:.1f}]")
                best_machine = "DONE"
                break

            machine_rows = [r for r in plan if r["Machine"] == m]
            yieldable = [
                r for r in machine_rows
                if part_category.get(r["Part"], "Stranger") in ("Stranger", "Repeater")
                and r.get("Phase") not in (0,)
                and (current_inventory.get(r["Part"], 0) / effective_daily(r["Part"])
                     if effective_daily(r["Part"]) > 0 else 999) > r_days_now
            ]
            if not yieldable:
                continue
            yieldable.sort(key=lambda r: (effective_daily(r["Part"]), -float(r.get("Production_Qty", 0))))
            reclaimable_detail = []
            for row in yieldable:
                row_run = float(row.get("Run_Hours", 0))
                if row_run <= 0:
                    continue
                if row_run > MIN_RUN_HOURS:
                    reclaimable_detail.append((row, round(row_run - MIN_RUN_HOURS, 4), "partial"))
                else:
                    reclaimable_detail.append((row, round(row_run, 4), "full_remove"))
            total_reclaimable = sum(x[1] for x in reclaimable_detail)
            # Use base rate for shortfall hours calculation in displacement check
            r_rate_m = effective_rate(runner, m)
            hours_needed_m = max(MIN_RUN_HOURS, shortfall_qty / r_rate_m if r_rate_m > 0 else MIN_RUN_HOURS)
            runner_eff_after  = round(free_hrs + total_reclaimable - co_hrs, 4)
            if runner_eff_after < max(MIN_RUN_HOURS, hours_needed_m):
                continue
            disruption = total_reclaimable
            if disruption < best_disruption:
                best_disruption = disruption
                best_machine    = m
                best_co_hrs     = co_hrs
                best_victims    = reclaimable_detail

        if best_machine is None:
            runner_priority_log.append({
                "Runner_Part": runner, "Demand_Driver": demand_driver(runner),
                "Runner_Days_SOD": round(r_days_sod, 2), "Runner_Days_Now": round(r_days_now, 2),
                "Fixed_Machine": fixed_m or "—", "Fixed_Used": "FAILED",
                "Runner_Daily": round(r_daily, 2), "Runner_Inv_SOD": round(r_inv_sod, 0),
                "Runner_Shortfall": round(shortfall_qty, 0),
                "Machine_Assigned": "—", "Hours_Needed": 0,
                "Hours_Assigned": 0, "Qty_Produced": 0, "Displacement_Used": "N/A",
                "Victims": "—", "Total_Hours_Reclaimed": "—",
                "Terminal_Relaxed": "YES" if t_relaxed else "No",
                "Result": "FAILED — no eligible machine",
            })
            continue
        if best_machine == "DONE":
            continue

        # V17: use effective rate for the best machine
        r_rate_best = effective_rate(runner, best_machine)
        hours_needed_best = max(MIN_RUN_HOURS, shortfall_qty / r_rate_best if r_rate_best > 0 else MIN_RUN_HOURS)

        hours_to_free   = hours_needed_best
        victim_log_parts = []
        total_reclaimed  = 0.0
        for (victim_row, reclaimable_hrs, reclaim_type) in best_victims:
            if hours_to_free <= 0.001:
                break
            vpart    = victim_row["Part"]
            # V17: victim rate on best_machine
            vr_val   = effective_rate(vpart, best_machine)
            v_run_orig = float(victim_row.get("Run_Hours", 0))
            carve_hrs  = round(min(reclaimable_hrs, hours_to_free), 4)
            if carve_hrs <= 0:
                continue
            new_run_hrs = round(v_run_orig - carve_hrs, 4)
            if new_run_hrs < MIN_RUN_HOURS:
                plan.remove(victim_row)
                machine_hours[best_machine] = round(machine_hours.get(best_machine, 0) - v_run_orig, 4)
                current_inventory[vpart]    = round(current_inventory.get(vpart, 0) - round(v_run_orig * vr_val, 0), 0)
                actually_freed = v_run_orig
                victim_log_parts.append(f"{vpart} REMOVED")
            else:
                lost_qty = round(carve_hrs * vr_val, 0)
                victim_row["Run_Hours"]      = new_run_hrs
                victim_row["Production_Qty"] = round(new_run_hrs * vr_val, 0)
                victim_row["Total_Hrs_Used"] = round(float(victim_row.get("Changeover_Hrs", 0)) + new_run_hrs, 3)
                victim_row["Type"] = str(victim_row.get("Type", "")) + " [YIELDED]"
                machine_hours[best_machine] = round(machine_hours.get(best_machine, 0) - carve_hrs, 4)
                current_inventory[vpart]    = round(current_inventory.get(vpart, 0) - lost_qty, 0)
                actually_freed = carve_hrs
                victim_log_parts.append(f"{vpart} -{carve_hrs:.2f}h")
            hours_to_free   = round(hours_to_free - actually_freed, 4)
            total_reclaimed = round(total_reclaimed + actually_freed, 4)

        used_now = machine_hours.get(best_machine, 0)
        free_now = round(AVAILABLE_HOURS - used_now, 4)
        eff_free = round(free_now - best_co_hrs, 4)
        if eff_free < MIN_RUN_HOURS:
            runner_priority_log.append({
                "Runner_Part": runner, "Demand_Driver": demand_driver(runner),
                "Runner_Days_SOD": round(r_days_sod, 2), "Runner_Days_Now": round(r_days_now, 2),
                "Fixed_Machine": fixed_m or "—", "Fixed_Used": "FAILED",
                "Runner_Daily": round(r_daily, 2), "Runner_Inv_SOD": round(r_inv_sod, 0),
                "Runner_Shortfall": round(shortfall_qty, 0),
                "Machine_Assigned": best_machine, "Hours_Needed": round(hours_needed_best, 3),
                "Hours_Assigned": 0, "Qty_Produced": 0, "Displacement_Used": "Yes",
                "Victims": "; ".join(victim_log_parts), "Total_Hours_Reclaimed": round(total_reclaimed, 3),
                "Terminal_Relaxed": "YES" if t_relaxed else "No",
                "Result": "FAILED — safety check after carve",
            })
            continue

        run_hrs    = max(MIN_RUN_HOURS, min(eff_free, hours_needed_best))
        qty        = round(run_hrs * r_rate_best, 0)
        fixed_used = (fixed_m is not None and best_machine == fixed_m)
        machine_hours[best_machine]       = round(used_now + best_co_hrs + run_hrs, 4)
        current_inventory[runner]         = round(current_inventory.get(runner, 0) + qty, 0)
        machine_last_part[best_machine]   = runner
        already_planned.add(runner)
        purge = best_co_hrs > vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS) + 0.001
        dem_met = is_demand_met(runner, r_inv_b, qty)
        base_r_best = rate.get(runner, 1)
        mrun_label  = machine_run_label(runner, best_machine)
        tools_cnt   = tools_available.get(runner, 1)
        plan.append({
            "Part": runner, "Color": r_color, "Category": "Runner",
            "Fixed_Machine": fixed_m or "—",
            "Fixed_Used": "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
            "Machine": best_machine,
            "Machine_Run_Type": machine_run_type.get(best_machine, "HALF"),    # V17
            "Run_Mode": mrun_label,                                              # V17
            "Run_Hours": round(run_hrs, 3),
            "Changeover_Hrs": round(best_co_hrs, 3),
            "Total_Hrs_Used": round(best_co_hrs + run_hrs, 3),
            "Base_Rate_Per_Hour": round(base_r_best, 2),                         # V17
            "Rate_Per_Hour": round(r_rate_best, 2),
            "Production_Qty": qty,
            "Inventory_Before": round(r_inv_b, 0),
            "Demand_Today_Required": round(max(0.0, dem_d - r_inv_b), 2),
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(indent_daily.get(runner, 0.0), 2),
            "Effective_Daily": round(r_daily, 2),
            "Demand_Driver": demand_driver(runner),
            "Monthly_Indent": round(r_monthly, 0),
            "Today_Target": round(today_target_qty.get(runner, 0), 0),
            "Changeover": "No" if best_co_hrs == 0 else "Yes",
            "Color_Purge": "Yes" if purge else "No",
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Terminal_Note": t_note if t_relaxed else "—",
            "Type": "Runner-Priority (displacement)" + (" [TERMINAL-RELAXED]" if t_relaxed else ""),
            "Role": "Primary",
            "Tools_Available": tools_cnt,
            "Tools_Used": 2 if (machine_run_type.get(best_machine, "HALF") == "FULL" and tools_cnt >= 2) else 1,  # V17
            "Runner_Lock": "No",
            "Priority_Score": r_score, "Phase": 1,
            "Indent_Met": "YES" if qty >= shortfall_qty else f"NO — need {r_daily:.2f}/d",
            "Demand_Met": "YES" if dem_met else "NO",
            "Stagger_Adjusted": "No",
        })
        runner_priority_log.append({
            "Runner_Part": runner, "Demand_Driver": demand_driver(runner),
            "Runner_Days_SOD": round(r_days_sod, 2), "Runner_Days_Now": round(r_days_now, 2),
            "Fixed_Machine": fixed_m or "—",
            "Fixed_Used": "YES" if fixed_used else "N/A",
            "Runner_Daily": round(r_daily, 2), "Runner_Inv_SOD": round(r_inv_sod, 0),
            "Runner_Shortfall": round(shortfall_qty, 0),
            "Machine_Assigned": best_machine, "Hours_Needed": round(hours_needed_best, 3),
            "Hours_Assigned": round(run_hrs, 3), "Qty_Produced": qty,
            "Displacement_Used": "Yes", "Victims": "; ".join(victim_log_parts),
            "Total_Hours_Reclaimed": round(total_reclaimed, 3),
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Result": "PLANNED — displacement",
        })
        print(f"    {runner:30s} -> {best_machine}  run={run_hrs:.2f}h  qty={qty:.0f}  "
              f"victims=[{'; '.join(victim_log_parts)}]"
              + (" [TERMINAL RELAXED]" if t_relaxed else "")
              + f"  [{machine_run_type.get(best_machine,'HALF')} rate={r_rate_best:.1f}]")

    return runner_priority_log

# =============================================================
# SECTION 19 — DISPLACEMENT PRE-PASS
# =============================================================

def displace_for_zero_inv(part, machine_hours, machine_last_part,
                           current_inventory, plan, already_planned, priority_scores):
    """Displace an over-stocked part to make room for a zero-inventory part."""
    daily      = effective_daily(part)
    category   = part_category.get(part, "Stranger")
    score      = priority_scores.get(part, 0)
    inv_before = inventory.get(part, 0)
    compatible = vt_compat.get(part, [])
    fixed_m    = part_fixed_machine.get(part)
    _, t_note, t_relaxed = terminal_blocked(part)
    if not compatible:
        return False

    candidate_machines = [
        m for m in compatible
        if m not in _phase_a_machines
        and round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4) < MIN_RUN_HOURS
        and machine_has_capacity_for_new_part(m, part, plan)
    ]
    if not candidate_machines:
        return False

    best_machine    = None
    best_victim_row = None
    best_victim_days = -1
    part_days_cov   = current_inventory.get(part, 0) / daily if daily > 0 else 0

    for m in candidate_machines:
        for row in [r for r in plan if r["Machine"] == m and r.get("Phase") not in (0,)]:
            vpart  = row["Part"]
            vdaily = effective_daily(vpart)
            vinv   = current_inventory.get(vpart, 0)
            vdays  = vinv / vdaily if vdaily > 0 else 999
            if vdays <= part_days_cov or vdays < SAFETY_DAYS or vinv <= 0:
                continue
            vrun = float(row.get("Run_Hours", 0))
            if vrun - MIN_RUN_HOURS < MIN_RUN_HOURS:
                continue
            if vdays > best_victim_days:
                best_victim_days = vdays
                best_victim_row  = row
                best_machine     = m

    if best_machine is None or best_victim_row is None:
        return False

    vpart    = best_victim_row["Part"]
    # V17: effective rate of victim on best_machine
    vr_val   = effective_rate(vpart, best_machine)
    reduce_h = MIN_RUN_HOURS
    lost_qty = round(reduce_h * vr_val, 0)
    best_victim_row["Run_Hours"]      = round(float(best_victim_row["Run_Hours"]) - reduce_h, 3)
    best_victim_row["Production_Qty"] = round(float(best_victim_row["Production_Qty"]) - lost_qty, 0)
    best_victim_row["Total_Hrs_Used"] = round(
        float(best_victim_row.get("Changeover_Hrs", 0)) + float(best_victim_row["Run_Hours"]), 3)
    best_victim_row["Type"] = str(best_victim_row.get("Type", "")) + " [DISPLACED]"
    current_inventory[vpart]   = round(current_inventory.get(vpart, 0) - lost_qty, 0)
    machine_hours[best_machine] = round(machine_hours.get(best_machine, 0) - reduce_h, 4)

    p_col  = part_color.get(part, "UNKNOWN")
    last   = machine_last_part.get(best_machine)
    l_col  = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
    co_hrs = 0.0 if (last is None or last == part) else (
        vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS)
        + (COLOR_PURGE_HRS if p_col != l_col and p_col != "UNKNOWN" and l_col not in ("UNKNOWN", "NEEDS_PURGE") else 0.0)
    )

    # V17: effective rate for the displaced part on best_machine
    r_val = effective_rate(part, best_machine)
    if r_val <= 0:
        r_val = rate.get(part, 1)

    eff_free = round(AVAILABLE_HOURS - machine_hours.get(best_machine, 0) - co_hrs, 4)
    run_hrs  = max(MIN_RUN_HOURS, min(eff_free, max(0.0, daily / r_val if r_val > 0 else MIN_RUN_HOURS)))
    qty      = round(run_hrs * r_val, 0)

    machine_hours[best_machine]     = round(machine_hours.get(best_machine, 0) + co_hrs + run_hrs, 4)
    current_inventory[part]         = round(current_inventory.get(part, 0) + qty, 0)
    machine_last_part[best_machine] = part
    already_planned.add(part)

    has_purge  = co_hrs > vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS) + 0.001
    fixed_used = (fixed_m is not None and best_machine == fixed_m)
    dem_d = demand_daily_raw.get(part, 0.0)
    base_r     = rate.get(part, 1)
    mrun_label = machine_run_label(part, best_machine)
    tools_cnt  = tools_available.get(part, 1)

    plan.append({
        "Part": part, "Color": p_col, "Category": category,
        "Fixed_Machine": fixed_m or "—",
        "Fixed_Used": "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
        "Machine": best_machine,
        "Machine_Run_Type": machine_run_type.get(best_machine, "HALF"),    # V17
        "Run_Mode": mrun_label,                                              # V17
        "Run_Hours": round(run_hrs, 3),
        "Changeover_Hrs": round(co_hrs, 3),
        "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
        "Base_Rate_Per_Hour": round(base_r, 2),                              # V17
        "Rate_Per_Hour": round(r_val, 2),
        "Production_Qty": qty,
        "Inventory_Before": round(inv_before, 0),
        "Demand_Today_Required": round(max(0.0, dem_d - inv_before), 2),
        "Demand_Daily": round(dem_d, 2),
        "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
        "Effective_Daily": round(daily, 2),
        "Demand_Driver": demand_driver(part),
        "Monthly_Indent": round(effective_monthly(part), 0),
        "Today_Target": round(today_target_qty.get(part, 0), 0),
        "Changeover": "No" if co_hrs == 0 else "Yes",
        "Color_Purge": "Yes" if has_purge else "No",
        "Terminal_Relaxed": "YES" if t_relaxed else "No",
        "Terminal_Note": t_note if t_relaxed else "—",
        "Type": "Displacement [ZERO-INV]" + (" [TERMINAL-RELAXED]" if t_relaxed else ""),
        "Role": "Primary", "Tools_Available": tools_cnt,
        "Tools_Used": 2 if (machine_run_type.get(best_machine, "HALF") == "FULL" and tools_cnt >= 2) else 1,  # V17
        "Runner_Lock": "No", "Priority_Score": score,
        "Phase": 1,
        "Indent_Met": "YES" if qty >= daily else f"NO — need {daily:.2f}/d",
        "Demand_Met": "YES" if is_demand_met(part, inv_before, qty) else "NO",
        "Stagger_Adjusted": "No",
    })
    print(f"      DISPLACEMENT  {part:26s} -> {best_machine:15s}  run={run_hrs:.2f}h  qty={qty:.0f}"
          f"  [{machine_run_type.get(best_machine,'HALF')}]")
    return True

# =============================================================
# SECTION 20 — CO STAGGER
# =============================================================

MIN_CO_GAP_HRS = 20 / 60.0

def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours") or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine": m, "part_before": m_rows[i-1]["Part"],
                    "part_after": row["Part"], "co_duration": co_h,
                    "natural_start": cursor, "row_before": m_rows[i-1],
                    "row_after": row, "actual_start": None, "wait_hrs": 0.0,
                })
            cursor += co_h + run_h
    return events

def _recompute_natural_start(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
    return cursor

def _machine_spare(m, plan):
    used = sum(float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0) for r in plan if r["Machine"] == m)
    return max(0.0, AVAILABLE_HOURS - used)

def _finish_time_of_co(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    total = 0.0
    for row in plan:
        if row["Machine"] != m:
            continue
        if row is target:
            break
        total += float(row.get("Run_Hours") or 0) + float(row.get("Changeover_Hrs") or 0)
    return round(total, 4)

def _extend_row_before(ev, wait_hrs, plan, machine_hours):
    m      = ev["machine"]
    spare  = _machine_spare(m, plan)
    ext    = min(wait_hrs, spare)
    if ext <= 0:
        return 0.0, 0
    rb    = ev["row_before"]
    # V17: effective rate for the part on this machine
    r_val = effective_rate(rb["Part"], m)
    if r_val <= 0:
        r_val = rate.get(rb["Part"], 1.0)
    extra = round(ext * r_val, 0)
    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + ext, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3)
    rb["Stagger_Adjusted"] = f"CO wait +{round(ext*60,1)}min"
    machine_hours[m] = round(machine_hours.get(m, 0) + ext, 4)
    return ext, extra

def stagger_co_by_quantity(plan, machines, scenario_id, current_inventory):
    events = _collect_co_events(plan, machines)
    if len(events) < 2:
        return 0
    adjustments = 0
    for _ in range(len(events) * 2):
        for ev in events:
            ev["_ft"] = _finish_time_of_co(ev, plan)
        events.sort(key=lambda e: e["_ft"])
        conflict = None
        for i in range(len(events) - 1):
            required = events[i]["co_duration"] + MIN_CO_GAP_HRS
            gap      = events[i+1]["_ft"] - events[i]["_ft"]
            if gap < required - 0.001:
                conflict = (events[i], events[i+1], gap, required)
                break
        if conflict is None:
            break
        ev_early, ev_late, gap, required_gap = conflict
        shortfall_hrs = required_gap - gap

        row_late = ev_late["row_before"]
        p_late   = row_late["Part"]
        m_late   = ev_late["machine"]
        # V17: effective rate
        r_late   = effective_rate(p_late, m_late)
        if r_late <= 0:
            r_late = rate.get(p_late, 1)
        inv_l    = current_inventory.get(p_late, 0)
        cap_qty  = effective_opd_cap_qty(p_late, scenario_id, inv_l)
        headroom = max(0.0, cap_qty - inv_l)
        push_qty = round(shortfall_hrs * r_late, 0)
        used_m   = sum(float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0) for r in plan if r["Machine"] == m_late)
        free_m   = max(0.0, AVAILABLE_HOURS - used_m)
        can_push = push_qty <= headroom and shortfall_hrs <= free_m + 0.001 and r_late > 0 and m_late not in _phase_a_machines

        if can_push:
            row_late["Run_Hours"]      = round(float(row_late.get("Run_Hours") or 0) + shortfall_hrs, 3)
            row_late["Production_Qty"] = round(float(row_late.get("Production_Qty") or 0) + push_qty, 0)
            row_late["Total_Hrs_Used"] = round(float(row_late.get("Changeover_Hrs") or 0) + float(row_late["Run_Hours"]), 3)
            current_inventory[p_late]  = round(current_inventory.get(p_late, 0) + push_qty, 0)
            adjustments += 1
            continue

        row_early = ev_early["row_before"]
        p_early   = row_early["Part"]
        m_early   = ev_early["machine"]
        # V17: effective rate
        r_early   = effective_rate(p_early, m_early)
        if r_early <= 0:
            r_early = rate.get(p_early, 1)
        produced_e = float(row_early.get("Production_Qty") or 0)
        daily_e    = effective_daily(p_early)
        min_qty_e  = max(0.0, daily_e - inventory.get(p_early, 0))
        max_pull   = max(0.0, produced_e - min_qty_e)
        pull_qty   = round(shortfall_hrs * r_early, 0)
        can_pull   = (pull_qty <= max_pull and r_early > 0
                      and produced_e - pull_qty >= MIN_RUN_HOURS * r_early
                      and ev_early["machine"] not in _phase_a_machines
                      and row_early.get("Phase") not in (0,))

        if can_pull:
            row_early["Run_Hours"]      = round(float(row_early.get("Run_Hours") or 0) - shortfall_hrs, 3)
            row_early["Production_Qty"] = round(produced_e - pull_qty, 0)
            row_early["Total_Hrs_Used"] = round(float(row_early.get("Changeover_Hrs") or 0) + float(row_early["Run_Hours"]), 3)
            current_inventory[p_early]  = round(current_inventory.get(p_early, 0) - pull_qty, 0)
            adjustments += 1
            continue
        break
    return adjustments

def stagger_changeovers(plan, machines, machine_hours):
    print(f"\n  Tool-Changer Serial Queue Scheduler")
    events = _collect_co_events(plan, machines)
    if not events:
        print(f"  No changeovers — tool changer idle")
        return
    events.sort(key=lambda e: e["natural_start"])
    if len(events) > MAX_DAILY_CO:
        events = events[:MAX_DAILY_CO]
    tool_changer_free_at = 0.0
    total_extra_pcs = 0
    for idx, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_hrs      = round(actual_start - natural_start, 4)
        tool_changer_free_at = actual_start + co_h
        if wait_hrs > 0.001:
            _, extra_pcs = _extend_row_before(ev, wait_hrs, plan, machine_hours)
            total_extra_pcs += extra_pcs
        ev["actual_start"] = actual_start
        ev["wait_hrs"]     = wait_hrs
    print(f"  {len(events)} CO events processed  |  Extra pcs from wait fill: {total_extra_pcs}")

# =============================================================
# SECTION 21 — UTILIZATION ENFORCER
# =============================================================

def _extend_existing_on_machine(m, plan, machine_hours, current_inventory,
                                 priority_scores, ceiling_days, remaining):
    consumed = 0.0
    parts_on_m = sorted(
        [row for row in plan if row["Machine"] == m],
        key=lambda r: float(priority_scores.get(r["Part"], 0) or 0),
        reverse=True,
    )
    for row in parts_on_m:
        if remaining < 0.001:
            break
        p_ext   = row["Part"]
        if is_hard_skip(p_ext):
            continue
        # V17: effective rate on this machine
        r_ext   = effective_rate(p_ext, m)
        if r_ext <= 0:
            continue
        daily_p = effective_daily(p_ext)
        inv_now = current_inventory.get(p_ext, 0)
        headroom = max(0.0, ceiling_days * daily_p - inv_now) if daily_p > 0 else 0
        ext_hrs  = min(remaining, headroom / r_ext if r_ext > 0 else 0)
        if ext_hrs < 0.001:
            continue
        extra_qty = round(ext_hrs * r_ext, 0)
        row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
        row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
        row["Total_Hrs_Used"] = round(float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
        row["Type"] = str(row.get("Type", "")) + f"+Ext{ceiling_days}d"
        machine_hours[m]         = round(machine_hours.get(m, 0) + ext_hrs, 4)
        current_inventory[p_ext] = round(current_inventory.get(p_ext, 0) + extra_qty, 0)
        remaining = round(remaining - ext_hrs, 4)
        consumed += ext_hrs
    return consumed, remaining


def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id, priority_scores):
    print(f"\n  UTILIZATION ENFORCER (demand-protected, {MAX_PARTS_PER_MACHINE} parts/machine cap)")
    micro_idle_log = []
    floor_hrs = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)

    machines_by_util = sorted(vt_machines, key=lambda m: machine_hours.get(m, 0))

    for m in machines_by_util:
        if m in _phase_a_machines:
            continue

        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.05:
            continue
        last_on_m  = machine_last_part.get(m)
        last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"

        if machine_hours.get(m, 0) < floor_hrs:
            needed = round(floor_hrs - machine_hours.get(m, 0), 4)
            _, remaining = _extend_existing_on_machine(
                m, plan, machine_hours, current_inventory,
                priority_scores, opd_cap(scenario_id), min(needed, remaining))
            remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)

        if remaining < 0.05:
            continue

        if remaining >= MIN_RUN_HOURS:
            machine_compat = machine_compatible_parts.get(m, [])
            unplanned = []
            for p in machine_compat:
                if rate.get(p, 0) <= 0:
                    continue
                if indent_monthly.get(p, 0) <= 0 and demand_daily_raw.get(p, 0) <= 0:
                    continue
                t_blk, _, _ = terminal_blocked(p)
                if t_blk:
                    continue
                pfm = part_fixed_machine.get(p)
                if pfm is not None and pfm != m:
                    continue
                if pfm is None and m in machine_fixed_parts:
                    continue
                if not machine_has_capacity_for_new_part(m, p, plan):
                    continue
                daily_p = effective_daily(p)
                inv_now = current_inventory.get(p, 0)
                cap_q   = effective_opd_cap_qty(p, scenario_id, inv_now)
                if daily_p > 0 and inv_now >= cap_q:
                    continue
                unplanned.append(p)

            def _sort_key(p):
                total_qty_today = _get_part_total_qty(p, plan)
                is_unplanned    = 0 if total_qty_today == 0 else 1
                needs_co        = 0 if (last_on_m is None or last_on_m == p) else 1
                p_col           = part_color.get(p, "UNKNOWN")
                same_col        = 0 if (needs_co == 1 and p_col == last_color and p_col != "UNKNOWN") else 1
                cat_pri         = {"Runner": 0, "Repeater": 1, "Stranger": 2}.get(part_category.get(p, "Stranger"), 2)
                inv_now_p       = current_inventory.get(p, 0)
                days_cov        = inv_now_p / effective_daily(p) if effective_daily(p) > 0 else 999
                dem_d           = demand_daily_raw.get(p, 0.0)
                dem_gap         = max(0.0, dem_d - inv_now_p) if dem_d > 0 else 0.0
                # FIX-1: parts with a demand gap rank BEFORE the CO penalty.
                no_demand_gap   = 0 if dem_gap > 0 else 1
                return (is_unplanned, no_demand_gap, needs_co, same_col, cat_pri, -dem_gap, days_cov)

            unplanned.sort(key=_sort_key)

            for p in unplanned:
                if remaining < MIN_RUN_HOURS:
                    break
                if not machine_has_capacity_for_new_part(m, p, plan):
                    continue
                co_hrs   = _co_hrs_for(p, m, machine_last_part)
                eff_free = round(remaining - co_hrs, 4)
                if eff_free < MIN_RUN_HOURS:
                    continue
                if co_hrs > 0 and _current_co_count(plan) >= MAX_DAILY_CO:
                    continue
                daily_p   = effective_daily(p)
                # V17: effective rate
                r_val     = effective_rate(p, m)
                if r_val <= 0:
                    continue
                inv_now_p = current_inventory.get(p, 0)
                cap_qty   = effective_opd_cap_qty(p, scenario_id, inv_now_p)
                headroom  = max(0.0, cap_qty - inv_now_p)
                if headroom <= 0:
                    continue
                shortfall = max(0.0, daily_p - inv_now_p)
                min_run   = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
                run_hrs   = max(min_run, min(eff_free, headroom / r_val if r_val > 0 else eff_free))
                qty       = round(run_hrs * r_val, 0)
                inv_before_p = inventory.get(p, 0)
                dem_d_p = demand_daily_raw.get(p, 0.0)
                _, t_note_p, t_relaxed_p = terminal_blocked(p)
                machine_hours[m]      = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
                current_inventory[p]  = round(current_inventory.get(p, 0) + qty, 0)
                machine_last_part[m]  = p
                already_planned.add(p)
                remaining = round(remaining - co_hrs - run_hrs, 4)
                last_on_m  = machine_last_part.get(m)
                last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"
                base_r_p   = rate.get(p, 1)
                mrun_label = machine_run_label(p, m)
                tools_cnt  = tools_available.get(p, 1)
                plan.append({
                    "Part": p, "Color": part_color.get(p, "UNKNOWN"),
                    "Category": part_category.get(p, "Stranger"),
                    "Fixed_Machine": part_fixed_machine.get(p, "—"),
                    "Fixed_Used": "N/A — filler", "Machine": m,
                    "Machine_Run_Type": machine_run_type.get(m, "HALF"),    # V17
                    "Run_Mode": mrun_label,                                   # V17
                    "Run_Hours": round(run_hrs, 3), "Changeover_Hrs": round(co_hrs, 3),
                    "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
                    "Base_Rate_Per_Hour": round(base_r_p, 2),                 # V17
                    "Rate_Per_Hour": round(r_val, 2),
                    "Production_Qty": qty,
                    "Inventory_Before": round(inv_before_p, 0),
                    "Demand_Today_Required": round(max(0.0, dem_d_p - inv_before_p), 2),
                    "Demand_Daily": round(dem_d_p, 2),
                    "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
                    "Effective_Daily": round(daily_p, 2),
                    "Demand_Driver": demand_driver(p),
                    "Monthly_Indent": round(effective_monthly(p), 0),
                    "Today_Target": round(today_target_qty.get(p, 0), 0),
                    "Changeover": "No" if co_hrs == 0 else "Yes",
                    "Color_Purge": "No",
                    "Terminal_Relaxed": "YES" if t_relaxed_p else "No",
                    "Terminal_Note": t_note_p if t_relaxed_p else "—",
                    "Type": "Filler",
                    "Role": "Primary",
                    "Tools_Available": tools_cnt,
                    "Tools_Used": 2 if (machine_run_type.get(m, "HALF") == "FULL" and tools_cnt >= 2) else 1,  # V17
                    "Runner_Lock": "No",
                    "Priority_Score": round(priority_scores.get(p, 0), 2),
                    "Phase": 2, "Stagger_Adjusted": "No",
                    "Indent_Met": "YES" if (inv_before_p + qty) >= (daily_p - 0.5) else "NO",
                    "Demand_Met": "YES" if is_demand_met(p, inv_before_p, qty) else ("N/A" if dem_d_p == 0 else "NO"),
                })
                print(f"    [FILL] {p:28s} -> {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}  driver={demand_driver(p)}"
                      f"  [{machine_run_type.get(m,'HALF')}]")

        for ceiling in [opd_cap(scenario_id), STRATEGIC_BUFFER_DAYS, ABSOLUTE_MAX_DAYS]:
            remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
            if remaining < 0.001:
                break
            _, remaining = _extend_existing_on_machine(
                m, plan, machine_hours, current_inventory,
                priority_scores, ceiling, remaining)

        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining >= MIN_RUN_HOURS:
            indent_met_parts = _parts_with_indent_met(plan)
            # FIX-3: runner-spare multi-machine expansion disabled.
            runner_spare = []
            runner_spare.sort(key=lambda p: (
                0 if (last_on_m is None or last_on_m == p) else 1,
                0 if (part_color.get(p, "UNKNOWN") == last_color and last_color != "UNKNOWN") else 1,
                -priority_scores.get(p, 0),
            ))
            for p in runner_spare:
                if remaining < MIN_RUN_HOURS:
                    break
                co_hrs  = _co_hrs_for(p, m, machine_last_part)
                eff_free = round(remaining - co_hrs, 4)
                if eff_free < MIN_RUN_HOURS:
                    continue
                if co_hrs > 0 and _current_co_count(plan) >= MAX_DAILY_CO:
                    continue
                daily_p   = effective_daily(p)
                # V17: effective rate
                r_val     = effective_rate(p, m)
                if r_val <= 0:
                    continue
                inv_now_p = current_inventory.get(p, 0)
                cap_qty   = effective_opd_cap_qty(p, scenario_id, inv_now_p)
                headroom  = max(0.0, cap_qty - inv_now_p)
                if headroom <= 0:
                    continue
                run_hrs = max(MIN_RUN_HOURS, min(eff_free, headroom / r_val if r_val > 0 else eff_free))
                qty     = round(run_hrs * r_val, 0)
                inv_before_p = inventory.get(p, 0)
                dem_d_p = demand_daily_raw.get(p, 0.0)
                _, t_note_s, t_relaxed_s = terminal_blocked(p)
                tools_used_now = sum(1 for r in plan if r["Part"] == p) + 1
                machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
                current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
                machine_last_part[m] = p
                remaining = round(remaining - co_hrs - run_hrs, 4)
                base_r_s   = rate.get(p, 1)
                mrun_label = machine_run_label(p, m)
                tools_cnt  = tools_available.get(p, 1)
                plan.append({
                    "Part": p, "Color": part_color.get(p, "UNKNOWN"),
                    "Category": part_category.get(p, "Stranger"),
                    "Fixed_Machine": part_fixed_machine.get(p, "—"),
                    "Fixed_Used": "N/A — runner-rerun", "Machine": m,
                    "Machine_Run_Type": machine_run_type.get(m, "HALF"),    # V17
                    "Run_Mode": mrun_label,                                   # V17
                    "Run_Hours": round(run_hrs, 3), "Changeover_Hrs": round(co_hrs, 3),
                    "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
                    "Base_Rate_Per_Hour": round(base_r_s, 2),                 # V17
                    "Rate_Per_Hour": round(r_val, 2),
                    "Production_Qty": qty,
                    "Inventory_Before": round(inv_before_p, 0),
                    "Demand_Today_Required": round(max(0.0, dem_d_p - inv_before_p), 2),
                    "Demand_Daily": round(dem_d_p, 2),
                    "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
                    "Effective_Daily": round(daily_p, 2),
                    "Demand_Driver": demand_driver(p),
                    "Monthly_Indent": round(effective_monthly(p), 0),
                    "Today_Target": round(today_target_qty.get(p, 0), 0),
                    "Changeover": "No" if co_hrs == 0 else "Yes", "Color_Purge": "No",
                    "Terminal_Relaxed": "YES" if t_relaxed_s else "No",
                    "Terminal_Note": t_note_s if t_relaxed_s else "—",
                    "Type": "Runner-Rerun",
                    "Role": f"Tool-Expansion (tool {tools_used_now})",
                    "Tools_Available": tools_cnt,
                    "Tools_Used": 2 if (machine_run_type.get(m, "HALF") == "FULL" and tools_cnt >= 2) else 1,  # V17
                    "Runner_Lock": "No",
                    "Priority_Score": round(priority_scores.get(p, 0), 2),
                    "Phase": 3, "Stagger_Adjusted": "No",
                    "Indent_Met": "YES" if (inv_before_p + qty) >= (daily_p - 0.5) else "NO",
                    "Demand_Met": "YES" if is_demand_met(p, inv_before_p, qty) else ("N/A" if dem_d_p == 0 else "NO"),
                })
                last_on_m  = p
                last_color = part_color.get(p, "UNKNOWN")
                print(f"    [S3-RUNNER] {p:25s} -> {m:15s}  {run_hrs:.2f}h  tool {tools_used_now}")
                break

        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining > 0.001:
            _, remaining = _extend_existing_on_machine(
                m, plan, machine_hours, current_inventory,
                priority_scores, ABSOLUTE_MAX_DAYS, remaining)

        final_remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if final_remaining >= 0.25:
            util_final = round((1 - final_remaining / AVAILABLE_HOURS) * 100, 1)
            micro_idle_log.append({
                "Machine": m, "Idle_Hrs": round(final_remaining, 3),
                "Utilization_Pct": util_final,
                "Note": f"Parts={parts_on_machine(m, plan)}/{MAX_PARTS_PER_MACHINE}  Exhausted compatible parts",
            })

    return micro_idle_log

# =============================================================
# SECTION 22 — STRATEGIC BUFFER FILLER
# =============================================================

def strategic_buffer_score(part, current_inventory):
    daily = effective_daily(part)
    inv   = current_inventory.get(part, 0.0)
    r_val = rate.get(part, 1.0)
    cat   = part_category.get(part, "Stranger")
    if daily <= 0 or r_val <= 0:
        return 0.0
    velocity = daily / max(float(inv), 1.0)
    cat_w = {"Runner": 1.0, "Repeater": 0.7, "Stranger": 0.4}.get(cat, 0.4)
    return round(velocity * cat_w * STRATEGIC_PRIORITY_DISCOUNT * 100, 2)

def strategic_buffer_filler(plan, machine_hours, machine_last_part,
                              all_parts, already_planned,
                              current_inventory, scenario_id):
    print("\n" + "─"*65)
    print(f"  STRATEGIC BUFFER FILLER")
    print("─"*65)
    filled_count = 0
    strat_scores = {p: strategic_buffer_score(p, current_inventory) for p in all_parts}
    machines_by_util = sorted(vt_machines, key=lambda m: machine_hours.get(m, 0))

    for m in machines_by_util:
        if m in _phase_a_machines:
            continue
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.001:
            continue
        last_on_m  = machine_last_part.get(m)
        last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"

        for ceiling_days in [STRATEGIC_BUFFER_DAYS, ABSOLUTE_MAX_DAYS]:
            if remaining < 0.001:
                break
            consumed, remaining = _extend_existing_on_machine(
                m, plan, machine_hours, current_inventory,
                strat_scores, ceiling_days, remaining)
            if consumed > 0:
                filled_count += 1
            remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)

        if remaining < MIN_RUN_HOURS:
            if remaining > 0.001:
                _, remaining = _extend_existing_on_machine(
                    m, plan, machine_hours, current_inventory,
                    strat_scores, ABSOLUTE_MAX_DAYS, remaining)
            continue

        if _current_co_count(plan) < MAX_DAILY_CO:
            machine_compat = machine_compatible_parts.get(m, [])
            candidates = []
            for p in machine_compat:
                t_blk, _, _ = terminal_blocked(p)
                if t_blk:
                    continue
                if rate.get(p, 0) <= 0:
                    continue
                if indent_monthly.get(p, 0) <= 0 and demand_daily_raw.get(p, 0) <= 0:
                    continue
                pfm = part_fixed_machine.get(p)
                if pfm is not None and pfm != m:
                    continue
                if pfm is None and m in machine_fixed_parts:
                    continue
                if not machine_has_capacity_for_new_part(m, p, plan):
                    continue
                # FIX-3: skip parts already assigned to any machine
                if any(row["Part"] == p for row in plan):
                    continue
                daily_p = effective_daily(p)
                inv_p   = current_inventory.get(p, 0)
                if daily_p > 0 and inv_p >= ABSOLUTE_MAX_DAYS * daily_p:
                    continue
                candidates.append(p)

            def _strat_sort(p):
                total_qty_today = _get_part_total_qty(p, plan)
                is_unplanned    = 0 if total_qty_today == 0 else 1
                needs_co        = 0 if (last_on_m is None or last_on_m == p) else 1
                p_col           = part_color.get(p, "UNKNOWN")
                same_col        = 0 if (needs_co == 1 and p_col == last_color and p_col != "UNKNOWN") else 1
                sc              = strat_scores.get(p, 0)
                return (is_unplanned, needs_co, same_col, -sc)

            candidates.sort(key=_strat_sort)

            for p in candidates:
                if remaining < MIN_RUN_HOURS:
                    break
                if not machine_has_capacity_for_new_part(m, p, plan):
                    continue
                co_hrs   = _co_hrs_for(p, m, machine_last_part)
                eff_free = round(remaining - co_hrs, 4)
                if eff_free < MIN_RUN_HOURS:
                    continue
                if co_hrs > 0 and _current_co_count(plan) >= MAX_DAILY_CO:
                    continue
                # V17: effective rate
                r_val     = effective_rate(p, m)
                if r_val <= 0:
                    continue
                daily_p   = effective_daily(p)
                inv_now   = current_inventory.get(p, 0)
                headroom  = max(0.0, STRATEGIC_BUFFER_DAYS * daily_p - inv_now)
                if headroom <= 0:
                    headroom = max(0.0, ABSOLUTE_MAX_DAYS * daily_p - inv_now)
                if headroom <= 0:
                    continue
                run_hrs   = max(MIN_RUN_HOURS, min(eff_free, headroom / r_val if r_val > 0 else eff_free))
                qty       = round(run_hrs * r_val, 0)
                inv_before_p = inventory.get(p, 0)
                dem_d_p = demand_daily_raw.get(p, 0.0)
                has_purge = co_hrs > vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS) + 0.001
                _, t_note_sb, t_relaxed_sb = terminal_blocked(p)
                machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
                current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
                machine_last_part[m] = p
                already_planned.add(p)
                remaining = round(remaining - co_hrs - run_hrs, 4)
                filled_count += 1
                base_r_sb  = rate.get(p, 1)
                mrun_label = machine_run_label(p, m)
                tools_cnt  = tools_available.get(p, 1)
                plan.append({
                    "Part": p, "Color": part_color.get(p, "UNKNOWN"),
                    "Category": part_category.get(p, "?"),
                    "Fixed_Machine": part_fixed_machine.get(p, "—"),
                    "Fixed_Used": "N/A — strategic buffer", "Machine": m,
                    "Machine_Run_Type": machine_run_type.get(m, "HALF"),    # V17
                    "Run_Mode": mrun_label,                                   # V17
                    "Run_Hours": round(run_hrs, 3), "Changeover_Hrs": round(co_hrs, 3),
                    "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
                    "Base_Rate_Per_Hour": round(base_r_sb, 2),                # V17
                    "Rate_Per_Hour": round(r_val, 2), "Production_Qty": qty,
                    "Inventory_Before": round(inv_before_p, 0),
                    "Demand_Today_Required": round(max(0.0, dem_d_p - inv_before_p), 2),
                    "Demand_Daily": round(dem_d_p, 2),
                    "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
                    "Effective_Daily": round(daily_p, 2),
                    "Demand_Driver": demand_driver(p),
                    "Monthly_Indent": round(effective_monthly(p), 0),
                    "Today_Target": 0,
                    "Changeover": "No" if co_hrs == 0 else "Yes",
                    "Color_Purge": "Yes" if has_purge else "No",
                    "Terminal_Relaxed": "YES" if t_relaxed_sb else "No",
                    "Terminal_Note": t_note_sb if t_relaxed_sb else "—",
                    "Type": "Strategic-Buffer",
                    "Role": "Buffer-Fill",
                    "Tools_Available": tools_cnt,
                    "Tools_Used": 2 if (machine_run_type.get(m, "HALF") == "FULL" and tools_cnt >= 2) else 1,  # V17
                    "Runner_Lock": "No",
                    "Priority_Score": round(strat_scores.get(p, 0), 2),
                    "Phase": 4, "Stagger_Adjusted": "No",
                    "Indent_Met": "YES" if (inv_before_p + qty) >= (daily_p - 0.5) else "NO",
                    "Demand_Met": "YES" if is_demand_met(p, inv_before_p, qty) else ("N/A" if dem_d_p == 0 else "NO"),
                })
                last_on_m  = machine_last_part.get(m)
                last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"
                print(f"    [SB-NEW] {p:28s} -> {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}  driver={demand_driver(p)}"
                      f"  [{machine_run_type.get(m,'HALF')}]")

        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining > 0.001:
            _, remaining = _extend_existing_on_machine(
                m, plan, machine_hours, current_inventory,
                strat_scores, ABSOLUTE_MAX_DAYS, remaining)

    print(f"\n  Strategic buffer complete: {filled_count} extensions/additions")
    return filled_count

# =============================================================

# =============================================================
# SECTION 22B — PRE-DISTRIBUTION: ASSIGN PARTS TO MACHINES EVENLY
# =============================================================

def pre_distribute_parts_to_machines(eligible_parts, machines_list):
    """
    V16: Before any hour-assignment, distribute terminal-eligible parts evenly
    across compatible machines so that no machine is left idle simply because
    the greedy pass happened to favour another.

    Algorithm:
      1. Sort parts by daily demand (high first).
      2. For each part, find compatible machines that still have capacity
         under the dynamic cap.
      3. Assign the part to the compatible machine with the fewest assigned
         parts so far (round-robin style).

    Returns
    -------
    machine_assignment : dict  {machine: [part, ...]}
    dynamic_max        : int   effective parts-per-machine cap (ceiling)
    """
    global _dynamic_max_parts

    eligible_set = set(eligible_parts)

    # Build per-machine eligibility (only non-fixed machines)
    fixed_machines = set(machine_fixed_parts.keys())
    machine_eligible = {}
    for m in machines_list:
        if m in fixed_machines:
            continue
        compat = [p for p in machine_compatible_parts.get(m, []) if p in eligible_set]
        if compat:
            machine_eligible[m] = set(compat)

    active_machines = list(machine_eligible.keys())
    n_machines = len(active_machines)
    n_parts    = len(eligible_parts)

    if n_machines == 0:
        return {m: [] for m in machines_list}, MAX_PARTS_PER_MACHINE

    # Dynamic ceiling
    dynamic_max = math.ceil(n_parts / n_machines) if n_machines > 0 else MAX_PARTS_PER_MACHINE
    dynamic_max = max(1, min(dynamic_max, MAX_PARTS_PER_MACHINE))
    _dynamic_max_parts = dynamic_max

    print(f"\n  PRE-DISTRIBUTION: {n_parts} eligible parts / {n_machines} active machines "
          f"=> {dynamic_max} parts/machine cap (was {MAX_PARTS_PER_MACHINE})")

    # Sort parts: demand descending, then terminal coverage descending
    sorted_parts = sorted(eligible_parts,
                          key=lambda p: (-demand_daily_raw.get(p, 0.0),
                                         -terminal_coverage_ratio(p)))

    machine_assignment = {m: [] for m in machines_list}

    for p in sorted_parts:
        compatible = [
            m for m in active_machines
            if p in machine_eligible.get(m, set())
            and len(machine_assignment[m]) < dynamic_max
        ]
        if not compatible:
            continue
        # Assign to the machine with the fewest parts assigned so far
        best_m = min(compatible, key=lambda m: len(machine_assignment[m]))
        machine_assignment[best_m].append(p)

    # Log distribution
    for m in active_machines:
        assigned = machine_assignment[m]
        mtype    = machine_run_type.get(m, "HALF")
        print(f"    {m:<20} [{mtype}]  assigned {len(assigned)} parts: "
              f"{', '.join(assigned[:5])}{'...' if len(assigned) > 5 else ''}")

    return machine_assignment, dynamic_max


# =============================================================
# SECTION 22C — IDLE MACHINE RESCUE
# =============================================================

def rescue_idle_machines(plan, machine_hours, machine_last_part, current_inventory,
                          already_planned, priority_scores, scenario_id,
                          terminal_blocked_parts_info):
    """
    V16: After Pass 2, any machine with zero planned parts is idle.
    This pass attempts to fix that by:

    Step 1 — DIRECT: assign any unplanned terminal-eligible part that is
             compatible with the idle machine.

    Step 2 — SWAP: if no unplanned compatible part exists but there is a
             planned part on an over-full machine (at dynamic cap) and that
             part is also compatible with the idle machine, move it there.
             The freed slot on the over-full machine then gets the highest-
             priority unplanned part compatible with that machine.

    The loop repeats until no idle machines remain or no further progress
    can be made.
    """
    blocked_set  = {r["Part"] for r in terminal_blocked_parts_info}
    fixed_set    = set(machine_fixed_parts.keys())
    fixed_parts_ = set(part_fixed_machine.keys())

    print("\n" + "─"*65)
    print(f"  IDLE MACHINE RESCUE (V16)")
    print("─"*65)

    rescued_count = 0
    max_iterations = len(vt_machines) * 3  # safety limit

    for _iter in range(max_iterations):
        idle_machines = [
            m for m in vt_machines
            if m not in _phase_a_machines
            and m not in fixed_set
            and parts_on_machine(m, plan) == 0
        ]

        if not idle_machines:
            break

        made_progress = False

        for idle_m in idle_machines:
            # All terminal-eligible non-fixed parts compatible with idle_m
            compat_all = [
                p for p in machine_compatible_parts.get(idle_m, [])
                if p not in blocked_set
                and p not in fixed_parts_
                and idle_m not in fixed_set
                and rate.get(p, 0) > 0
                and effective_daily(p) > 0
            ]

            unplanned_compat = [p for p in compat_all if p not in already_planned]
            planned_compat   = [p for p in compat_all if p in already_planned]

            # ── Step 1: Direct assignment ─────────────────────────────────
            unplanned_compat.sort(key=lambda p: -priority_scores.get(p, 0))

            for p in unplanned_compat:
                if parts_on_machine(idle_m, plan) >= _dynamic_max_parts:
                    break
                skip, _ = should_skip(p)
                if skip:
                    continue
                t_blk, _, _ = terminal_blocked(p)
                if t_blk:
                    continue
                co_hrs = _co_hrs_for(p, idle_m, machine_last_part)
                free   = round(AVAILABLE_HOURS - machine_hours.get(idle_m, 0), 4)
                eff    = round(free - co_hrs, 4)
                if eff < MIN_RUN_HOURS:
                    continue
                # V17: effective rate
                r_val   = effective_rate(p, idle_m)
                if r_val <= 0:
                    continue
                inv_now = current_inventory.get(p, 0)
                cap_qty = effective_opd_cap_qty(p, scenario_id, inv_now)
                hdroom  = max(0.0, cap_qty - inv_now)
                if hdroom <= 0:
                    continue
                run_hrs = max(MIN_RUN_HOURS, min(eff, hdroom / r_val if r_val > 0 else eff))
                qty     = round(run_hrs * r_val, 0)
                inv_b   = inventory.get(p, 0)
                _, t_note, t_relaxed = terminal_blocked(p)

                machine_hours[idle_m]    = round(machine_hours.get(idle_m, 0) + co_hrs + run_hrs, 4)
                current_inventory[p]     = round(current_inventory.get(p, 0) + qty, 0)
                machine_last_part[idle_m] = p
                already_planned.add(p)

                row = _make_plan_row(p, idle_m, run_hrs, co_hrs, qty, scenario_id,
                                     "IdleRescue-Direct", "Primary", phase=3)
                row["Priority_Score"]    = priority_scores.get(p, 0)
                row["Inventory_Before"]  = round(inv_b, 0)
                row.setdefault("Terminal_Reminder", "—")
                plan.append(row)
                print(f"    [IDLE-RESCUE Direct] {p:28s} -> {idle_m:15s}  {run_hrs:.2f}h  qty={qty:.0f}"
                      f"  [{machine_run_type.get(idle_m,'HALF')}]")
                rescued_count += 1
                made_progress = True

            # ── Step 2: Swap if still idle ────────────────────────────────
            if parts_on_machine(idle_m, plan) > 0:
                continue  # direct assignment succeeded

            # Find unplanned parts that couldn't fit on another machine
            unplanned_any = [
                p for p in compat_all
                if p not in already_planned
                and not should_skip(p)[0]
                and not terminal_blocked(p)[0]
            ]
            unplanned_any.sort(key=lambda p: -priority_scores.get(p, 0))

            for unplanned_p in unplanned_any:
                if parts_on_machine(idle_m, plan) >= _dynamic_max_parts:
                    break

                # Find over-full machines that have a part also compatible with idle_m
                for overcrowded_m in vt_compat.get(unplanned_p, []):
                    if (overcrowded_m == idle_m
                            or overcrowded_m in fixed_set
                            or overcrowded_m in _phase_a_machines):
                        continue
                    if machine_has_capacity_for_new_part(overcrowded_m, unplanned_p, plan):
                        continue  # still has room — direct, not a swap candidate

                    # Find lowest-demand part on overcrowded_m compatible with idle_m
                    swappable = [
                        r for r in plan
                        if r["Machine"] == overcrowded_m
                        and r.get("Phase") not in (0,)
                        and r["Part"] in machine_compatible_parts.get(idle_m, [])
                        and demand_daily_raw.get(r["Part"], 0) < demand_daily_raw.get(unplanned_p, 0)
                    ]
                    if not swappable:
                        continue
                    swappable.sort(key=lambda r: demand_daily_raw.get(r["Part"], 0))
                    victim_row  = swappable[0]
                    victim_part = victim_row["Part"]

                    # Check idle_m can run victim_part
                    co_v   = _co_hrs_for(victim_part, idle_m, machine_last_part)
                    free_i = round(AVAILABLE_HOURS - machine_hours.get(idle_m, 0), 4)
                    eff_i  = round(free_i - co_v, 4)
                    if eff_i < MIN_RUN_HOURS:
                        continue

                    # Perform swap: remove victim from overcrowded_m
                    v_run = float(victim_row.get("Run_Hours", 0))
                    v_co  = float(victim_row.get("Changeover_Hrs", 0))
                    v_qty = float(victim_row.get("Production_Qty", 0))
                    # V17: victim rate on overcrowded machine
                    v_r   = effective_rate(victim_part, overcrowded_m)
                    if v_r <= 0:
                        v_r = rate.get(victim_part, 1)
                    plan.remove(victim_row)
                    machine_hours[overcrowded_m] = round(
                        machine_hours.get(overcrowded_m, 0) - v_run - v_co, 4)
                    current_inventory[victim_part] = round(
                        current_inventory.get(victim_part, 0) - v_qty, 0)

                    # Add victim to idle machine
                    # V17: victim rate on idle machine
                    v_r_idle = effective_rate(victim_part, idle_m)
                    if v_r_idle <= 0:
                        v_r_idle = rate.get(victim_part, 1)
                    new_run_i = min(eff_i, max(MIN_RUN_HOURS, v_run))
                    new_qty_i = round(new_run_i * v_r_idle, 0)
                    machine_hours[idle_m] = round(
                        machine_hours.get(idle_m, 0) + co_v + new_run_i, 4)
                    current_inventory[victim_part] = round(
                        current_inventory.get(victim_part, 0) + new_qty_i, 0)
                    machine_last_part[idle_m] = victim_part

                    row_v = _make_plan_row(victim_part, idle_m, new_run_i, co_v, new_qty_i,
                                           scenario_id, "IdleRescue-Swap", "Primary", phase=3)
                    row_v["Inventory_Before"]  = round(inventory.get(victim_part, 0), 0)
                    row_v.setdefault("Terminal_Reminder", "—")
                    plan.append(row_v)

                    # Now overcrowded_m has a free slot — assign unplanned_p there
                    co_u   = _co_hrs_for(unplanned_p, overcrowded_m, machine_last_part)
                    free_o = round(AVAILABLE_HOURS - machine_hours.get(overcrowded_m, 0), 4)
                    eff_o  = round(free_o - co_u, 4)

                    if (eff_o >= MIN_RUN_HOURS
                            and machine_has_capacity_for_new_part(overcrowded_m, unplanned_p, plan)):
                        # V17: effective rate for unplanned_p on overcrowded_m
                        r_u     = effective_rate(unplanned_p, overcrowded_m)
                        if r_u <= 0:
                            r_u = rate.get(unplanned_p, 1)
                        inv_u   = current_inventory.get(unplanned_p, 0)
                        cap_u   = effective_opd_cap_qty(unplanned_p, scenario_id, inv_u)
                        hdroom_u = max(0.0, cap_u - inv_u)
                        run_u   = max(MIN_RUN_HOURS, min(eff_o,
                                      hdroom_u / r_u if r_u > 0 else eff_o))
                        qty_u   = round(run_u * r_u, 0)

                        machine_hours[overcrowded_m]   = round(
                            machine_hours.get(overcrowded_m, 0) + co_u + run_u, 4)
                        current_inventory[unplanned_p] = round(
                            current_inventory.get(unplanned_p, 0) + qty_u, 0)
                        machine_last_part[overcrowded_m] = unplanned_p
                        already_planned.add(unplanned_p)

                        row_u = _make_plan_row(unplanned_p, overcrowded_m, run_u, co_u, qty_u,
                                               scenario_id, "IdleRescue-Swap-New", "Primary",
                                               phase=3)
                        row_u["Inventory_Before"] = round(inventory.get(unplanned_p, 0), 0)
                        row_u.setdefault("Terminal_Reminder", "—")
                        plan.append(row_u)
                        print(f"    [IDLE-RESCUE Swap] {victim_part}->{idle_m}  "
                              f"{unplanned_p}->{overcrowded_m}")
                        rescued_count += 1
                        made_progress = True
                    break

        if not made_progress:
            break

    remaining_idle = [m for m in vt_machines
                      if m not in _phase_a_machines
                      and m not in fixed_set
                      and parts_on_machine(m, plan) == 0]
    print(f"\n  Idle machine rescue complete: {rescued_count} parts placed"
          f"  |  Still idle: {len(remaining_idle)}")
    if remaining_idle:
        print(f"  Machines still idle (no eligible parts): {', '.join(remaining_idle)}")
    return rescued_count


# SECTION 23 — PASS 4: HIGH-DEMAND RESCUE
# =============================================================

def high_demand_rescue(plan, machine_hours, machine_last_part, current_inventory,
                        already_planned, not_planned, priority_scores):
    print("\n" + "─"*65)
    print(f"  PASS 4: HIGH-DEMAND RESCUE")
    print("─"*65)

    rescue_candidates = sorted(
        [r for r in not_planned
         if demand_daily_raw.get(r.get("Part", ""), 0) > 0
         and "capacity" in r.get("Reason", "").lower()],
        key=lambda r: -demand_daily_raw.get(r.get("Part", ""), 0)
    )

    rescued = []
    for entry in rescue_candidates:
        part    = entry.get("Part", "")
        dem_d   = demand_daily_raw.get(part, 0)
        inv_now = current_inventory.get(part, 0)
        compatible = vt_compat.get(part, [])
        fixed_m    = part_fixed_machine.get(part)
        _, t_note, t_relaxed = terminal_blocked(part)

        if fixed_m:
            machines_to_try = [fixed_m] if fixed_m in compatible else []
        else:
            machines_to_try = [m for m in compatible if m not in machine_fixed_parts]

        if not machines_to_try:
            continue

        placed = False

        # FIX-4: enforce terminal constraint in rescue pass
        t_blk_r, t_note_r, _ = terminal_blocked(part)
        if t_blk_r:
            print(f"  RESCUE BLOCKED (terminal) -> {part}  reason: {t_note_r[:60]}")
            continue

        for m in machines_to_try:
            co_hrs   = _co_hrs_for(part, m, machine_last_part)
            free_hrs = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
            eff_free = round(free_hrs - co_hrs, 4)
            slot_ok  = machine_has_capacity_for_new_part(m, part, plan)

            # V17: effective rate for this machine
            r_val = effective_rate(part, m)
            if r_val <= 0:
                continue

            if slot_ok and eff_free >= MIN_RUN_HOURS:
                if co_hrs > 0 and _current_co_count(plan) >= MAX_DAILY_CO and inv_now > 0:
                    continue
                run_hrs = max(MIN_RUN_HOURS, min(eff_free, max(0, dem_d - inv_now) / r_val if r_val > 0 else MIN_RUN_HOURS))
                qty = round(run_hrs * r_val, 0)
                machine_hours[m]        = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
                current_inventory[part] = round(current_inventory.get(part, 0) + qty, 0)
                machine_last_part[m]    = part
                already_planned.add(part)
                inv_b = inventory.get(part, 0)
                daily_p = effective_daily(part)
                dem_d_p = demand_daily_raw.get(part, 0.0)
                base_r  = rate.get(part, 1)
                mrun_label = machine_run_label(part, m)
                tools_cnt  = tools_available.get(part, 1)
                plan.append({
                    "Part": part, "Color": part_color.get(part, "UNKNOWN"),
                    "Category": part_category.get(part, "Stranger"),
                    "Fixed_Machine": fixed_m or "—",
                    "Fixed_Used": "YES" if fixed_m and m == fixed_m else "N/A",
                    "Machine": m,
                    "Machine_Run_Type": machine_run_type.get(m, "HALF"),    # V17
                    "Run_Mode": mrun_label,                                   # V17
                    "Run_Hours": round(run_hrs, 3), "Changeover_Hrs": round(co_hrs, 3),
                    "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
                    "Base_Rate_Per_Hour": round(base_r, 2),                   # V17
                    "Rate_Per_Hour": round(r_val, 2), "Production_Qty": qty,
                    "Inventory_Before": round(inv_b, 0),
                    "Demand_Today_Required": round(max(0, dem_d - inv_b), 2),
                    "Demand_Daily": round(dem_d_p, 2),
                    "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
                    "Effective_Daily": round(daily_p, 2),
                    "Demand_Driver": demand_driver(part),
                    "Monthly_Indent": round(effective_monthly(part), 0),
                    "Today_Target": round(today_target_qty.get(part, 0), 0),
                    "Changeover": "No" if co_hrs == 0 else "Yes", "Color_Purge": "No",
                    "Terminal_Relaxed": "YES" if t_relaxed else "No",
                    "Terminal_Note": t_note if t_relaxed else "—",
                    "Type": "RESCUE-FreeCapacity",
                    "Role": "Primary",
                    "Tools_Available": tools_cnt,
                    "Tools_Used": 2 if (machine_run_type.get(m, "HALF") == "FULL" and tools_cnt >= 2) else 1,  # V17
                    "Runner_Lock": "No",
                    "Priority_Score": priority_scores.get(part, 0),
                    "Phase": 4,
                    "Indent_Met": "YES" if (inv_b + qty) >= (daily_p - 0.5) else "NO",
                    "Demand_Met": "YES" if is_demand_met(part, inv_b, qty) else "NO",
                    "Stagger_Adjusted": "No",
                })
                rescued.append(part)
                print(f"  RESCUE(free)  {part:<28} -> {m:<18}  qty={qty:.0f}  dem={dem_d:.2f}"
                      f"  [{machine_run_type.get(m,'HALF')}]")
                placed = True
                break

            if not slot_ok:
                parts_on_m = [r for r in plan
                               if r["Machine"] == m
                               and r.get("Phase") not in (0,)
                               and m not in _phase_a_machines]
                if not parts_on_m:
                    continue
                parts_on_m.sort(key=lambda r: demand_daily_raw.get(r["Part"], 0))
                victim_row  = parts_on_m[0]
                victim_part = victim_row["Part"]
                victim_dem  = demand_daily_raw.get(victim_part, 0)
                if dem_d <= victim_dem:
                    continue
                v_run = float(victim_row.get("Run_Hours", 0))
                # V17: victim rate on machine m
                v_r   = effective_rate(victim_part, m)
                if v_r <= 0:
                    v_r = rate.get(victim_part, 1)
                v_co  = float(victim_row.get("Changeover_Hrs", 0))
                plan.remove(victim_row)
                machine_hours[m] = round(machine_hours.get(m, 0) - v_run - v_co, 4)
                current_inventory[victim_part] = round(
                    current_inventory.get(victim_part, 0) - round(v_run * v_r, 0), 0)
                not_planned.append({"Part": victim_part,
                                     "Reason": f"Displaced by higher-demand {part} (dem={dem_d:.2f}>{victim_dem:.2f})"})
                already_planned.discard(victim_part)
                print(f"  RESCUE(displace) {part:<23} evicts {victim_part:<20} (dem {dem_d:.2f}>{victim_dem:.2f}) on {m}")

                co_hrs2  = _co_hrs_for(part, m, machine_last_part)
                free2    = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
                eff2     = round(free2 - co_hrs2, 4)
                if eff2 < MIN_RUN_HOURS:
                    plan.append(victim_row)
                    machine_hours[m] = round(machine_hours.get(m, 0) + v_run + v_co, 4)
                    current_inventory[victim_part] = round(
                        current_inventory.get(victim_part, 0) + round(v_run * v_r, 0), 0)
                    not_planned[:] = [r for r in not_planned if r.get("Part") != victim_part]
                    already_planned.add(victim_part)
                    continue
                # V17: effective rate for part on m
                r_val2 = effective_rate(part, m)
                if r_val2 <= 0:
                    continue
                run_hrs2 = max(MIN_RUN_HOURS, min(eff2, max(0, dem_d - inv_now) / r_val2 if r_val2 > 0 else MIN_RUN_HOURS))
                qty2 = round(run_hrs2 * r_val2, 0)
                machine_hours[m]        = round(machine_hours.get(m, 0) + co_hrs2 + run_hrs2, 4)
                current_inventory[part] = round(current_inventory.get(part, 0) + qty2, 0)
                machine_last_part[m]    = part
                already_planned.add(part)
                inv_b2  = inventory.get(part, 0)
                daily_p2 = effective_daily(part)
                dem_d_p2 = demand_daily_raw.get(part, 0.0)
                base_r2  = rate.get(part, 1)
                mrun_label2 = machine_run_label(part, m)
                tools_cnt2  = tools_available.get(part, 1)
                plan.append({
                    "Part": part, "Color": part_color.get(part, "UNKNOWN"),
                    "Category": part_category.get(part, "Stranger"),
                    "Fixed_Machine": fixed_m or "—",
                    "Fixed_Used": "YES" if fixed_m and m == fixed_m else "N/A",
                    "Machine": m,
                    "Machine_Run_Type": machine_run_type.get(m, "HALF"),    # V17
                    "Run_Mode": mrun_label2,                                  # V17
                    "Run_Hours": round(run_hrs2, 3), "Changeover_Hrs": round(co_hrs2, 3),
                    "Total_Hrs_Used": round(co_hrs2 + run_hrs2, 3),
                    "Base_Rate_Per_Hour": round(base_r2, 2),                  # V17
                    "Rate_Per_Hour": round(r_val2, 2), "Production_Qty": qty2,
                    "Inventory_Before": round(inv_b2, 0),
                    "Demand_Today_Required": round(max(0, dem_d - inv_b2), 2),
                    "Demand_Daily": round(dem_d_p2, 2),
                    "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
                    "Effective_Daily": round(daily_p2, 2),
                    "Demand_Driver": demand_driver(part),
                    "Monthly_Indent": round(effective_monthly(part), 0),
                    "Today_Target": round(today_target_qty.get(part, 0), 0),
                    "Changeover": "No" if co_hrs2 == 0 else "Yes", "Color_Purge": "No",
                    "Terminal_Relaxed": "YES" if t_relaxed else "No",
                    "Terminal_Note": t_note if t_relaxed else "—",
                    "Type": "RESCUE-Displaced",
                    "Role": "Primary",
                    "Tools_Available": tools_cnt2,
                    "Tools_Used": 2 if (machine_run_type.get(m, "HALF") == "FULL" and tools_cnt2 >= 2) else 1,  # V17
                    "Runner_Lock": "No",
                    "Priority_Score": priority_scores.get(part, 0),
                    "Phase": 4,
                    "Indent_Met": "YES" if (inv_b2 + qty2) >= (daily_p2 - 0.5) else "NO",
                    "Demand_Met": "YES" if is_demand_met(part, inv_b2, qty2) else "NO",
                    "Stagger_Adjusted": "No",
                })
                rescued.append(part)
                placed = True
                break

        if placed:
            not_planned[:] = [r for r in not_planned if r.get("Part") != part]

    print(f"  Rescue pass: {len(rescued)} parts rescued — {rescued}")
    return rescued

# =============================================================
# SECTION 24 — VALIDATION
# =============================================================

def validate_plan_rows(plan, current_inventory):
    violations = []
    for row in plan:
        p     = row["Part"]
        run_h = float(row.get("Run_Hours", 0) or 0)
        v = []
        if run_h < MIN_RUN_HOURS - 0.001:
            v.append(f"Run_Hours={run_h:.3f} < MIN={MIN_RUN_HOURS}")
        if v:
            violations.append({
                "Part": p, "Machine": row.get("Machine", "—"),
                "Run_Hours": run_h,
                "Demand_Daily": demand_daily_raw.get(p, 0.0),
                "Terminal_Relaxed": row.get("Terminal_Relaxed", "No"),
                "Violations": " | ".join(v),
            })
    if violations:
        print(f"\n  VALIDATION: {len(violations)} violation(s)")
    else:
        print(f"\n  Validation: all rows OK")
    return violations

# =============================================================
# SECTION 24B — V17: EXCEL SAFE-WRITE SANITIZER
# =============================================================

def _safe_df(df):
    """
    Sanitize a DataFrame for safe Excel writing with openpyxl.

    Root cause of "repair needed" dialog in Excel:
      Mixed-type columns — e.g. a column that contains both numeric values
      (float/int) in regular rows and the string "—" in summary/total rows.
      openpyxl writes these as alternating NUMERIC and STRING cells which
      triggers Excel's repair dialog.

    Fix: if a column contains BOTH numeric and string values, convert the
    entire column to string (Excel shows the data correctly as text).
    Pure numeric columns remain numeric.  Pure string columns remain strings.
    None / NaN in object columns become empty strings.
    """
    if df is None or df.empty:
        return df
    out = df.copy()
    for col in out.columns:
        series = out[col]
        vals   = [v for v in series if v is not None and not (isinstance(v, float) and math.isnan(v))]
        has_numeric = any(isinstance(v, (int, float)) and not isinstance(v, bool) for v in vals)
        has_string  = any(isinstance(v, str) for v in vals)
        if has_numeric and has_string:
            # Mixed — convert everything to string
            out[col] = series.apply(
                lambda x: ""
                if x is None or (isinstance(x, float) and math.isnan(x))
                else str(x)
            )
    # Ensure object columns have no None/NaN left (causes openpyxl warnings)
    for col in out.select_dtypes(include=["object"]).columns:
        out[col] = out[col].fillna("").astype(str)
    return out

# =============================================================
# SECTION 25 — OUTPUT SHEET BUILDERS
# =============================================================

def build_multi_machine_view(plan):
    if not plan:
        return pd.DataFrame()
    part_rows = defaultdict(list)
    for row in plan:
        part_rows[row["Part"]].append(row)
    multi = {p: rows for p, rows in part_rows.items() if len(rows) > 1}
    if not multi:
        return pd.DataFrame()
    output_rows = []
    for part, rows in sorted(multi.items(), key=lambda x: -sum(r["Production_Qty"] for r in x[1])):
        daily     = effective_daily(part)
        total_qty = sum(float(r["Production_Qty"]) for r in rows)
        for row in rows:
            output_rows.append({
                "Part": part, "Machines_Used": len(rows), "Machine": row["Machine"],
                "Machine_Run_Type": row.get("Machine_Run_Type", "HALF"),    # V17
                "Run_Mode": row.get("Run_Mode", "HALF-RUN (1 tool)"),       # V17
                "Role": row.get("Role", "Primary"),
                "Run_Hours": round(float(row["Run_Hours"]), 2),
                "Base_Rate_Per_Hour": round(float(row.get("Base_Rate_Per_Hour", 0)), 2),  # V17
                "Rate_Per_Hour": round(float(row.get("Rate_Per_Hour", 0)), 2),
                "Production_Qty": round(float(row["Production_Qty"]), 0),
                "Effective_Daily": round(daily, 2),
                "Demand_Driver": demand_driver(part),
                "Total_Qty_All_Machines": round(total_qty, 0),
                "Phase": row.get("Phase", 1),
                "Type": row.get("Type", "—"),
            })
    return pd.DataFrame(output_rows)

def build_production_vs_indent(plan, all_parts):
    if not plan:
        return pd.DataFrame()
    part_qty      = defaultdict(float)
    part_machines = defaultdict(list)
    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row["Machine"])
    rows = []
    for p in sorted(part_qty.keys()):
        daily    = effective_daily(p)
        ind_d    = indent_daily.get(p, 0.0)
        dem_d    = demand_daily_raw.get(p, 0.0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_qty[p], 0)
        inv_after = round(inv_b + produced, 0)
        gap = round(produced - daily, 0)
        gap_dir = "OVER" if gap > 0 else ("UNDER" if gap < 0 else "MET")
        demand_met = is_demand_met(p, inv_b, produced)
        indent_met = (inv_b + produced) >= (ind_d - 0.5) if ind_d > 0 else True
        _, t_note, t_relaxed = terminal_blocked(p)
        rows.append({
            "Part": p, "Category": part_category.get(p, "Stranger"),
            "Machines": ", ".join(dict.fromkeys(part_machines[p])),
            "Total_Qty_Produced": produced,
            "Indent_Daily": round(ind_d, 2),
            "Demand_Daily": round(dem_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Terminal_Coverage_Ratio": terminal_coverage_ratio(p),
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Gap_vs_Effective_Daily": gap, "Gap_Direction": gap_dir,
            "Inventory_Before": round(inv_b, 0), "Inventory_After": inv_after,
            "Days_Coverage_After": round(inv_after / daily, 2) if daily > 0 else 0,
            "Demand_Met": "YES" if demand_met else ("N/A" if dem_d == 0 else "NO"),
            "Indent_Met": "YES" if indent_met else "NO",
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        order_map = {"UNDER": 0, "MET": 1, "OVER": 2}
        df["_sort"] = df["Gap_Direction"].map(order_map)
        df = df.sort_values(["_sort", "Gap_vs_Effective_Daily"]).drop(columns=["_sort"]).reset_index(drop=True)
    return df

def build_inventory_target_sheet(plan, all_parts, scenario_id):
    part_produced = defaultdict(float)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))
    rows = []
    for p in sorted(all_parts):
        daily    = effective_daily(p)
        ind_d    = indent_daily.get(p, 0.0)
        dem_d    = demand_daily_raw.get(p, 0.0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_produced.get(p, 0), 0)
        inv_after = round(inv_b + produced, 0)
        days_after = round(inv_after / daily, 2) if daily > 0 else 0
        target_qty = round(TARGET_DAYS * daily, 0)
        if inv_after == 0:
            status = "CRITICAL"
        elif days_after < SAFETY_DAYS:
            status = "BELOW_SAFETY"
        elif days_after < TARGET_DAYS:
            status = "BUILDING"
        else:
            status = "AT_TARGET"
        _, t_note, t_relaxed = terminal_blocked(p)
        demand_met = is_demand_met(p, inv_b, produced)
        rows.append({
            "Part": p, "Category": part_category.get(p, "Stranger"),
            "Indent_Daily": round(ind_d, 2),
            "Demand_Daily": round(dem_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Terminal_Coverage_Ratio": terminal_coverage_ratio(p),
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Inv_Before": round(inv_b, 0),
            "Produced_Today": produced, "Inv_After": inv_after,
            "Days_Coverage_After": days_after, "Target_Qty_5days": target_qty,
            "Buffer_Status": status,
            "Demand_Met": "YES" if demand_met else ("N/A" if dem_d == 0 else "NO"),
            "Scheduled_Today": "YES" if produced > 0 else "NO",
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        status_order = {"CRITICAL": 0, "BELOW_SAFETY": 1, "BUILDING": 2, "AT_TARGET": 3}
        df["_sort"] = df["Buffer_Status"].map(status_order)
        df = df.sort_values(["_sort"], ascending=True).drop(columns=["_sort"]).reset_index(drop=True)
    return df

def build_forward_look(current_inventory_after, all_parts):
    rows = []
    for p in all_parts:
        daily = effective_daily(p)
        if daily <= 0:
            continue
        inv_now = current_inventory_after.get(p, 0)
        days_now = inv_now / daily
        days_until_safety = max(0.0, round((inv_now - SAFETY_DAYS * daily) / daily, 1))
        days_until_zero   = max(0.0, round(inv_now / daily, 1))
        alert = ""
        if days_until_zero <= FORWARD_LOOK_DAYS:
            alert = f"ZERO-STOCK RISK in {days_until_zero:.1f} days"
        elif days_until_safety <= FORWARD_LOOK_DAYS:
            alert = f"BELOW SAFETY in {days_until_safety:.1f} days"
        if alert:
            rows.append({
                "Part": p, "Category": part_category.get(p, "Stranger"),
                "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
                "Effective_Daily": round(daily, 2),
                "Inv_After_Today": round(inv_now, 0),
                "Days_Coverage_Today": round(days_now, 2),
                "Days_Until_Safety": days_until_safety,
                "Days_Until_Zero": days_until_zero,
                "Alert": alert,
                "Action": "ESCALATE" if days_until_zero <= 2 else "Plan next 1-3 days",
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("Days_Until_Zero").reset_index(drop=True)
    return df

def build_terminal_status_sheet():
    rows = []
    all_terminals_known = set(terminal_status.keys())
    for terms in part_terminals.values():
        for t in terms:
            all_terminals_known.add(t)
    for t in sorted(all_terminals_known):
        inv = terminal_status.get(t, None)
        parts_needing = [p for p, terms in part_terminals.items() if t in terms]
        parts_hard_blocked = []
        parts_relaxed      = []
        for p in parts_needing:
            dem_d = demand_daily_raw.get(p, 0.0)
            ind_d = indent_daily.get(p, 0.0)
            req_qty = max(dem_d, ind_d)
            r_val   = rate.get(p, 0.0)
            min_run_qty = MIN_RUN_HOURS * r_val if r_val > 0 else 0.0
            effective_req = max(req_qty, min_run_qty)
            hard_floor    = effective_req * (1.0 - TERMINAL_RELAXATION_PCT)
            t_inv = inv if inv is not None else 0
            if t_inv < hard_floor:
                parts_hard_blocked.append(
                    f"{p}(need={effective_req:.0f},floor={hard_floor:.0f},inv={t_inv:.0f})")
            elif t_inv < effective_req:
                parts_relaxed.append(
                    f"{p}(need={effective_req:.0f},inv={t_inv:.0f},{int(TERMINAL_RELAXATION_PCT*100)}%relax)")
        rows.append({
            "Terminal": t,
            "Inventory": round(inv, 0) if inv is not None else "NOT IN SHEET",
            "Parts_Requiring": ", ".join(sorted(parts_needing)) if parts_needing else "—",
            "Parts_Hard_Blocked": ", ".join(parts_hard_blocked) if parts_hard_blocked else "—",
            "Parts_Relaxed": ", ".join(parts_relaxed) if parts_relaxed else "—",
            "Impact": (
                "HARD BLOCKING" if parts_hard_blocked else
                ("RELAXED" if parts_relaxed else
                 ("Adequate" if parts_needing else "No parts"))
            ),
            "Terminal_Logic": f"required=max(demand,indent,min_run_qty), hard_floor=required*(1-{int(TERMINAL_RELAXATION_PCT*100)}%)",
        })
    return pd.DataFrame(rows)

def build_fixed_machine_status(plan, current_inventory):
    rows = []
    for machine, fixed_parts in sorted(machine_fixed_parts.items()):
        phase    = "A" if machine in _phase_a_machines else "B"
        hrs_used = sum(float(r.get("Run_Hours", 0)) + float(r.get("Changeover_Hrs", 0))
                       for r in plan if r["Machine"] == machine)
        for p in fixed_parts:
            daily  = effective_daily(p)
            inv_b  = inventory.get(p, 0)
            inv_now = current_inventory.get(p, inv_b)
            days_now = round(inv_now / daily, 2) if daily > 0 else 0
            produced = round(inv_now - inv_b, 0)
            _, t_note, t_relaxed = terminal_blocked(p)
            dem_met = is_demand_met(p, inv_b, produced)
            rows.append({
                "Machine": machine,
                "Machine_Run_Type": machine_run_type.get(machine, "HALF"),  # V17
                "Phase": f"Phase {phase}",
                "Part": p,
                "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
                "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
                "Effective_Daily": round(daily, 2),
                "Demand_Driver": demand_driver(p),
                "Terminal_Relaxed": "YES" if t_relaxed else "No",
                "Inv_Before": round(inv_b, 0), "Produced_Today": produced,
                "Inv_After": round(inv_now, 0), "Days_Coverage_After": days_now,
                "Demand_Met": "YES" if dem_met else ("N/A" if demand_daily_raw.get(p, 0) == 0 else "NO"),
                "Machine_Hrs_Used": round(hrs_used, 2),
            })
    return pd.DataFrame(rows)

def build_part_decision_sheet(all_parts, plan, not_planned_list, deferred_list, priority_scores):
    not_planned_reasons = {r["Part"]: r["Reason"] for r in not_planned_list if "Part" in r}
    deferred_reasons    = {r["Part"]: r["Reason"] for r in deferred_list if "Part" in r}

    part_produced = defaultdict(float)
    part_machines = defaultdict(list)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))
        part_machines[row["Part"]].append(row["Machine"])

    rows = []
    for p in sorted(all_parts):
        inv_b    = inventory.get(p, 0)
        dem_d    = demand_daily_raw.get(p, 0.0)
        ind_d    = indent_daily.get(p, 0.0)
        daily    = effective_daily(p)
        r_val    = rate.get(p, None)
        produced = round(part_produced.get(p, 0), 0)
        inv_after = round(inv_b + produced, 0)
        machines  = ", ".join(dict.fromkeys(part_machines[p])) if p in part_machines else "—"
        cat       = part_category.get(p, "Stranger")
        fixed_m   = part_fixed_machine.get(p, "—")
        score     = round(priority_scores.get(p, 0), 2)

        dem_gap              = max(0.0, dem_d - inv_b)
        dem_covered_by_stock = dem_d > 0 and inv_b >= dem_d
        dem_met              = is_demand_met(p, inv_b, produced)

        td = get_terminal_detail_for_part(p)
        t_blocked = td["Terminal_Blocked"] == "YES"
        t_relaxed = td["Terminal_Relaxed"] == "YES"
        t_reason  = td["Terminal_Block_Reason"]

        in_plan = produced > 0
        if in_plan:
            plan_reason = f"PLANNED{'(TERMINAL RELAXED)' if t_relaxed else ''} — dem={dem_d:.2f} inv_before={inv_b:.0f}"
        elif p in not_planned_reasons:
            plan_reason = f"NOT PLANNED — {not_planned_reasons[p]}"
        elif p in deferred_reasons:
            plan_reason = f"DEFERRED — {deferred_reasons[p]}"
        elif t_blocked:
            plan_reason = f"BLOCKED — {t_reason}"
        elif r_val is None or r_val == 0:
            plan_reason = "SKIPPED — zero/missing cycle time"
        else:
            skip, skip_reason = should_skip(p)
            plan_reason = f"SKIPPED — {skip_reason}" if skip else "Not scheduled — no machine capacity"

        rows.append({
            "Part": p, "Category": cat,
            "Fixed_Machine": fixed_m,
            "Priority_Score": score,
            "Inventory_Before": round(inv_b, 0),
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(ind_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Gap_Today": round(dem_gap, 2),
            "Demand_Covered_By_Stock": "YES" if dem_covered_by_stock else ("NO" if dem_d > 0 else "N/A"),
            "Produced_Today": produced,
            "Inventory_After": inv_after,
            "Machines_Used": machines,
            "Demand_Met": "YES" if dem_met else ("N/A" if dem_d == 0 else "NO"),
            "Indent_Met": "YES" if (inv_after >= daily - 0.5 and daily > 0) else ("N/A" if daily == 0 else "NO"),
            "In_Plan": "YES" if in_plan else "NO",
            "Plan_Reason": plan_reason,
            "Terminals_Required": td.get("Terminals_Required", "None"),
            "Terminal_Required_Qty": td.get("Terminal_Required_Qty", "—"),
            "Terminal_Hard_Floor": td.get("Terminal_Hard_Floor", "—"),
            "Terminal_Status_Detail": td.get("Terminal_Status_Detail", "—"),
            "Terminal_Hard_Blocked": td.get("Terminal_Blocked", "No"),
            "Terminal_Relaxed": td.get("Terminal_Relaxed", "No"),
        })
    return pd.DataFrame(rows)


def build_planned_vs_unplanned_sheet(all_parts, plan, not_planned_list, deferred_list,
                                      terminal_blocked_parts):
    """
    Comparison between planned and unplanned parts.
    Sorted: Unplanned first (by demand desc), then Planned (by demand desc).
    """
    not_planned_set = {r["Part"] for r in not_planned_list if "Part" in r}
    not_planned_reasons = {r["Part"]: r["Reason"] for r in not_planned_list if "Part" in r}
    deferred_set    = {r["Part"] for r in deferred_list if "Part" in r}
    deferred_reasons = {r["Part"]: r["Reason"] for r in deferred_list if "Part" in r}
    terminal_block_set = {r["Part"] for r in terminal_blocked_parts if "Part" in r}
    terminal_block_reasons = {r["Part"]: r["Reason"] for r in terminal_blocked_parts if "Part" in r}

    part_produced = defaultdict(float)
    part_machines_planned = defaultdict(list)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))
        part_machines_planned[row["Part"]].append(row["Machine"])

    rows = []
    for p in all_parts:
        inv_b   = inventory.get(p, 0.0)
        dem_d   = demand_daily_raw.get(p, 0.0)
        ind_d   = indent_daily.get(p, 0.0)
        daily   = effective_daily(p)
        r_val   = rate.get(p, None)
        produced = round(part_produced.get(p, 0), 0)
        inv_after = round(inv_b + produced, 0)
        days_before = round(inv_b / daily, 2) if daily > 0 else 0
        days_after  = round(inv_after / daily, 2) if daily > 0 else 0

        compat_machines = vt_compat.get(p, [])
        compat_str = ", ".join(compat_machines) if compat_machines else "None"

        td = get_terminal_detail_for_part(p)
        t_blocked_flag = td["Terminal_Blocked"] == "YES"
        t_relaxed_flag = td["Terminal_Relaxed"] == "YES"

        is_planned  = produced > 0
        is_not_plan = p in not_planned_set
        is_deferred = p in deferred_set
        is_t_block  = p in terminal_block_set

        if is_planned:
            status = "PLANNED" + (" (Terminal Relaxed)" if t_relaxed_flag else "")
        elif is_t_block:
            status = "TERMINAL BLOCKED"
        elif is_not_plan:
            status = "NOT PLANNED"
        elif is_deferred:
            status = "DEFERRED"
        else:
            skip, skip_reason = should_skip(p)
            if skip:
                status = "SKIPPED"
            elif r_val is None or r_val == 0:
                status = "SKIPPED — no rate"
            elif daily == 0:
                status = "SKIPPED — no demand/indent"
            else:
                status = "NOT SCHEDULED"

        if is_planned:
            reason = f"Produced {produced:.0f} pcs on: {', '.join(dict.fromkeys(part_machines_planned[p]))}"
        elif is_t_block:
            reason = terminal_block_reasons.get(p, td["Terminal_Block_Reason"])
        elif is_not_plan:
            reason = not_planned_reasons.get(p, "—")
        elif is_deferred:
            reason = deferred_reasons.get(p, "—")
        else:
            skip, skip_reason = should_skip(p)
            reason = skip_reason if skip else "No machine capacity available"

        rows.append({
            "Part": p,
            "Category": part_category.get(p, "Stranger"),
            "Color": part_color.get(p, "UNKNOWN"),
            "Fixed_Machine": part_fixed_machine.get(p, "—"),
            "Status": status,
            "Status_Group": "PLANNED" if is_planned else "NOT PLANNED",
            "Reason": reason,
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(ind_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Inventory_Before": round(inv_b, 0),
            "Days_Coverage_Before": days_before,
            "Produced_Today": produced,
            "Inventory_After": inv_after,
            "Days_Coverage_After": days_after,
            "Rate_Per_Hour": round(r_val, 2) if r_val else 0,
            "Hours_Needed_For_Daily": round(daily / r_val, 2) if (r_val and r_val > 0 and daily > 0) else 0,
            "Compatible_Machines_Count": len(compat_machines),
            "Compatible_Machines": compat_str,
            "Machines_Used_Today": ", ".join(dict.fromkeys(part_machines_planned[p])) if is_planned else "—",
            "Terminals_Required": td.get("Terminals_Required", "None"),
            "Terminal_Required_Qty": td.get("Terminal_Required_Qty", "—"),
            "Terminal_Hard_Floor_Qty": td.get("Terminal_Hard_Floor", "—"),
            "Terminal_Status_Detail": td.get("Terminal_Status_Detail", "—"),
            "Terminal_Blocked": td.get("Terminal_Blocked", "No"),
            "Terminal_Relaxed": td.get("Terminal_Relaxed", "No"),
            "Terminal_Coverage_Ratio": td.get("Terminal_Coverage_Ratio", 1.0),
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        group_order = {"NOT PLANNED": 0, "PLANNED": 1}
        df["_group_sort"] = df["Status_Group"].map(group_order).fillna(0)
        df = df.sort_values(["_group_sort", "Demand_Daily"], ascending=[True, False])
        df = df.drop(columns=["_group_sort", "Status_Group"]).reset_index(drop=True)
    return df


def build_daily_totals_sheet(plan, machine_hours, vt_machines_list):
    if not plan:
        return pd.DataFrame()
    total_qty     = sum(float(r.get("Production_Qty", 0)) for r in plan)
    total_co      = sum(1 for r in plan if r.get("Changeover") == "Yes")
    total_co_hrs  = sum(float(r.get("Changeover_Hrs", 0)) for r in plan)
    total_run_hrs = sum(float(r.get("Run_Hours", 0)) for r in plan)
    parts_scheduled = len({r["Part"] for r in plan})
    machines_active = len({r["Machine"] for r in plan if r.get("Machine") in vt_machines_list})
    parts_with_demand = [p for p in {r["Part"] for r in plan} if demand_daily_raw.get(p, 0) > 0]
    demand_met_count  = sum(
        1 for p in parts_with_demand
        if is_demand_met(p, inventory.get(p, 0),
                         sum(float(r.get("Production_Qty", 0)) for r in plan if r["Part"] == p))
    )
    # V17: count full/half run machines used
    full_run_used = len({r["Machine"] for r in plan
                         if machine_run_type.get(r.get("Machine",""), "HALF") == "FULL"})
    half_run_used = len({r["Machine"] for r in plan
                         if machine_run_type.get(r.get("Machine",""), "HALF") == "HALF"})

    rows = [
        {"Metric": "Planning Date",                                     "Value": str(PLANNING_DATE)},
        {"Metric": "Total Production Qty (all parts, all machines)",    "Value": round(total_qty, 0)},
        {"Metric": "Total Changeovers Today",                           "Value": total_co},
        {"Metric": "Total Changeover Hours",                            "Value": round(total_co_hrs, 2)},
        {"Metric": "Total Run Hours (across all machines)",             "Value": round(total_run_hrs, 2)},
        {"Metric": "Parts Scheduled",                                   "Value": parts_scheduled},
        {"Metric": "Machines Active",                                   "Value": machines_active},
        {"Metric": "Machines — FULL RUN (2 tools)",                     "Value": full_run_used},   # V17
        {"Metric": "Machines — HALF RUN (1 tool)",                      "Value": half_run_used},   # V17
        {"Metric": "Parts With Demand",                                 "Value": len(parts_with_demand)},
        {"Metric": "Demand Met (production + stock)",                   "Value": demand_met_count},
        {"Metric": "Demand NOT Met",                                    "Value": len(parts_with_demand) - demand_met_count},
        {"Metric": "Max Parts Per Machine",                             "Value": MAX_PARTS_PER_MACHINE},
        {"Metric": "Available Hours Per Machine",                       "Value": AVAILABLE_HOURS},
        {"Metric": "Min Run Hours (NON-NEGOTIABLE)",                    "Value": MIN_RUN_HOURS},
        {"Metric": "Terminal Relaxation %",                             "Value": f"{int(TERMINAL_RELAXATION_PCT*100)}%"},
        {"Metric": "Terminal Logic",
         "Value": f"required=max(demand,indent,min_run_qty); hard_block if inv < required*(1-{int(TERMINAL_RELAXATION_PCT*100)}%)"},
        {"Metric": "Full Run Multiplier",                               "Value": f"{FULL_RUN_MULTIPLIER}x (applies when tools>=2 on FULL machine)"},  # V17
        {"Metric": "",                                                  "Value": ""},
        {"Metric": "=== MACHINE-WISE TOTALS ===",                      "Value": ""},
    ]
    for m in sorted(vt_machines_list):
        m_rows  = [r for r in plan if r["Machine"] == m]
        mtype   = machine_run_type.get(m, "HALF")
        if not m_rows:
            rows.append({"Metric": f"{m} [{mtype}] — IDLE", "Value": 0})
            continue
        m_qty    = sum(float(r.get("Production_Qty", 0)) for r in m_rows)
        m_co     = sum(1 for r in m_rows if r.get("Changeover") == "Yes")
        m_hrs    = machine_hours.get(m, 0)
        m_util   = round(m_hrs / AVAILABLE_HOURS * 100, 1)
        m_parts  = len({r["Part"] for r in m_rows})
        rows.append({
            "Metric": f"{m} [{mtype}] — Parts={m_parts}/{MAX_PARTS_PER_MACHINE} | CO={m_co} | Util={m_util}% | Hrs={round(m_hrs,2)}",
            "Value": round(m_qty, 0)
        })
    return pd.DataFrame(rows)

def build_planning_summary_sheet(plan, not_planned_list, deferred_list,
                                  all_parts, machine_hours, vt_machines_list,
                                  already_planned_set, scenario_desc):
    rows = []
    def _sep(label=""):
        rows.append({"Section": f"── {label} ──", "Metric": "", "Count_or_Value": "", "Detail": ""})
    def _row(section, metric, value, detail=""):
        rows.append({"Section": section, "Metric": metric, "Count_or_Value": value, "Detail": detail})

    part_produced = defaultdict(float)
    for r in plan:
        part_produced[r["Part"]] += float(r.get("Production_Qty", 0))

    total_qty     = sum(float(r.get("Production_Qty", 0)) for r in plan)
    total_co      = sum(1 for r in plan if r.get("Changeover") == "Yes")
    total_co_hrs  = sum(float(r.get("Changeover_Hrs", 0)) for r in plan)
    total_run_hrs = sum(float(r.get("Run_Hours", 0)) for r in plan)

    _sep("SCENARIO")
    _row("Scenario", "Active Scenario", scenario_desc)

    # V17: Half/Full Run summary
    _sep("HALF / FULL RUN MACHINE LOGIC (V17)")
    full_machines = [m for m in vt_machines_list if machine_run_type.get(m, "HALF") == "FULL"]
    half_machines = [m for m in vt_machines_list if machine_run_type.get(m, "HALF") == "HALF"]
    _row("Run Type", "Full Run Machines (2 tools simultaneously)", len(full_machines),
         ", ".join(full_machines) if full_machines else "None configured")
    _row("Run Type", "Half Run Machines (1 tool at a time)", len(half_machines),
         f"{len(half_machines)} machines")
    _row("Run Type", "Full Run Rate Multiplier",
         f"{FULL_RUN_MULTIPLIER}x", "Applies when part.tools >= 2 on a FULL machine")
    _row("Run Type", "How to configure",
         "Add 'Run_Type' column (FULL/HALF) to VT_Changeover sheet",
         "Default = HALF if column absent")

    _sep("TERMINAL LOGIC (V14)")
    _row("Terminal Logic", "Required Qty",
         f"max(demand_daily, indent_daily, min_run_qty={MIN_RUN_HOURS}h x rate)",
         "Must be available in terminal inventory")
    _row("Terminal Logic", f"Hard Block",
         f"terminal_inv < required * (1 - {int(TERMINAL_RELAXATION_PCT*100)}%)",
         "No production allowed")
    _row("Terminal Logic", "Relaxed (Allowed)",
         f"required*(1-{int(TERMINAL_RELAXATION_PCT*100)}%) <= terminal_inv < required",
         "Production allowed, flagged in output")
    _row("Terminal Logic", "OK",
         "terminal_inv >= required", "Full production allowed")
    _row("Terminal Logic", "MIN_RUN_HOURS", f"{MIN_RUN_HOURS}h", "NON-NEGOTIABLE — never relaxed")

    _sep("PLANNING COUNTS")
    _row("Planning Counts", "Total Parts in Universe",    len(all_parts))
    _row("Planning Counts", "Parts Successfully Planned", len({r["Part"] for r in plan}))
    _row("Planning Counts", "Parts NOT Planned",          len(not_planned_list))
    _row("Planning Counts", "Parts Deferred",             len(deferred_list))

    _sep("PRODUCTION TOTALS")
    _row("Production Totals", "Total Production Qty",     round(total_qty, 0))
    _row("Production Totals", "Total Changeovers Today",  total_co)
    _row("Production Totals", "Total Changeover Hours",   round(total_co_hrs, 2))
    _row("Production Totals", "Total Run Hours",          round(total_run_hrs, 2))

    _sep("CATEGORY BREAKDOWN")
    for cat in ("Runner", "Repeater", "Stranger"):
        cat_p = len({r["Part"] for r in plan if part_category.get(r["Part"], "Stranger") == cat})
        cat_np = sum(1 for r in not_planned_list if part_category.get(r.get("Part",""), "Stranger") == cat)
        cat_d  = sum(1 for r in deferred_list if part_category.get(r.get("Part",""), "Stranger") == cat)
        cat_qty = sum(float(r.get("Production_Qty", 0)) for r in plan if part_category.get(r["Part"], "Stranger") == cat)
        _row("Category", f"{cat} — Planned",     cat_p, f"Qty: {round(cat_qty,0):.0f}")
        _row("Category", f"{cat} — Not Planned", cat_np)
        _row("Category", f"{cat} — Deferred",    cat_d)

    _sep("INVENTORY STATUS AFTER TODAY")
    status_counts = {"AT_TARGET": 0, "BUILDING": 0, "BELOW_SAFETY": 0, "CRITICAL": 0, "NO_DEMAND": 0}
    for p in all_parts:
        daily = effective_daily(p)
        inv_after = inventory.get(p, 0) + part_produced.get(p, 0)
        if daily <= 0:
            status_counts["NO_DEMAND"] += 1
        else:
            d = inv_after / daily
            if d >= TARGET_DAYS:    status_counts["AT_TARGET"] += 1
            elif d >= SAFETY_DAYS:  status_counts["BUILDING"] += 1
            elif d >= 1:            status_counts["BELOW_SAFETY"] += 1
            else:                   status_counts["CRITICAL"] += 1
    _row("Inventory Status", "AT_TARGET  (>= 5 days)",         status_counts["AT_TARGET"])
    _row("Inventory Status", "BUILDING   (>= 3 days, < 5)",    status_counts["BUILDING"])
    _row("Inventory Status", "BELOW_SAFETY (>= 1 day, < 3)",   status_counts["BELOW_SAFETY"])
    _row("Inventory Status", "CRITICAL   (< 1 day or zero)",   status_counts["CRITICAL"])
    _row("Inventory Status", "NO_DEMAND / NO_INDENT",           status_counts["NO_DEMAND"])

    _sep("DEMAND COVERAGE")
    demand_parts = [p for p in all_parts if demand_daily_raw.get(p, 0) > 0]
    dem_stock = dem_produced = dem_not = 0
    dem_fail_list = []
    for p in demand_parts:
        inv_b    = inventory.get(p, 0)
        produced = part_produced.get(p, 0)
        dem_d    = demand_daily_raw.get(p, 0)
        if inv_b >= dem_d:
            dem_stock += 1
        elif is_demand_met(p, inv_b, produced):
            dem_produced += 1
        else:
            dem_not += 1
            dem_fail_list.append(p)
    _row("Demand Coverage", "Parts with Demand",                    len(demand_parts))
    _row("Demand Coverage", "Met — by existing stock alone",        dem_stock)
    _row("Demand Coverage", "Met — by production today",            dem_produced)
    _row("Demand Coverage", "NOT Met (shortfall remains)",          dem_not,
         ", ".join(dem_fail_list) if dem_fail_list else "—")

    _sep("TERMINAL STATUS (V14 Logic)")
    hard_blocked_count = sum(1 for p in all_parts if terminal_blocked(p)[0])
    relaxed_count      = sum(1 for p in all_parts if terminal_blocked(p)[2])
    _row("Terminal Status", "Parts HARD Blocked (terminal inv < floor)", hard_blocked_count)
    _row("Terminal Status", f"Parts Scheduled with {int(TERMINAL_RELAXATION_PCT*100)}% Relaxation", relaxed_count)

    _sep("MACHINE UTILISATION")
    util_list = []
    for m in sorted(vt_machines_list):
        used     = machine_hours.get(m, 0)
        util_pct = round(used / AVAILABLE_HOURS * 100, 1)
        util_list.append(util_pct)
        parts_on = len({r["Part"] for r in plan if r["Machine"] == m})
        co_on    = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")
        mtype    = machine_run_type.get(m, "HALF")
        status   = "FULL" if used >= AVAILABLE_HOURS - 0.3 else "GOOD" if util_pct >= 98 else "OK" if util_pct >= 90 else "UNDERUSED"
        _row("Machine Util", m, f"{util_pct}%",
             f"[{mtype}] Hours={round(used,2)}/{AVAILABLE_HOURS}  Parts={parts_on}/{MAX_PARTS_PER_MACHINE}  CO={co_on}  [{status}]")
    if util_list:
        _row("Machine Util", "AVERAGE UTILISATION", f"{round(sum(util_list)/len(util_list),1)}%")

    _sep("PARTS NOT PLANNED — REASONS")
    if not_planned_list:
        for r in not_planned_list:
            _row("Not Planned", r.get("Part", "?"), "NOT PLANNED", r.get("Reason", "—"))
    else:
        _row("Not Planned", "—", "All eligible parts planned", "")

    _sep("PARTS DEFERRED — REASONS")
    if deferred_list:
        for r in deferred_list:
            _row("Deferred", r.get("Part", "?"), "DEFERRED", r.get("Reason", "—"))
    else:
        _row("Deferred", "—", "No deferred parts", "")

    return pd.DataFrame(rows)

def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv    = inventory.get(p, 0.0)
        monthly = effective_monthly(p)
        daily  = effective_daily(p)
        skip, skip_reason = should_skip(p)
        days_cov = inv / daily if daily > 0 else 0
        _, t_note, t_relaxed = terminal_blocked(p)
        if inv == 0.0 and monthly > 0:
            status = "ZERO INV"
        elif skip and inv >= TARGET_DAYS * daily:
            status = "AT TARGET"
        elif skip:
            status = "SKIPPED"
        else:
            status = "PRODUCTION NEEDED"
        rows.append({
            "Part": p,
            "Indent_Monthly": round(indent_monthly.get(p, 0.0), 0),
            "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Terminal_Coverage_Ratio": terminal_coverage_ratio(p),
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Inventory_Now": round(inv, 0),
            "Days_Coverage": round(days_cov, 2),
            "Indent_Status": status,
            "Skip_Reason": skip_reason,
        })
    return pd.DataFrame(rows)

def build_audit(all_universe_parts, matrix_parts, zero_rate_set):
    audit_rows = []
    for part in all_universe_parts:
        inv    = inventory.get(part, 0.0)
        r_val  = rate.get(part, None)
        monthly = effective_monthly(part)
        ind_d  = indent_daily.get(part, 0.0)
        dem_d  = demand_daily_raw.get(part, 0.0)
        daily  = effective_daily(part)
        days_cov = inv / daily if daily > 0 else 0
        t_blk, t_rsn, t_relaxed = terminal_blocked(part)
        tcov = terminal_coverage_ratio(part)

        if part in zero_rate_set or r_val is None:
            status = "ZERO/MISSING CYCLE TIME"
        elif part not in matrix_parts:
            status = "NOT IN VT_MATRIX"
        elif monthly == 0:
            status = "ZERO INDENT+DEMAND"
        elif daily <= MIN_DAILY_INDENT and dem_d <= 0:
            status = "SKIPPED (LOW INDENT, NO DEMAND)"
        elif daily > 0 and inv >= TARGET_DAYS * daily:
            status = "AT TARGET — SKIP"
        elif t_blk:
            status = "BLOCKED — TERMINAL (HARD)"
        elif t_relaxed:
            status = "ENTERS SCHEDULER [TERMINAL RELAXED]"
        else:
            status = "ENTERS SCHEDULER"

        audit_rows.append({
            "Part": part, "Color": part_color.get(part, "UNKNOWN"),
            "Indent_Daily": round(ind_d, 2),
            "Demand_Daily": round(dem_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(part),
            "Terminal_Coverage_Ratio": round(tcov, 4),
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Terminal_Note": t_rsn if (t_relaxed or t_blk) else "—",
            "Inventory": round(inv, 0), "Days_Coverage": round(days_cov, 2),
            "Base_Rate_Per_Hour": round(r_val, 2) if r_val else "—",   # V17
            "Status": status,
        })
    return pd.DataFrame(audit_rows)

def build_machine_wise_plan(plan_df, machine_hours):
    if plan_df.empty:
        return pd.DataFrame()
    rows = []
    for m in vt_machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue
        mtype = machine_run_type.get(m, "HALF")
        for _, pr in machine_rows.iterrows():
            p = pr.get("Part", "—")
            rows.append({
                "Machine": m,
                "Machine_Run_Type": mtype,                                 # V17
                "Part": p,
                "Color": part_color.get(p, "UNKNOWN"),
                "Category": part_category.get(p, "Stranger"),
                "Role": pr.get("Role", "Primary"),
                "Phase": pr.get("Phase", 1),
                "Run_Mode": pr.get("Run_Mode", f"{mtype}-RUN"),            # V17
                "Run_Hours": round(float(pr.get("Run_Hours", 0) or 0), 2),
                "Changeover_Hrs": round(float(pr.get("Changeover_Hrs", 0) or 0), 3),
                "Production_Qty": round(float(pr.get("Production_Qty", 0) or 0), 0),
                "Base_Rate_Per_Hour": round(float(pr.get("Base_Rate_Per_Hour", 0) or 0), 2),  # V17
                "Rate_Per_Hour": round(float(pr.get("Rate_Per_Hour", 0) or 0), 2),
                "Inventory_Before": round(float(pr.get("Inventory_Before", 0) or 0), 0),
                "Demand_Daily": round(float(pr.get("Demand_Daily", 0) or 0), 2),
                "Demand_Today_Required": round(float(pr.get("Demand_Today_Required", 0) or 0), 2),
                "Demand_Met": str(pr.get("Demand_Met", "—")),
                "Effective_Daily": round(float(pr.get("Effective_Daily", 0) or 0), 2),
                "Demand_Driver": str(pr.get("Demand_Driver", "—")),
                "Terminal_Relaxed": str(pr.get("Terminal_Relaxed", "No")),
                "Terminal_Note": str(pr.get("Terminal_Note", "—")),
                "Type": str(pr.get("Type", "Primary") or "Primary"),
                "Row_Type": "Part",
            })
        co_total  = machine_rows["Changeover_Hrs"].apply(lambda x: float(x) if pd.notna(x) else 0).sum()
        run_total = machine_rows["Run_Hours"].apply(lambda x: float(x) if pd.notna(x) else 0).sum()
        qty_total = machine_rows["Production_Qty"].apply(lambda x: float(x) if pd.notna(x) else 0).sum()
        hrs_total = round(co_total + run_total, 2)
        n_parts   = len(machine_rows["Part"].unique())
        # V17 FIX: keep all summary cells as strings to avoid mixed-type column issue
        rows.append({
            "Machine": m,
            "Machine_Run_Type": mtype,
            "Part": f"TOTAL — {m}  [{mtype}] [Parts={n_parts}/{MAX_PARTS_PER_MACHINE}]",
            "Color": "—", "Category": "—", "Role": "—", "Phase": "—",
            "Run_Mode": f"Total {hrs_total}h / {AVAILABLE_HOURS}h  |  Util {round(hrs_total/AVAILABLE_HOURS*100,1)}%",
            "Run_Hours": round(run_total, 2),
            "Changeover_Hrs": round(co_total, 2),
            "Production_Qty": round(qty_total, 0),
            "Base_Rate_Per_Hour": "—",
            "Rate_Per_Hour": "—",
            "Inventory_Before": "—",
            "Demand_Daily": "—", "Demand_Today_Required": "—", "Demand_Met": "—",
            "Effective_Daily": "—", "Demand_Driver": "—",
            "Terminal_Relaxed": "—", "Terminal_Note": "—",
            "Type": "—",
            "Row_Type": "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})
    return pd.DataFrame(rows)

def build_co_queue(plan, machines):
    events = _collect_co_events(plan, machines)
    if not events:
        return pd.DataFrame()
    events.sort(key=lambda e: e["natural_start"])
    rows = []
    tool_changer_free_at = 0.0
    for pos, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        tool_changer_free_at = actual_start + co_h
        before_color = part_color.get(ev["part_before"], "UNKNOWN")
        after_color  = part_color.get(ev["part_after"],  "UNKNOWN")
        color_change = before_color != after_color and before_color != "UNKNOWN" and after_color != "UNKNOWN"
        rows.append({
            "Queue_Position": pos, "Machine": ev["machine"],
            "Machine_Run_Type": machine_run_type.get(ev["machine"], "HALF"),  # V17
            "Part_Before": ev["part_before"], "Part_After": ev["part_after"],
            "Color_Before": before_color, "Color_After": after_color,
            "Color_Change": "YES" if color_change else "No",
            "CO_Duration_Hrs": round(co_h, 3),
            "Natural_Start_Hrs": round(natural_start, 3),
            "Actual_Start_Hrs":  round(actual_start, 3),
            "Wait_Hrs": round(actual_start - natural_start, 3),
        })
    return pd.DataFrame(rows)

def build_daily_planning_report(plan, not_planned_list, deferred_list,
                                 terminal_blocked_parts_info, all_parts, priority_scores):
    """
    Master planning report: one row per part, sorted into three blocks:
      A — Terminal Blocked
      B — Terminal clear, PLANNED
      C — Terminal clear, NOT planned
    Plus summary totals.
    """
    terminal_block_set     = {r["Part"] for r in terminal_blocked_parts_info if "Part" in r}
    terminal_block_reasons = {r["Part"]: r["Reason"] for r in terminal_blocked_parts_info if "Part" in r}
    not_planned_set        = {r["Part"] for r in not_planned_list if "Part" in r}
    not_planned_reasons    = {r["Part"]: r["Reason"] for r in not_planned_list if "Part" in r}
    deferred_set           = {r["Part"] for r in deferred_list if "Part" in r}
    deferred_reasons       = {r["Part"]: r["Reason"] for r in deferred_list if "Part" in r}

    part_qty      = defaultdict(float)
    part_machines = defaultdict(list)
    part_co       = defaultdict(int)
    part_type     = {}
    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row.get("Machine", "—"))
        if row.get("Changeover") == "Yes":
            part_co[p] += 1
        if p not in part_type:
            part_type[p] = row.get("Type", "—")

    def planned_reason(p):
        ptype = part_type.get(p, "—")
        machines_str = ", ".join(dict.fromkeys(part_machines[p]))
        qty = round(part_qty[p], 0)
        inv_b = inventory.get(p, 0)
        dem_d = demand_daily_raw.get(p, 0)
        if "DEMAND-FIRST" in ptype:
            gap = round(max(0, dem_d - inv_b), 0)
            return f"Demand gap = {gap:.0f} pcs — scheduled in Pass 0 (demand-first) on {machines_str}"
        if "Indent-Build" in ptype:
            return f"Indent/buffer build — scheduled in Pass 2 on {machines_str}"
        if "Fixed-PhaseA" in ptype or "Fixed-PhaseB" in ptype:
            return f"Fixed machine assignment — scheduled in fixed-machine pass on {machines_str}"
        if "RESCUE" in ptype:
            return f"High-demand rescue (Pass 4) — placed on {machines_str} after initial passes"
        if "Runner-Priority" in ptype or "Runner" in ptype:
            return f"Runner priority enforcement — scheduled on {machines_str}"
        if "Strategic-Buffer" in ptype:
            return f"Strategic buffer fill — idle machine capacity used on {machines_str}"
        if "Filler" in ptype:
            return f"Utilization filler (Pass 3) — idle capacity used on {machines_str}"
        return f"Planned ({ptype}) on {machines_str}"

    rows = []
    all_sorted = sorted(all_parts)

    # BLOCK A — Terminal Blocked
    block_a = sorted(
        [p for p in all_sorted if p in terminal_block_set],
        key=lambda p: -demand_daily_raw.get(p, 0)
    )
    for p in block_a:
        inv_b   = inventory.get(p, 0)
        dem_d   = demand_daily_raw.get(p, 0.0)
        daily   = effective_daily(p)
        dem_gap = round(max(0, dem_d - inv_b), 2)
        rows.append({
            "Section":              "A — Terminal Blocked",
            "Part":                 p,
            "Category":             part_category.get(p, "Stranger"),
            "Color":                part_color.get(p, "UNKNOWN"),
            "Fixed_Machine":        part_fixed_machine.get(p, "—"),
            "Terminal_Blocked":     "YES",
            "Terminal_Block_Reason": terminal_block_reasons.get(p, "—"),
            "Planning_Status":      "TERMINAL BLOCKED — excluded from all planning passes",
            "Planning_Reason":      terminal_block_reasons.get(p, "—"),
            "Demand_Daily":         round(dem_d, 2),
            "Effective_Daily":      round(daily, 2),
            "Inventory_Before":     round(inv_b, 0),
            "Demand_Gap":           dem_gap,
            "Production_Qty":       0,
            "Machines_Used":        "—",
            "Changeovers_This_Part": 0,
            "Priority_Score":       round(priority_scores.get(p, 0), 2),
        })

    # BLOCK B — Terminal clear, PLANNED
    block_b = sorted(
        [p for p in all_sorted
         if p not in terminal_block_set
         and part_qty.get(p, 0) > 0],
        key=lambda p: -demand_daily_raw.get(p, 0)
    )
    for p in block_b:
        inv_b   = inventory.get(p, 0)
        dem_d   = demand_daily_raw.get(p, 0.0)
        daily   = effective_daily(p)
        dem_gap = round(max(0, dem_d - inv_b), 2)
        qty     = round(part_qty[p], 0)
        _, _, t_relaxed = terminal_blocked(p)
        rows.append({
            "Section":              "B — Planned (terminal clear)",
            "Part":                 p,
            "Category":             part_category.get(p, "Stranger"),
            "Color":                part_color.get(p, "UNKNOWN"),
            "Fixed_Machine":        part_fixed_machine.get(p, "—"),
            "Terminal_Blocked":     "NO" + (" (relaxed)" if t_relaxed else ""),
            "Terminal_Block_Reason": "—",
            "Planning_Status":      "PLANNED",
            "Planning_Reason":      planned_reason(p),
            "Demand_Daily":         round(dem_d, 2),
            "Effective_Daily":      round(daily, 2),
            "Inventory_Before":     round(inv_b, 0),
            "Demand_Gap":           dem_gap,
            "Production_Qty":       qty,
            "Machines_Used":        ", ".join(dict.fromkeys(part_machines[p])),
            "Changeovers_This_Part": part_co.get(p, 0),
            "Priority_Score":       round(priority_scores.get(p, 0), 2),
        })

    # BLOCK C — Terminal clear, NOT planned
    def not_planned_reason(p):
        if p in not_planned_reasons:
            return not_planned_reasons[p]
        if p in deferred_reasons:
            return "Deferred — " + deferred_reasons[p]
        r_val = rate.get(p, None)
        daily = effective_daily(p)
        if r_val is None or r_val == 0:
            return "Skipped — zero or missing production rate (cycle time)"
        if daily == 0:
            return "Skipped — no demand and no indent"
        if not vt_compat.get(p):
            return "Not in VT compatibility matrix — no compatible machine"
        skip, skip_reason = should_skip(p)
        if skip:
            return "Skipped — " + skip_reason
        inv_b = inventory.get(p, 0)
        dem_d = demand_daily_raw.get(p, 0)
        if daily > 0 and inv_b >= daily:
            return f"Skipped — inventory ({inv_b:.0f}) already covers daily demand ({daily:.2f})"
        return "Not scheduled — no machine capacity available in any pass"

    block_c = sorted(
        [p for p in all_sorted
         if p not in terminal_block_set
         and part_qty.get(p, 0) == 0],
        key=lambda p: -demand_daily_raw.get(p, 0)
    )
    for p in block_c:
        inv_b   = inventory.get(p, 0)
        dem_d   = demand_daily_raw.get(p, 0.0)
        daily   = effective_daily(p)
        dem_gap = round(max(0, dem_d - inv_b), 2)
        status  = "DEFERRED" if p in deferred_set else \
                  "NOT PLANNED" if p in not_planned_set else "SKIPPED"
        rows.append({
            "Section":              "C — Not Planned (terminal clear)",
            "Part":                 p,
            "Category":             part_category.get(p, "Stranger"),
            "Color":                part_color.get(p, "UNKNOWN"),
            "Fixed_Machine":        part_fixed_machine.get(p, "—"),
            "Terminal_Blocked":     "NO",
            "Terminal_Block_Reason": "—",
            "Planning_Status":      status,
            "Planning_Reason":      not_planned_reason(p),
            "Demand_Daily":         round(dem_d, 2),
            "Effective_Daily":      round(daily, 2),
            "Inventory_Before":     round(inv_b, 0),
            "Demand_Gap":           dem_gap,
            "Production_Qty":       0,
            "Machines_Used":        "—",
            "Changeovers_This_Part": 0,
            "Priority_Score":       round(priority_scores.get(p, 0), 2),
        })

    total_co    = sum(1 for r in plan if r.get("Changeover") == "Yes")
    total_qty   = round(sum(float(r.get("Production_Qty", 0)) for r in plan), 0)
    total_parts = len({r["Part"] for r in plan})

    blank = {col: "" for col in rows[0].keys()} if rows else {}
    def summary_row(label, value):
        r = {col: "" for col in (rows[0].keys() if rows else ["Section", "Part"])}
        r["Section"] = "SUMMARY"
        r["Part"]    = label
        r["Production_Qty"] = value
        return r

    if rows:
        rows.append(blank)
        rows.append(blank)
        rows.append(summary_row("─── DAILY TOTALS ───────────────────────────", ""))
        rows.append(summary_row("Total Parts Planned (produced qty > 0)",         total_parts))
        rows.append(summary_row("Total Parts NOT Planned (terminal-clear, unscheduled)",
                                 len(block_c)))
        rows.append(summary_row("Total Parts Terminal Blocked (hard blocked)",    len(block_a)))
        rows.append(summary_row("Total Parts Deferred (inventory at/above skip level)",
                                 len(deferred_set)))
        rows.append(summary_row("─── PRODUCTION TOTALS ──────────────────────", ""))
        rows.append(summary_row("Total Production Qty (all parts, all machines)", total_qty))
        rows.append(summary_row("Total Changeovers Planned Today",                total_co))
        rows.append(summary_row("Total Parts in Universe (all known parts)",      len(all_parts)))

    df = pd.DataFrame(rows)
    return df

# =============================================================
# SECTION 26 — MAIN SCHEDULER
# =============================================================

def schedule(parts, label=""):
    global _phase_a_machines, _dynamic_max_parts
    print("\n" + "─"*65)
    print(f"  {label}  |  {len(parts)} parts  |  {len(vt_machines)} machines")
    print("─"*65)

    scenario_id, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")
    print(f"  OPD cap: {opd_cap(scenario_id)} days")

    horizon_df = compute_indent_horizon(parts)

    print("\n" + "─"*65)
    print(f"  TERMINAL ELIGIBILITY PRE-FILTER")
    print(f"  Logic: required=max(demand,indent,min_run_qty)")
    print(f"         hard_block if terminal_inv < required*(1-{int(TERMINAL_RELAXATION_PCT*100)}%)")
    print(f"         Parts sorted: largest demand first (fixed parts excluded)")
    print("─"*65)

    eligible_non_fixed, eligible_fixed, terminal_blocked_parts_info = get_terminal_eligible_parts(
        parts, exclude_fixed=True)

    if terminal_blocked_parts_info:
        print(f"\n  TERMINAL BLOCKED ({len(terminal_blocked_parts_info)} parts — excluded from planning):")
        for info in terminal_blocked_parts_info:
            p = info["Part"]
            print(f"    X {p:<30}  {info['Reason'][:80]}")
    else:
        print(f"\n  No parts hard-blocked by terminals.")

    print(f"\n  Terminal-eligible parts: {len(eligible_non_fixed)} non-fixed + {len(eligible_fixed)} fixed")

    eligible_all = list(dict.fromkeys(eligible_non_fixed + eligible_fixed + list(part_fixed_machine.keys())))
    active_parts_eligible = [
        p for p in eligible_all
        if not should_skip(p)[0] and effective_daily(p) > 0
    ]

    priority_scores, score_rows = compute_priority_scores(active_parts_eligible)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()


    # ── PRE-DISTRIBUTION (V16) ────────────────────────────────
    machine_pre_assignment, _dynamic_max_parts = pre_distribute_parts_to_machines(
        active_parts_eligible, vt_machines
    )

    machine_hours         = {m: 0.0 for m in vt_machines}
    machine_last_part     = {m: machine_state.get(m) for m in vt_machines}
    current_inventory     = inventory.copy()
    inventory_start_of_day = inventory.copy()

    plan            = []
    already_planned = set()
    not_planned     = []
    deferred        = []

    # ── FIXED MACHINE PASS ─────────────────────────────────────
    if machine_fixed_parts:
        _phase_a_machines, _ = schedule_fixed_machines(
            machine_hours, machine_last_part, current_inventory,
            plan, already_planned, priority_scores, scenario_id
        )
    else:
        _phase_a_machines = set()

    # ── PASS 0: DEMAND-FIRST ───────────────────────────────────
    print("\n" + "─"*65)
    print(f"  PASS 0: DEMAND-FIRST (largest demand first)")
    print("─"*65)

    demand_parts_sorted = sorted(
        [p for p in eligible_non_fixed
         if demand_daily_raw.get(p, 0) > 0
         and current_inventory.get(p, 0) < demand_daily_raw.get(p, 0)
         and p not in already_planned],
        key=lambda p: (-demand_daily_raw.get(p, 0), -terminal_coverage_ratio(p))
    )

    for part in demand_parts_sorted:
        rows = assign_demand_for_part(
            part, scenario_id, machine_hours, machine_last_part,
            current_inventory, plan, already_planned, priority_scores
        )
        if rows:
            plan.extend(rows)
            print(f"    DEMAND-FIRST {part:28s}  gap={demand_daily_raw.get(part,0)-inventory.get(part,0):.0f}  "
                  f"qty={sum(float(r['Production_Qty']) for r in rows):.0f}")
        else:
            not_planned.append({
                "Part": part,
                "Reason": f"Demand gap={demand_daily_raw.get(part,0)-current_inventory.get(part,0):.0f} but no machine capacity"
            })

    # ── PASS 1: RUNNER PRIORITY ────────────────────────────────
    runner_log = enforce_runner_priority(
        plan, machine_hours, machine_last_part,
        current_inventory, already_planned,
        priority_scores, inventory_start_of_day
    )

    # ── PASS 2: INVENTORY BUILD (priority-sorted) ──────────────
    print("\n" + "─"*65)
    print(f"  PASS 2: INVENTORY BUILD (priority-sorted)")
    print("─"*65)

    sorted_active = sorted(
        active_parts_eligible,
        key=lambda p: priority_scores.get(p, 0),
        reverse=True
    )

    for part in sorted_active:
        if part in already_planned:
            _do_inv_build_on_existing(
                part, plan, scenario_id, machine_hours, current_inventory,
                effective_daily(part), current_inventory.get(part, 0)
            )
            continue

        skip, skip_reason = should_skip(part)
        if skip:
            if inventory.get(part, 0) == 0 and effective_daily(part) > 0:
                displaced = displace_for_zero_inv(
                    part, machine_hours, machine_last_part,
                    current_inventory, plan, already_planned, priority_scores
                )
                if not displaced:
                    deferred.append({"Part": part, "Reason": skip_reason})
            else:
                deferred.append({"Part": part, "Reason": skip_reason})
            continue

        if not vt_compat.get(part):
            not_planned.append({"Part": part, "Reason": "Not in compatibility matrix"})
            continue

        new_rows = assign_inventory_build(
            part, scenario_id, machine_hours, machine_last_part,
            current_inventory, plan, already_planned, priority_scores
        )
        if not new_rows and part not in already_planned:
            not_planned.append({"Part": part, "Reason": "No machine capacity available"})

    # ── PASS 2B: IDLE MACHINE RESCUE (V16) ───────────────────
    _terminal_blocked_info = [
        {"Part": p}
        for p in vt_compat.keys()
        if terminal_blocked(p)[0]
    ]
    rescue_idle_machines(
        plan, machine_hours, machine_last_part, current_inventory,
        already_planned, priority_scores, scenario_id,
        _terminal_blocked_info
    )

    # ── PASS 3: UTILIZATION ENFORCER ──────────────────────────
    micro_idle_log = utilization_enforcer(
        plan, machine_hours, machine_last_part,
        list(vt_compat.keys()), already_planned,
        current_inventory, scenario_id, priority_scores
    )

    # ── PASS 4: HIGH-DEMAND RESCUE ────────────────────────────
    rescued = high_demand_rescue(
        plan, machine_hours, machine_last_part,
        current_inventory, already_planned, not_planned, priority_scores
    )

    # ── STRATEGIC BUFFER FILLER ───────────────────────────────
    strategic_buffer_filler(
        plan, machine_hours, machine_last_part,
        list(vt_compat.keys()), already_planned,
        current_inventory, scenario_id
    )

    # ── RESEQUENCE + RECONCILE ────────────────────────────────
    plan = resequence_machine_rows(plan, machine_state)
    reconcile_machine_hours(plan, machine_hours)

    # ── STAGGER CHANGEOVERS ───────────────────────────────────
    stagger_adj = stagger_co_by_quantity(plan, vt_machines, scenario_id, current_inventory)
    stagger_changeovers(plan, vt_machines, machine_hours)
    reconcile_machine_hours(plan, machine_hours)

    # ── VALIDATION ────────────────────────────────────────────
    violations = validate_plan_rows(plan, current_inventory)

    # ── UPDATE PRIORITY SCORES IN PLAN ────────────────────────
    for row in plan:
        if row.get("Priority_Score") == 0:
            row["Priority_Score"] = priority_scores.get(row["Part"], 0)

    # ── PRINT SUMMARY ─────────────────────────────────────────
    print("\n" + "─"*65)
    print(f"  SCHEDULE COMPLETE")
    print(f"  Planned: {len({r['Part'] for r in plan})} parts")
    print(f"  Not planned: {len(not_planned)}")
    print(f"  Deferred: {len(deferred)}")
    print(f"  Changeovers: {sum(1 for r in plan if r.get('Changeover')=='Yes')}")
    print(f"  Stagger adjustments: {stagger_adj}")
    print(f"  Violations: {len(violations)}")
    avg_util = round(
        sum(machine_hours.get(m, 0) for m in vt_machines) / len(vt_machines) / AVAILABLE_HOURS * 100, 1
    ) if vt_machines else 0
    print(f"  Avg machine utilization: {avg_util}%")
    print("─"*65)

    # ── BUILD OUTPUT SHEETS ───────────────────────────────────
    plan_df = pd.DataFrame(plan) if plan else pd.DataFrame()
    all_parts_universe = list(set(
        list(vt_compat.keys()) +
        list(indent_monthly.keys()) +
        list(demand_daily_raw.keys())
    ))
    matrix_parts_set = set(vt_compat.keys())
    zero_rate_set    = set(data_zero_rate["Material"].tolist())

    sheets = {
        "VT_Plan":               plan_df,
        "VT_Machine_Plan":       build_machine_wise_plan(plan_df, machine_hours),
        "VT_Priority_Scores":    score_df,
        "VT_Prod_vs_Indent":     build_production_vs_indent(plan, all_parts_universe),
        "VT_Inventory_Target":   build_inventory_target_sheet(plan, all_parts_universe, scenario_id),
        "VT_Forward_Look":       build_forward_look(current_inventory, all_parts_universe),
        "VT_Part_Decision":      build_part_decision_sheet(
                                     all_parts_universe, plan, not_planned, deferred, priority_scores),
        "VT_Planned_vs_Unplanned": build_planned_vs_unplanned_sheet(
                                     all_parts_universe, plan, not_planned, deferred,
                                     terminal_blocked_parts_info),
        "VT_Multi_Machine":      build_multi_machine_view(plan),
        "VT_Terminal_Status":    build_terminal_status_sheet(),
        "VT_Fixed_Machines":     build_fixed_machine_status(plan, current_inventory),
        "VT_Runner_Priority":    pd.DataFrame(runner_log) if runner_log else pd.DataFrame(),
        "VT_CO_Queue":           build_co_queue(plan, vt_machines),
        "VT_Micro_Idle":         pd.DataFrame(micro_idle_log) if micro_idle_log else pd.DataFrame(),
        "VT_Audit":              build_audit(all_parts_universe, matrix_parts_set, zero_rate_set),
        "VT_Daily_Totals":       build_daily_totals_sheet(plan, machine_hours, vt_machines),
        "VT_Summary":            build_planning_summary_sheet(
                                     plan, not_planned, deferred,
                                     all_parts_universe, machine_hours, vt_machines,
                                     already_planned, scenario_desc),
        "VT_Indent_Horizon":     horizon_df,
        "VT_Violations":         pd.DataFrame(violations) if violations else pd.DataFrame(),
        "VT_Planning_Report":    build_daily_planning_report(
                                     plan, not_planned, deferred,
                                     terminal_blocked_parts_info,
                                     all_parts_universe, priority_scores),
    }

    return sheets, machine_hours, machine_last_part, current_inventory, scenario_desc

# =============================================================
# SECTION 27 — RUN & WRITE OUTPUT
# =============================================================

print("\n" + "─"*65)
print(f"  RUNNING VT SCHEDULER")
print("─"*65)

all_vt_parts = list(set(
    list(vt_compat.keys()) +
    list(indent_monthly.keys()) +
    list(demand_daily_raw.keys())
))

sheets, final_machine_hours, final_machine_last_part, final_inventory, scenario_desc = schedule(
    all_vt_parts, label="VT SCHEDULER"
)

# ── Save machine state ────────────────────────────────────────
save_machine_state(final_machine_last_part)

# ── Write Excel output ────────────────────────────────────────
# V17 FIX: apply _safe_df() to every sheet before writing so that
# mixed-type columns (numeric rows + "—" summary rows) do not trigger
# Excel's "repair needed" dialog when the file is opened.
print(f"\n  Writing output: {output_path}")
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        safe_sheet_name = sheet_name[:31]   # Excel sheet name limit
        if df is not None and not df.empty:
            safe = _safe_df(df)
            safe.to_excel(writer, sheet_name=safe_sheet_name, index=False)
            print(f"    Sheet '{safe_sheet_name}' written  ({len(df)} rows)")
        else:
            pd.DataFrame({"Note": [f"No data for {sheet_name}"]}).to_excel(
                writer, sheet_name=safe_sheet_name, index=False
            )
            print(f"    Sheet '{safe_sheet_name}' written  (empty placeholder)")

print("\n" + "─"*65)
print(f"  Smart APS V17 COMPLETE")
print(f"  Output file : {output_path}")
print(f"  Planning date: {PLANNING_DATE}")
print(f"  Scenario    : {scenario_desc}")
total_planned = len({r["Part"] for r in sheets["VT_Plan"].to_dict("records")}) if not sheets["VT_Plan"].empty else 0
print(f"  Parts planned: {total_planned}")
avg_util = round(
    sum(final_machine_hours.get(m, 0) for m in vt_machines) / len(vt_machines) / AVAILABLE_HOURS * 100, 1
) if vt_machines else 0
print(f"  Avg utilization: {avg_util}%")

# ── V17: Print half/full run machine summary ──────────────────
full_run_machines_used = [m for m in vt_machines if machine_run_type.get(m, "HALF") == "FULL"]
print(f"\n  V17 Half/Full Run Summary:")
print(f"    Full Run machines configured : {len(full_run_machines_used)}"
      + (f"  -> {full_run_machines_used}" if full_run_machines_used else " (add Run_Type=FULL column to VT_Changeover)"))
print(f"    Half Run machines            : {len(vt_machines) - len(full_run_machines_used)}")
print(f"    Full Run rate multiplier     : {FULL_RUN_MULTIPLIER}x (for parts with >= 2 tools)")
print(f"    Sheet breaking fix           : _safe_df() applied to all {len(sheets)} sheets")
print(f"{'='*65}")


─────────────────────────────────────────────────────────────────
  Smart APS V17  —  Half/Full Run + Sheet Fix
  Planning date    : 2026-04-09
  Indent month     : April 2026
  Working days     : 26  (30 days - 4 Sundays)
  Safety floor     : 3d  |  Target ceiling : 5d
  Max parts/machine: 3
  Runner priority  : inv < 2.0x daily
  Max daily CO     : 25
  Forward look     : 7 days
  Terminal relax   : 15%  (min-run NON-NEGOTIABLE)
  Terminal logic   : required = max(demand_daily, indent_daily)
                     hard_block if inv < required*(1-15%)
  Full Run mult    : 2.0x  (applies when tools>=2 on FULL machine)

Loading data...
  VT_Fixed sheet loaded  (13 rows)
  Terminal data loaded from  : C:/Users/Ex0164/Important codes/terminals and raw marterial - vt.xlsx
  Machine run types loaded   : 27 FULL, 3 HALF
  FULL run machines          : ['JSW 1ST', 'M.P-01', 'M.P-03', 'M.P-04', 'M.P-05', 'M.P-08', 'M.P-09', 'M.P-10', 'M.P-11', 'M.P-12', 'M.P-15', 'M.P-16', 'M.P-17', 'TOYO 1ST', 